##  CELL 1: Setup & Installation

In [25]:
# @title


# Install required packages
!pip install -q openai python-dotenv requests 2>/dev/null

# Optional: Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

print("✓ Dependencies installed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Dependencies installed


## CELL 2: Configuration & Environment

In [26]:
# @title
from __future__ import annotations

import os, sys, json, re, time, signal, hashlib, logging, logging.handlers
import traceback, tempfile, textwrap, importlib.util, subprocess, resource
import functools, threading, concurrent.futures, inspect
from collections import OrderedDict
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple, Union
from urllib.parse import urlparse

# Detect Colab environment
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

#  CONFIGURATION
_DEFAULT_ROOT = Path("/content/agent") if IS_COLAB else Path.home() /"agent_workspace"
#MODEL         = "mistralai/mistral-small-4-119b-2603"
MODEL         = "stepfun-ai/step-3.5-flash"
BASE_URL      = "https://integrate.api.nvidia.com/v1"
FAST_MODEL     = "mistralai/mistral-small-4-119b-2603"  # used for quick/express calls
FALLBACK_MODEL = "minimaxai/minimax-m2.5"                # used when primary model fails
MAX_TOKENS    = 61_072
MAX_ITERS     = 1000
CODE_TIMEOUT  = 60
SHELL_TIMEOUT = 120
WEB_TIMEOUT   = 20

def _cfg_int(env: str, default: int) -> int:
    if raw := os.environ.get(env, ""):
        try:
            return int(raw)
        except ValueError:
            print(f"  \033[93m[Config] WARNING: {env}={raw!r} not int; using {default}\033[0m",
                  file=sys.stderr)
    return default

def _cfg_bool(env: str, default: bool) -> bool:
    val = os.environ.get(env, "").lower()
    if val in ("1", "true", "yes", "on"):
        return True
    if val in ("0", "false", "no", "off"):
        return False
    return default

class Config:
    """Central configuration — override via environment variables."""
    # Model
    MODEL    = os.environ.get("NVIDIA_MODEL",    MODEL)
    FAST_MODEL    = os.environ.get("FAST_MODEL",    FAST_MODEL)
    FALLBACK_MODEL = os.environ.get("FALLBACK_MODEL", FALLBACK_MODEL)
    BASE_URL = os.environ.get("NVIDIA_BASE_URL", BASE_URL)
    MAX_TOKENS = _cfg_int("AGENT_MAX_TOKENS", MAX_TOKENS)
    MAX_ITERS  = _cfg_int("AGENT_MAX_ITERS",  MAX_ITERS)

    # Directories
    BASE_DIR    = Path(os.environ.get("AGENT_WORKSPACE", str(_DEFAULT_ROOT)))
    FILES_DIR   = BASE_DIR / "files"
    SUBTOOL_DIR = BASE_DIR / "subtools"
    SKILLS_DIR  = BASE_DIR / "skills"
    CORE_DIR    = BASE_DIR / "core"
    WORK_DIR    = BASE_DIR / "work"
    MEMORY_DIR  = BASE_DIR / "memory"
    MEM_FILE    = BASE_DIR / "memory.json"
    STATS_FILE  = BASE_DIR / "api_stats.jsonl"

    # Timeouts
    CODE_TIMEOUT  = _cfg_int("AGENT_CODE_TIMEOUT",  CODE_TIMEOUT)
    SHELL_TIMEOUT = _cfg_int("AGENT_SHELL_TIMEOUT", SHELL_TIMEOUT)
    WEB_TIMEOUT   = _cfg_int("AGENT_WEB_TIMEOUT",   WEB_TIMEOUT)

    # Limits
    MAX_FILE_SIZE_MB         = 50
    MAX_WEB_CONTENT          = 15_000
    MAX_PYTHON_MEMORY_MB     = 256
    SYSTEM_PROMPT_MEM_LIMIT  = 25
    SYSTEM_PROMPT_TOOL_LIMIT = 15
    MAX_PARALLEL_REQUESTS    = 20
    CONTEXT_TOKEN_BUDGET     = _cfg_int("AGENT_CTX_BUDGET", 262_000)
    TOOL_RESULT_CHARS_MAX    = 60_000
    TOOL_CACHE_SIZE          = 256
    API_RETRY_MAX            = 10

    # Per-tool timeouts (seconds) — override specific tools without touching global defaults
    TOOL_TIMEOUTS: dict[str, int] = {
        "web_fetch":      int(os.environ.get("AGENT_TIMEOUT_WEB_FETCH",   "30")),
        "web_search":     int(os.environ.get("AGENT_TIMEOUT_WEB_SEARCH",  "20")),
        "execute_python": int(os.environ.get("AGENT_TIMEOUT_PYTHON",      "60")),
        "shell_run":      int(os.environ.get("AGENT_TIMEOUT_SHELL",      "120")),
        "browser_fetch":  int(os.environ.get("AGENT_TIMEOUT_BROWSER",     "45")),
        "sql_query":      int(os.environ.get("AGENT_TIMEOUT_SQL",         "30")),
    }

    # Cost per 1M tokens (USD) — update when provider pricing changes
    # Set to 0 to disable cost tracking for a model
    MODEL_COSTS: dict[str, dict[str, float]] = {
        "stepfun-ai/step-3.5-flash":  {"in": 0.15,  "out": 0.60},
        "gpt-4o":                     {"in": 2.50,  "out": 10.00},
        "gpt-4o-mini":                {"in": 0.15,  "out": 0.60},
        "claude-opus-4-6":            {"in": 15.00, "out": 75.00},
        "claude-sonnet-4-6":          {"in": 3.00,  "out": 15.00},
    }

    @classmethod
    def tool_timeout(cls, tool_name: str) -> int:
        """Return the configured timeout for *tool_name*, falling back to SHELL_TIMEOUT."""
        return cls.TOOL_TIMEOUTS.get(tool_name, cls.SHELL_TIMEOUT)

    @classmethod
    def cost_for(cls, model: str, tokens_in: int, tokens_out: int) -> float:
        """Return estimated USD cost for a call. Returns 0.0 if model not in price table."""
        pricing = cls.MODEL_COSTS.get(model, {})
        if not pricing:
            return 0.0
        return (tokens_in * pricing.get("in", 0) + tokens_out * pricing.get("out", 0)) / 1_000_000
    API_RETRY_BACKOFF        = 1.5
    SKILLS_CACHE_TTL         = 30.0
    HEARTBEAT_INTERVAL       = max(60, min(_cfg_int("AGENT_HEARTBEAT_INTERVAL", 1800), 21600))  # 1min–6h
    EXPRESS_PROMPT = _cfg_bool("EXPRESS_PROMPT", False)
    DEBUG_STREAM_RAW = _cfg_bool("DEBUG_STREAM_RAW", True)

    @classmethod
    def setup_dirs(cls):
        for d in (cls.FILES_DIR, cls.SUBTOOL_DIR, cls.SKILLS_DIR,
                  cls.CORE_DIR, cls.WORK_DIR, cls.MEMORY_DIR):
            d.mkdir(parents=True, exist_ok=True)

    @classmethod
    def validate(cls):
        for ok, msg in (
            (cls.MAX_TOKENS > 0,                "MAX_TOKENS must be positive"),
            (cls.MAX_ITERS > 0,                 "MAX_ITERS must be positive"),
            (cls.CODE_TIMEOUT > 0,              "CODE_TIMEOUT must be positive"),
            (cls.CONTEXT_TOKEN_BUDGET > 10_000, "CONTEXT_TOKEN_BUDGET too small"),
            (cls.TOOL_RESULT_CHARS_MAX > 0,     "TOOL_RESULT_CHARS_MAX must be positive"),
            (cls.MAX_FILE_SIZE_MB > 0,          "MAX_FILE_SIZE_MB must be positive"),
            (cls.HEARTBEAT_INTERVAL >= 60,      "HEARTBEAT_INTERVAL must be >= 60 s"),
        ):
            if not ok:
                raise ValueError(f"[Config] {msg}")

Config.setup_dirs()
Config.validate()

print(f"✓ Configuration loaded")
print(f"  Workspace: {Config.BASE_DIR}")
print(f"  Model: {Config.MODEL}")

✓ Configuration loaded
  Workspace: /content/agent
  Model: stepfun-ai/step-3.5-flash


## CELL 3: Logging System

In [27]:
# @title
class AgentLogger:
    """JSON-structured rotating logger + colour console output + daily log files."""

    LOG_FILE = Config.BASE_DIR / "agent.log"

    _LEVELS: dict[str, int] = {
        "DEBUG":   logging.DEBUG,
        "INFO":    logging.INFO,
        "WARNING": logging.WARNING,
        "ERROR":   logging.ERROR,
    }
    _COLORS: dict[str, str] = {
        "debug":   "\033[90m",
        "info":    "\033[97m",
        "warning": "\033[93m",
        "error":   "\033[91m",
    }

    # Limits applied when emitting / formatting records
    _MSG_MAX   = 10_000
    _EXTRA_MAX = 2_000
    _FMT_EXTRA = 5   # max extra fields shown in format_record
    _FMT_MSG   = 200 # max msg chars shown in format_record

    # Rotating file handler settings
    _ROTATE_BYTES   = 10 * 1024 * 1024  # 10 MB
    _ROTATE_BACKUPS = 5

    def __init__(self) -> None:
        lvl = self._LEVELS.get(
            os.environ.get("AGENT_LOG_LEVEL", "DEBUG").upper(),
            logging.DEBUG,
        )
        self._logger = logging.getLogger("agent")
        self._logger.setLevel(lvl)
        self._logger.propagate = False

        if not self._logger.handlers:
            self._add_rotating_handler()
            self._add_daily_handler()

    # ── Handler setup ────────────────────────────────────────────────

    def _add_rotating_handler(self) -> None:
        try:
            fh = logging.handlers.RotatingFileHandler(
                self.LOG_FILE,
                maxBytes=self._ROTATE_BYTES,
                backupCount=self._ROTATE_BACKUPS,
                encoding="utf-8",
            )
            fh.setFormatter(logging.Formatter("%(message)s"))
            fh.setLevel(logging.DEBUG)
            self._logger.addHandler(fh)
        except Exception:
            pass  # Silently degrade — console output still works

    def _add_daily_handler(self) -> None:
        try:
            today = datetime.now().strftime("%Y-%m-%d")
            daily = Config.MEMORY_DIR / f"agent_{today}.log"
            dfh   = logging.FileHandler(daily, encoding="utf-8")
            dfh.setLevel(logging.DEBUG)
            dfh.setFormatter(logging.Formatter(
                "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
            ))
            self._logger.addHandler(dfh)
        except Exception:
            pass  # Silently degrade

    # ── Emit ─────────────────────────────────────────────────────────

    def _emit(self, level: str, msg: str, **extra: Any) -> None:
        rec: dict[str, Any] = {
            "ts":    datetime.now().isoformat(timespec="milliseconds"),
            "level": level,
            "msg":   str(msg)[: self._MSG_MAX],
            **{k: str(v)[: self._EXTRA_MAX] for k, v in extra.items()},
        }
        getattr(self._logger, level, self._logger.info)(json.dumps(rec))

    def debug(self,   msg: str, **kw: Any) -> None: self._emit("debug",   msg, **kw)
    def info(self,    msg: str, **kw: Any) -> None: self._emit("info",    msg, **kw)
    def warning(self, msg: str, **kw: Any) -> None: self._emit("warning", msg, **kw)
    def error(self,   msg: str, **kw: Any) -> None: self._emit("error",   msg, **kw)

    # ── Reading ──────────────────────────────────────────────────────

    def read_recent(self, n: int = 50) -> list[dict[str, Any]]:
        """Return the last *n* log records from the rotating log file."""
        if not self.LOG_FILE.exists():
            return []
        try:
            lines = self.LOG_FILE.read_text(encoding="utf-8", errors="replace").splitlines()
        except Exception:
            return []

        recs: list[dict[str, Any]] = []
        for line in reversed(lines):
            try:
                recs.append(json.loads(line))
                if len(recs) >= n:
                    break
            except Exception:
                continue  # Skip malformed lines

        return list(reversed(recs))

    # ── Formatting ───────────────────────────────────────────────────

    def format_record(self, r: dict[str, Any]) -> str:
        color  = self._COLORS.get(r.get("level", ""), "\033[97m")
        extra  = {k: v for k, v in r.items() if k not in ("ts", "level", "msg")}
        ext    = ""
        if extra:
            pairs = list(extra.items())[: self._FMT_EXTRA]
            ext   = "  " + "  ".join(f"{k}={v}" for k, v in pairs)
        ts    = r.get("ts", "?")[:23]
        level = r.get("level", "?").upper()
        msg   = r.get("msg",   "")[: self._FMT_MSG]
        return f"{color}{ts}  {level:<7}  {msg}{ext}\033[0m"

    # ── Maintenance ──────────────────────────────────────────────────

    def rotate_old_daily_logs(self, keep_days: int = 30) -> None:
        """Remove daily log files older than *keep_days* days."""
        try:
            cutoff = datetime.now() - timedelta(days=keep_days)
            for lf in Config.MEMORY_DIR.glob("agent_*.log"):
                try:
                    date_part = lf.stem.removeprefix("agent_")
                    # Skip files whose names don't match the expected date pattern
                    if not re.match(r'^\d{4}-\d{2}-\d{2}$', date_part):
                        continue
                    if datetime.strptime(date_part, "%Y-%m-%d") < cutoff:
                        lf.unlink()
                except (ValueError, OSError):
                    continue
        except Exception as e:
            self.warning(f"Log rotation failed: {e}")


logger = AgentLogger()
print("✓ Logger initialized")

✓ Logger initialized


## CELL 4: Utilities (Retry, Rate Limiter, Cache)

In [28]:
# @title

# ---------- Retry Decorator ----------

_RETRYABLE_KEYS: tuple[str, ...] = (
    "429", "rate limit", "too many",
    "503", "502", "500",
    "server error", "overloaded",
    "timeout", "connection",
)


import random as _random

def _parse_retry_after(exc: Exception) -> float | None:
    """
    Try to extract a Retry-After value (seconds) from an API exception.
    Checks: exception attributes, response headers, and error message text.
    Returns None if not found.
    """
    # Check for response object on the exception (openai SDK pattern)
    response = getattr(exc, "response", None)
    if response is not None:
        headers = getattr(response, "headers", {})
        if ra := headers.get("Retry-After") or headers.get("retry-after"):
            try:
                return float(ra)
            except (ValueError, TypeError):
                pass
    # Check error message for "retry after N seconds"
    msg = str(exc).lower()
    if m := re.search(r"retry.{0,10}after.{0,5}(\d+(?:\.\d+)?)", msg):
        try:
            return float(m.group(1))
        except ValueError:
            pass
    return None


def retry_api(max_retries: int | None = None, backoff: float | None = None):
    """
    Decorator that retries a function on transient API errors with
    exponential back-off + jitter. Respects Retry-After headers.
    Only retries exceptions whose string representation contains
    one of ``_RETRYABLE_KEYS``.
    """
    mr = max_retries if max_retries is not None else Config.API_RETRY_MAX
    bf = backoff     if backoff     is not None else Config.API_RETRY_BACKOFF

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(mr):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    is_last      = attempt == mr - 1
                    is_retryable = any(k in str(e).lower() for k in _RETRYABLE_KEYS)
                    if not is_retryable or is_last:
                        raise
                    # Honour Retry-After header if present, else exponential + jitter
                    retry_after = _parse_retry_after(e)
                    if retry_after is not None:
                        wait = retry_after
                    else:
                        wait = (bf ** attempt) + _random.uniform(0, 0.5)
                    print(f"\033[93m  [retry] {attempt + 1}/{mr} after {wait:.1f}s — {e}\033[0m")
                    time.sleep(wait)
        return wrapper
    return decorator


# ---------- Rate Limiter ----------

class RateLimiter:
    """Sliding-window rate limiter (thread-safe)."""

    _WINDOW_SECONDS = 60

    def __init__(self, calls_per_minute: int = 60) -> None:
        self._calls:     list[float] = []
        self._max_calls: int         = calls_per_minute
        self._lock       = threading.Lock()

    def wait_if_needed(self) -> None:
        """Block if the call rate would exceed the configured limit."""
        with self._lock:
            now    = time.time()
            cutoff = now - self._WINDOW_SECONDS
            # Discard timestamps outside the sliding window
            self._calls = [t for t in self._calls if t > cutoff]

            if len(self._calls) >= self._max_calls:
                # Sleep until the oldest call ages out of the window
                wait = self._WINDOW_SECONDS - (now - self._calls[0]) + 0.1
                if wait > 0:
                    time.sleep(wait)

            self._calls.append(time.time())


# ---------- LRU Tool Cache ----------

class ToolResultCache:
    """Thread-safe LRU cache for tool results with per-tool TTL support."""

    _WEB_TTL = 300  # seconds — applied to any tool whose name starts with "web_"

    def __init__(self, maxsize: int | None = None) -> None:
        self._cache:   OrderedDict[str, tuple[Any, float]] = OrderedDict()
        self._maxsize: int         = maxsize or Config.TOOL_CACHE_SIZE
        self._lock     = threading.Lock()

    # ── Internal helpers ─────────────────────────────────────────────

    def _key(self, tool: str, args: dict) -> str:
        """Return a short, stable cache key for *(tool, args)*."""
        raw = tool + json.dumps(args, sort_keys=True, default=str)
        return hashlib.sha256(raw.encode()).hexdigest()[:16]

    def _is_expired(self, tool: str, timestamp: float) -> bool:
        """Return True if the cached entry has exceeded its TTL."""
        return tool.startswith("web_") and (time.time() - timestamp) > self._WEB_TTL

    def _evict_to_fit(self) -> None:
        """Remove the oldest entries until the cache is within *_maxsize*."""
        while len(self._cache) > self._maxsize:
            self._cache.popitem(last=False)

    # ── Public API ───────────────────────────────────────────────────

    def get(self, tool: str, args: dict) -> Any | None:
        """Return the cached result for *(tool, args)*, or ``None`` on miss/expiry."""
        k = self._key(tool, args)
        with self._lock:
            if k not in self._cache:
                return None
            result, ts = self._cache[k]
            if self._is_expired(tool, ts):
                del self._cache[k]
                return None
            self._cache.move_to_end(k)
            return result

    def set(self, tool: str, args: dict, result: Any) -> None:
        """Store *result* for *(tool, args)*, evicting the oldest entry if needed."""
        k = self._key(tool, args)
        with self._lock:
            self._cache[k] = (result, time.time())
            self._cache.move_to_end(k)
            self._evict_to_fit()

    def invalidate(self, tool_prefix: str = "") -> None:
        """Remove all entries whose key contains *tool_prefix*."""
        with self._lock:
            stale = [k for k in self._cache if tool_prefix in k]
            for k in stale:
                del self._cache[k]

    @property
    def size(self) -> int:
        """Current number of cached entries."""
        return len(self._cache)


print("✓ Utilities loaded (retry, rate limiter, cache)")

✓ Utilities loaded (retry, rate limiter, cache)


## CELL 5: API Counter & Live Status Bar

In [29]:
# @title
# ---------- API Counter ----------

import threading
import time
import json
import sys
import re
import warnings
import os
from collections import deque
from datetime import datetime
from pathlib import Path
from shutil import get_terminal_size
from typing import Any, Callable, Dict, List, Optional, Protocol


# ── Type definition ───────────────────────────────────────────────────────────

class CostCalculator(Protocol):
    """Structural type for cost calculator callables — improves IDE autocomplete."""
    def __call__(self, model: str, tokens_in: int, tokens_out: int) -> float: ...


class APICounter:
    """Thread-safe counter for API requests, token usage, and session statistics."""

    def __init__(
        self,
        cost_calculator: Optional[CostCalculator] = None,
        stats_file: Optional[Path] = None,
    ) -> None:
        self._lock               = threading.Lock()
        self._total:             int   = 0
        self._errors:            int   = 0
        self._tokens_in:         int   = 0
        self._tokens_out:        int   = 0
        self._tokens_reasoning:  int   = 0
        self._by_model:          Dict[str, int] = {}
        self._cost_usd:          float = 0.0
        # FIX (from Doc 10): time.monotonic() is clock-jump-safe — time.time()
        # can jump backwards on NTP sync, making uptime/avg calculations wrong.
        self._start:             float = time.monotonic()
        self._cost_calculator    = cost_calculator
        self._stats_file         = stats_file
        self._cost_error_logged: bool  = False

    # ── Recording ────────────────────────────────────────────────────

    def record(
        self,
        model:            str  = "",
        tokens_in:        int  = 0,
        tokens_out:       int  = 0,
        tokens_reasoning: int  = 0,
        error:            bool = False,
    ) -> None:
        """Record a single API call and its token usage."""
        if tokens_in < 0 or tokens_out < 0 or tokens_reasoning < 0:
            raise ValueError("Token counts cannot be negative")

        # FIX: Calculate cost outside the lock so a slow/blocking cost_calculator
        # does not hold the lock while other threads are trying to record.
        # This was already done in the original — preserved and kept here.
        cost = 0.0
        if self._cost_calculator:
            try:
                cost = self._cost_calculator(model, tokens_in, tokens_out)
            except Exception as e:
                # FIX: Avoid a second lock acquisition inside the lock-guarded
                # block below. Read _cost_error_logged under the lock just once,
                # then log outside it, then update the flag under the lock again.
                # Original did a nested lock inside the outer with-block which
                # would deadlock on a non-reentrant threading.Lock.
                should_warn = False
                with self._lock:
                    if not self._cost_error_logged:
                        self._cost_error_logged = True
                        should_warn = True
                if should_warn:
                    warnings.warn(f"Cost calculation failed: {e}")

        with self._lock:
            self._total += 1
            if model:
                self._by_model[model] = self._by_model.get(model, 0) + 1
            self._tokens_in        += tokens_in
            self._tokens_out       += tokens_out
            self._tokens_reasoning += tokens_reasoning
            if error:
                self._errors += 1
            self._cost_usd += cost

    def next_request_number(self) -> int:
        """Return the number that the *next* request will receive."""
        with self._lock:
            return self._total + 1

    # ── Thread-safe snapshot ─────────────────────────────────────────

    def _snapshot(self) -> Dict[str, Any]:
        """Thread-safe snapshot of current state."""
        with self._lock:
            return {
                "total_requests":    self._total,
                "errors":            self._errors,
                "tokens_prompt":     self._tokens_in,
                "tokens_response":   self._tokens_out,
                "tokens_reasoning":  self._tokens_reasoning,
                "by_model":          dict(self._by_model),
                "cost_usd":          self._cost_usd,
            }

    # ── Properties ───────────────────────────────────────────────────

    @property
    def total(self) -> int:
        """Total number of requests recorded so far."""
        with self._lock:
            return self._total

    # ── Reporting ────────────────────────────────────────────────────

    def summary(self) -> Dict[str, Any]:
        """Return a snapshot of all counters as a plain dict."""
        snap    = self._snapshot()
        elapsed = time.monotonic() - self._start   # consistent with __init__
        tokens_total = (
            snap["tokens_prompt"]
            + snap["tokens_response"]
            + snap["tokens_reasoning"]
        )
        return {
            **snap,
            "success":              snap["total_requests"] - snap["errors"],
            "tokens_total":         tokens_total,
            "uptime_seconds":       round(elapsed, 1),
            "avg_requests_per_min": round(
                snap["total_requests"] / max(elapsed / 60, 0.01), 2
            ),
        }

    def __str__(self) -> str:
        s        = self.summary()
        cost_str = f" │ ~${s['cost_usd']:.4f}" if s["cost_usd"] > 0 else ""
        return (
            f"Requests: {s['total_requests']} (✓{s['success']} ✗{s['errors']}) | "
            f"Prompt: {s['tokens_prompt']:,} │ Think: {s['tokens_reasoning']:,} │ "
            f"Resp: {s['tokens_response']:,} │ Avg: {s['avg_requests_per_min']}/min{cost_str}"
        )

    # ── Persistence ──────────────────────────────────────────────────

    def persist_stats(self) -> None:
        """Append the current session summary to the stats file."""
        stats_file = self._stats_file
        if stats_file is None:
            try:
                from config import Config  # type: ignore
                stats_file = Config.STATS_FILE
            except ImportError:
                warnings.warn("No stats file configured; skipping persistence")
                return

        try:
            entry = json.dumps(
                {**self.summary(), "session_end": datetime.now().isoformat()}
            )
            stats_file.parent.mkdir(parents=True, exist_ok=True)
            with open(stats_file, "a", encoding="utf-8") as f:
                f.write(entry + "\n")
        except Exception as e:
            warnings.warn(f"Failed to persist stats to {stats_file}: {e}")

    def load_historical(self) -> List[Dict[str, Any]]:
        """Return all previously persisted session summaries."""
        stats_file = self._stats_file
        if stats_file is None:
            try:
                from config import Config  # type: ignore
                stats_file = Config.STATS_FILE
            except ImportError:
                return []

        if not stats_file.exists():
            return []

        recs: List[Dict[str, Any]] = []
        try:
            for line in stats_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                # FIX: Skip blank lines before attempting JSON parse — original
                # passed empty strings to json.loads which raises JSONDecodeError
                # even though blank lines are perfectly normal in append-style logs.
                if not line:
                    continue
                try:
                    recs.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
        except Exception as e:
            warnings.warn(f"Failed to load historical stats: {e}")
        return recs

    # ── Reset ─────────────────────────────────────────────────────────

    def reset(self) -> None:
        """
        Reset all counters and restart the session clock.
        Useful when starting a new logical session without recreating the object.
        FIX: Was missing entirely — callers had no way to reset without
        creating a new instance, losing the cost_calculator and stats_file refs.
        """
        with self._lock:
            self._total            = 0
            self._errors           = 0
            self._tokens_in        = 0
            self._tokens_out       = 0
            self._tokens_reasoning = 0
            self._by_model         = {}
            self._cost_usd         = 0.0
            self._start            = time.monotonic()   # consistent with __init__
            self._cost_error_logged = False


# ---------- Live Status Bar ----------

class LiveStatusBar:
    """
    In-place terminal status bar that updates as a streaming API response
    arrives, then finalises with a summary line.

    Supports use as a context manager — finalize() is guaranteed to run
    even if an exception is thrown mid-stream:

        with LiveStatusBar(req_num) as bar:
            bar.add_reasoning(chunk)
            bar.add_response(chunk)
        # finalize() called automatically here
    """

    REDRAW_INTERVAL  = 0.35   # seconds between redraws
    _ANSI_RE         = re.compile(r"\033\[[0-9;]*m")
    _WORD_BUFFER_MAX = 30     # max words kept in the rolling snippet buffer

    # ── Colour palette ───────────────────────────────────────────────────────
    # All codes chosen to be readable on BOTH dark and light terminal backgrounds.
    #
    # REMOVED (problematic):
    #   \033[90m  dark gray      — invisible on dark bg; barely visible on light
    #   \033[97m  bright white   — invisible on light/white bg
    #   \033[93m  bright yellow  — washed out on light bg
    #   \033[92m  bright green   — washed out on light bg
    #   \033[91m  bright red     — acceptable but bold version is more consistent
    #
    # REPLACEMENTS:
    #   \033[2m   dim/faint      — reduces current fg intensity; works on both bgs
    #   \033[1m   bold           — uses terminal's own fg (dark on light, light on dark)
    #   \033[33;1m bold yellow   — standard yellow + bold; visible on both bgs
    #   \033[35;1m bold magenta  — adds bold so it doesn't disappear on light bg
    #   \033[36;1m bold cyan     — adds bold so it doesn't disappear on light bg
    #   \033[32;1m bold green    — standard green + bold; visible on both bgs
    #   \033[31;1m bold red      — standard red + bold; consistent with green
    #
    # FIX (from Doc 10): _C is a class-level constant for the colored case.
    # The NO_COLOR/non-TTY case is handled per-instance in __init__ by replacing
    # _C with a blank palette — so instances can differ if stdout changes between
    # instantiations (e.g. in tests that redirect stdout).
    _C_COLOR: Dict[str, str] = {
        "reset":     "\033[0m",
        "dim":       "\033[2m",
        "bold":      "\033[1m",
        "req":       "\033[33;1m",
        "reasoning": "\033[35;1m",
        "response":  "\033[36;1m",
        "success":   "\033[32;1m",
        "error":     "\033[31;1m",
        "snippet":   "\033[2m",
    }
    _C_PLAIN: Dict[str, str] = {k: "" for k in _C_COLOR}

    def __init__(self, req_number: int, last_n_words: int = 8) -> None:
        self.req_number:       int  = req_number
        self.last_n_words:     int  = last_n_words
        self.reasoning_tokens: int  = 0
        self.response_tokens:  int  = 0
        # FIX (from Doc 10): deque(maxlen=N) evicts automatically — no manual
        # slice needed on every append, and no new list allocation on overflow.
        self._words:           deque = deque(maxlen=self._WORD_BUFFER_MAX)
        self._lock             = threading.Lock()
        self._last_draw:       float = 0.0
        self._finalised:       bool  = False
        self._content_changed: bool  = False
        self._t0:              float = time.monotonic()

        # FIX (from Doc 10): Evaluate USE_COLOR per-instance so that redirected
        # stdout in tests or notebooks is correctly detected at instantiation time,
        # not baked in at module import. Respects the NO_COLOR env var standard.
        use_color = sys.stdout.isatty() and not os.getenv("NO_COLOR")
        self._C: Dict[str, str] = self._C_COLOR if use_color else self._C_PLAIN

        # Initial draw
        self._content_changed = True
        self._redraw(force=True)

    # ── Context manager (from Doc 10) ────────────────────────────────

    def __enter__(self) -> "LiveStatusBar":
        """Allow use as a context manager — guarantees finalize() on exit."""
        return self

    def __exit__(self, *args: Any) -> None:
        """Finalize on block exit, even if an exception was raised mid-stream."""
        self.finalize()

    # ── Token helpers ────────────────────────────────────────────────

    @staticmethod
    def _approx(text: str) -> int:
        """
        Cheap token-count approximation (1 token ≈ 4 chars) for real-time display.
        Intentionally slightly optimistic — display accuracy matters less than speed.
        Note: Agent._est_tokens uses // 3 (more conservative) for history budget
        management, where under-counting risks a context-window overflow.
        """
        return max(1, len(text) // 4)

    # ── Rendering ────────────────────────────────────────────────────

    def _truncate_snippet(self, snippet: str, max_len: int) -> str:
        """Truncate snippet at word boundary if possible."""
        if len(snippet) <= max_len:
            return snippet
        truncated  = snippet[:max_len]
        last_space = truncated.rfind(" ")
        if last_space > max_len * 0.7:
            return truncated[:last_space] + "…"
        return truncated + "…"

    def _render(self) -> str:
        """
        Build the status line string.
        FIX: Must be called with self._lock held (or before the lock is needed),
        because it reads self._words and token counts. In the original, _render()
        was called from _redraw() which acquired the lock AFTER calling _render(),
        meaning _render() read shared state without holding the lock — a data race
        under concurrent add_response() / add_reasoning() calls. Fixed by moving
        the render call inside the lock in _redraw().
        """
        total   = self.reasoning_tokens + self.response_tokens
        elapsed = time.monotonic() - self._t0
        tps     = total / elapsed if elapsed > 0.5 else 0.0

        C   = self._C
        sep = f"{C['dim']}│{C['reset']}"

        base = (
            f"{C['dim']}┤{C['reset']} {C['req']}REQ #{self.req_number:<3}{C['reset']} "
            f"{sep} 🧠 {C['reasoning']}{self.reasoning_tokens:>7,}{C['reset']} "
            f"{sep} 💬 {C['response']}{self.response_tokens:>7,}{C['reset']} "
            f"{sep} Σ {C['bold']}{total:>8,}{C['reset']}"
        )
        if tps > 0:
            base += f"  {C['dim']}{tps:>5.1f}t/s{C['reset']}"

        # FIX (from Doc 10 — fixed): get_terminal_size() called here per-render
        # so the budget adapts if the user resizes their terminal. Doc 10 put this
        # as a class attribute (evaluated once at import time) which never updated.
        # fallback=(120, 24) used when stdout is not a TTY (pipes, CI, notebooks).
        line_width = get_terminal_size(fallback=(120, 24)).columns
        budget     = line_width - len(self._ANSI_RE.sub("", base)) - 4

        # FIX (from Doc 10 — fixed): _words is now a deque; convert to list for
        # slicing then join. list(deque)[-N:] is O(N) and correct.
        snippet = " ".join(list(self._words)[-self.last_n_words:])
        if snippet and budget > 8:
            snippet = self._truncate_snippet(snippet, budget)
            return base + f"  {C['snippet']}…{snippet}{C['reset']}"
        return base

    def _redraw(self, force: bool = False) -> None:
        now = time.monotonic()
        with self._lock:
            if self._finalised:
                return
            if not force and not self._content_changed:
                return
            if not force and (now - self._last_draw) < self.REDRAW_INTERVAL:
                return
            # FIX: _render() now called inside the lock so it reads consistent
            # state. Original called sys.stdout.write("\r" + self._render()) but
            # _render() was not under the lock, creating a data race on _words
            # and token counts when streaming threads call add_response() concurrently.
            line = self._render()
            self._last_draw       = now
            self._content_changed = False

        # FIX: stdout write moved OUTSIDE the lock. Writing to stdout can block
        # (e.g. on a slow terminal, pipe, or notebook output cell). Holding the
        # lock during a blocking I/O call would stall all concurrent add_* calls
        # for the entire write duration — defeating the purpose of the lock.
        sys.stdout.write("\r" + line)
        sys.stdout.flush()

    # ── Streaming input ───────────────────────────────────────────────

    def add_reasoning(self, text: str) -> None:
        """Accumulate reasoning tokens from a streaming chunk."""
        # FIX: Update shared state under the lock before signalling a redraw.
        # Original updated self.reasoning_tokens without holding the lock, then
        # called _redraw() which acquired it — leaving a window where another
        # thread could read a partially-updated token count.
        with self._lock:
            self.reasoning_tokens += self._approx(text)
            self._content_changed  = True
        self._redraw()

    def add_response(self, text: str) -> None:
        """Accumulate response tokens and update the rolling word snippet."""
        with self._lock:
            self.response_tokens  += self._approx(text)
            if words := text.split():
                # FIX (from Doc 10): deque(maxlen=_WORD_BUFFER_MAX) evicts oldest
                # words automatically — no manual slice or list reallocation needed.
                self._words.extend(words)
            self._content_changed  = True
        self._redraw()

    def set_actual_tokens(self, reasoning: int = 0, response: int = 0) -> None:
        """Override approximated counts with the actual values from the API."""
        with self._lock:
            if reasoning > 0:
                self.reasoning_tokens = reasoning
            if response > 0:
                self.response_tokens = response
            self._content_changed = True
        self._redraw(force=True)

    # ── Finalisation ─────────────────────────────────────────────────

    def finalize(self, success: bool = True, elapsed: float = 0.0) -> None:
        """Print the final summary line and mark the bar as done."""
        with self._lock:
            if self._finalised:
                # FIX: Guard against double-finalise — original had no check so
                # calling finalize() twice would print a duplicate summary line.
                return
            self._finalised = True
            C    = self._C
            sep  = f"{C['dim']}│{C['reset']}"
            icon = f"{C['success']}✓{C['reset']}" if success else f"{C['error']}✗{C['reset']}"
            total        = self.reasoning_tokens + self.response_tokens
            time_suffix  = f"  {C['dim']}{elapsed:.1f}s{C['reset']}" if elapsed else ""
            line = (
                f"\r{icon} REQ #{self.req_number:<3} {sep} "
                f"🧠 {C['reasoning']}think {self.reasoning_tokens:>7,}{C['reset']}  {sep} "
                f"💬 resp {C['response']}{self.response_tokens:>7,}{C['reset']}  {sep} "
                f"Σ {C['bold']}{total:>8,} tok{C['reset']}{time_suffix}\n"
            )

        # FIX: stdout write outside the lock — same reason as _redraw().
        sys.stdout.write(line)
        sys.stdout.flush()


print("✓ API Counter and Status Bar ready")

✓ API Counter and Status Bar ready


## CELL 6: File Managers

In [30]:
# @title
# ---------- General File Manager (unrestricted) ----------

import shutil
import concurrent.futures
from datetime import datetime
from pathlib import Path
from typing import Any


class FileManager:
    """General-purpose file I/O helpers (not workspace-sandboxed)."""

    # ── Internal helpers ─────────────────────────────────────────────

    @staticmethod
    def _resolve(filepath: Path | str) -> Path:
        """
        Coerce *filepath* to an absolute ``Path``, anchoring relative
        paths under ``Config.BASE_DIR``.
        """
        p = filepath if isinstance(filepath, Path) else Path(filepath)
        return p if p.is_absolute() else Config.BASE_DIR / p

    # ── Public API ───────────────────────────────────────────────────

    @staticmethod
    def read_file(filepath: Path | str, default: str = "", encoding: str = "utf-8") -> str:
        """Return the text content of *filepath*, or *default* on any error."""
        p = FileManager._resolve(filepath)
        try:
            if p.exists():
                return p.read_text(encoding=encoding)
        except Exception as e:
            logger.warning(f"Failed to read {p}: {e}")
        return default

    @staticmethod
    def write_file(
        filepath:       Path | str,
        content:        str,
        mode:           str  = "w",
        encoding:       str  = "utf-8",
        create_parents: bool = True,
    ) -> bool:
        """Write *content* to *filepath*. Returns ``True`` on success."""
        p = FileManager._resolve(filepath)
        try:
            if create_parents:
                p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(content, encoding=encoding)
            return True
        except Exception as e:
            logger.error(f"Failed to write {p}: {e}")
            return False

    @staticmethod
    def append_file(
        filepath:      Path | str,
        content:       str,
        add_timestamp: bool = True,
        encoding:      str  = "utf-8",
    ) -> bool:
        """Append *content* to *filepath*, optionally prefixed with a timestamp."""
        p = FileManager._resolve(filepath)
        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            if add_timestamp:
                ts      = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                content = f"[{ts}] {content}"
            with open(p, "a", encoding=encoding) as f:
                f.write(content + "\n")
            return True
        except Exception as e:
            logger.error(f"Failed to append to {p}: {e}")
            return False

    @staticmethod
    def delete_path(path: Path | str) -> bool:
        """Delete *path* (file or directory tree). Returns ``True`` on success."""
        p = FileManager._resolve(path)
        try:
            if p.is_file():
                p.unlink()
            elif p.is_dir():
                shutil.rmtree(p)
            return True
        except Exception as e:
            logger.error(f"Failed to delete {p}: {e}")
            return False

    @staticmethod
    def list_directory(path: Path | str, pattern: str = "*") -> list[dict[str, Any]]:
        """Return metadata dicts for every entry matching *pattern* under *path*."""
        p = FileManager._resolve(path)
        try:
            results: list[dict[str, Any]] = []
            # FIX: Use rglob instead of glob for recursive matching
            for item in p.rglob(pattern):
                try:
                    stat = item.stat()
                    results.append({
                        "name":     item.name,
                        "path":     str(item),
                        "type":     "dir" if item.is_dir() else "file",
                        "size":     stat.st_size,
                        "modified": datetime.fromtimestamp(stat.st_mtime).isoformat(),
                    })
                except (OSError, PermissionError):
                    continue
            return results
        except Exception as e:
            logger.error(f"Failed to list {p}: {e}")
            return []


# ---------- Workspace-Sandboxed File Manager ----------

class SecureFiles:
    """File operations strictly confined to ``Config.FILES_DIR``."""

    _MAX_BYTES: int = Config.MAX_FILE_SIZE_MB * 1024 * 1024

    # ── Path resolution ───────────────────────────────────────────────

    def _r(self, p: str | Path) -> Path:
        """
        Resolve *p* to an absolute path inside the workspace, raising
        ``ValueError`` if it escapes ``Config.FILES_DIR``.
        """
        p = Path(str(p).strip())
        if p.is_absolute():
            p = p.resolve()
        else:
            p = (Config.FILES_DIR / p).resolve()
        try:
            p.relative_to(Config.FILES_DIR.resolve())
        except ValueError:
            raise ValueError(
                f"Path '{p}' is outside the workspace. "
                f"Use relative paths or paths under {Config.FILES_DIR}"
            )
        return p

    def _safe_resolve(self, path: str | Path) -> Path | str:
        """Return a resolved ``Path`` or an error string on escape attempts."""
        try:
            return self._r(path)
        except ValueError as e:
            return f"Error: {e}"

    def _check_size(self, p: Path) -> str | None:
        """Return an error string if *p* exceeds the configured size limit, else ``None``."""
        try:
            mb = p.stat().st_size / (1024 * 1024)
        except OSError:
            return "Error: Cannot access file size"
        return f"Error: File too large ({mb:.1f} MB)" if mb > Config.MAX_FILE_SIZE_MB else None

    # ── File operations ───────────────────────────────────────────────

    def read(self, path: str, start_line: int = 0, end_line: int = 0) -> str:
        p = self._safe_resolve(path)
        if isinstance(p, str):
            return p
        if not p.exists():
            return f"Not found: {p}"
        if err := self._check_size(p):
            return err
        try:
            text = p.read_text(encoding="utf-8", errors="replace")
            if not (start_line or end_line):
                return text
            lines = text.splitlines(keepends=True)
            total = len(lines)
            s     = max(0, (start_line or 1) - 1)
            e     = end_line or total
            return f"[lines {s + 1}–{min(e, total)} of {total}]\n" + "".join(lines[s:e])
        except Exception as e:
            return f"Read error: {e}"

    def write(self, path: str, content: str, append: bool = False, backup: bool = False) -> str:
        p = self._safe_resolve(path)
        if isinstance(p, str):
            return p

        content_bytes = content.encode("utf-8")

        # FIX: Check total size when appending (existing + new content)
        if p.exists() and append:
            try:
                existing_size = p.stat().st_size
                total_size = existing_size + len(content_bytes)
                if total_size > self._MAX_BYTES:
                    return (
                        f"Error: Content too large (total would be {total_size:,} bytes; "
                        f"limit {self._MAX_BYTES:,} bytes)."
                    )
            except OSError:
                return "Error: Cannot check existing file size"
        else:
            if len(content_bytes) > self._MAX_BYTES:
                return (
                    f"Error: Content too large ({len(content_bytes):,} bytes; "
                    f"limit {self._MAX_BYTES:,} bytes)."
                )

        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            if backup and p.exists() and not append:
                p.with_suffix(p.suffix + ".bak").write_bytes(p.read_bytes())
            # Use an explicit context manager to guarantee the file is closed
            with p.open("a" if append else "w", encoding="utf-8") as fh:
                fh.write(content)
            return f"Written → {p} ({len(content):,} chars)"
        except Exception as e:
            return f"Write error: {e}"

    def write_many(self, writes: list[dict]) -> list[str]:
        """Write multiple files in parallel (up to 8 workers)."""
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
            return list(pool.map(
                lambda w: self.write(w["path"], w["content"], w.get("append", False)),
                writes,
            ))

    def list(self, directory: str = "", metadata: bool = False) -> Any:
        """List workspace files, optionally with size/date metadata."""
        try:
            base = self._r(directory) if directory else Config.FILES_DIR
        except ValueError:
            return [f"Error: path outside workspace: {directory}"]
        if not base.exists():
            return []

        entries = sorted(
            (f for f in base.rglob("*") if not f.name.startswith(".")),
            key=lambda x: str(x),
        )
        if not metadata:
            return [str(f.relative_to(Config.FILES_DIR)) for f in entries if f.is_file()]

        result: list[dict[str, Any]] = []
        for f in entries:
            try:
                st = f.stat()
                result.append({
                    "name":       str(f.relative_to(Config.FILES_DIR)),
                    "size_bytes": st.st_size,
                    "modified":   datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M"),
                    "is_dir":     f.is_dir(),
                })
            except (OSError, PermissionError):
                continue
        return result

    def delete(self, path: str) -> str:
        p = self._safe_resolve(path)
        if isinstance(p, str):
            return p
        if not p.exists():
            return "Not found"
        if p.resolve() == Config.FILES_DIR.resolve():
            return "Error: Cannot delete workspace root"
        try:
            p.unlink()
            return f"Deleted: {p}"
        except Exception as e:
            return f"Delete error: {e}"

    def search(self, query: str, directory: str = "") -> list[str]:
        """Return file paths whose name or content contains *query* (case-insensitive)."""
        q = query.lower()
        hits: list[str] = []
        try:
            base = self._r(directory) if directory else Config.FILES_DIR
        except ValueError:
            return []

        for fp in self.list(directory):
            # Skip error entries from list()
            if isinstance(fp, str) and fp.startswith("Error:"):
                continue

            # Check filename first (fast path)
            if q in fp.lower():
                hits.append(fp)
                continue

            # Check file content
            try:
                p = self._r(fp)  # Resolve to absolute path
                if p.stat().st_size > self._MAX_BYTES:
                    continue
                text = p.read_text(encoding="utf-8", errors="replace")
                if q in text.lower():
                    hits.append(fp)
            except (OSError, ValueError):
                continue
        return hits

    def rename(self, src: str, dst: str) -> str:
        s = self._safe_resolve(src)
        d = self._safe_resolve(dst)
        if isinstance(s, str):
            return s
        if isinstance(d, str):
            return d
        if not s.exists():
            return f"Error: not found: {src}"
        # FIX: Prevent renaming to same path
        if s.resolve() == d.resolve():
            return "Error: source and destination are the same"
        try:
            d.parent.mkdir(parents=True, exist_ok=True)
            s.rename(d)
            logger.info("file_rename", src=src, dst=dst)
            return f"Renamed {src} → {dst}"
        except Exception as e:
            return f"Rename error: {e}"


print("✓ File managers ready (SecureFiles + FileManager)")

✓ File managers ready (SecureFiles + FileManager)


## CELL 7: Core Components (Memory, Diff, Code Exec, Web)


In [31]:
# @title
# ---------- Persistent Memory ----------

import json
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any
import textwrap
import subprocess
import resource
import os
import sys
import tempfile
import shutil
import concurrent.futures
from urllib.parse import urlparse
import ipaddress


class EfficientMemory:
    """JSON-line key-value store with tagging, fuzzy search, and persistence."""

    def __init__(self) -> None:
        self._data: dict[str, dict[str, Any]] = {}
        self._load()

    # ── Persistence ──────────────────────────────────────────────────

    def _load(self) -> None:
        """Populate in-memory store from the JSON-lines file on disk."""
        if not Config.MEM_FILE.exists():
            return
        try:
            for line in Config.MEM_FILE.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    self._data[rec["key"]] = rec
        except Exception:
            # Corrupt file — try to recover partial records before wiping
            self._data = {}
            bak = Config.MEM_FILE.with_suffix(".bak")
            if bak.exists():
                try:
                    for line in bak.read_text(encoding="utf-8").splitlines():
                        if line.strip():
                            rec = json.loads(line)
                            self._data[rec["key"]] = rec
                    warnings.warn(f"memory.json corrupt — recovered {len(self._data)} records from .bak")
                except Exception:
                    self._data = {}  # Both corrupt — start fresh

    def _flush(self) -> None:
        """Atomically persist the store via a temp file, keeping a .bak backup."""
        tmp = Config.MEM_FILE.with_suffix(".tmp")
        bak = Config.MEM_FILE.with_suffix(".bak")
        try:
            content = "\n".join(json.dumps(r, default=str) for r in self._data.values())
            tmp.write_text(content, encoding="utf-8")
            # Backup previous good file before replacing
            if Config.MEM_FILE.exists():
                shutil.copy2(Config.MEM_FILE, bak)
            tmp.replace(Config.MEM_FILE)
        except Exception:
            tmp.unlink(missing_ok=True)
            raise

    # ── Public API ───────────────────────────────────────────────────

    def save(self, key: str, value: Any, tag: str = "") -> str:
        self._data[key] = {"key": key, "value": value, "tag": tag,
                           "ts": datetime.now().isoformat()}
        self._flush()
        return f"Saved '{key}'"

    def get(self, key: str) -> Any:
        """Return the stored value for *key*, or ``None`` if absent."""
        rec = self._data.get(key)
        return rec["value"] if rec is not None else None

    def list_keys(self, tag: str = "") -> list[str]:
        if not tag:
            return list(self._data)
        return [k for k, v in self._data.items() if v.get("tag") == tag]

    def search(self, query: str, fuzzy: bool = True) -> dict[str, Any]:
        """
        Return entries whose key or value contains *query*.
        When *fuzzy* is ``True`` (default) all words must be present but
        need not be contiguous.
        Falls back to keyword search if sklearn is unavailable.
        """
        q = query.lower()
        words = q.split()
        result: dict[str, Any] = {}
        for k, v in self._data.items():
            hay = f"{k} {v.get('value', '')}".lower()
            if q in hay or (fuzzy and all(w in hay for w in words)):
                result[k] = v["value"]
        return result

    def semantic_search(self, query: str, top_k: int = 5) -> list[dict[str, Any]]:
        """
        TF-IDF semantic search over all memory values.
        Returns list of {key, value, score} dicts sorted by relevance.
        Falls back to fuzzy keyword search if sklearn not installed.
        """
        if not self._data:
            return []
        try:
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.metrics.pairwise import cosine_similarity
            import numpy as np

            keys = list(self._data.keys())
            docs = [f"{k} {self._data[k].get('value', '')}" for k in keys]
            corpus = docs + [query]

            vec = TfidfVectorizer(stop_words="english", min_df=1)
            tfidf = vec.fit_transform(corpus)
            q_vec = tfidf[-1]
            scores = cosine_similarity(q_vec, tfidf[:-1]).flatten()

            top_idx = np.argsort(scores)[::-1][:top_k]
            return [
                {"key": keys[i], "value": self._data[keys[i]]["value"], "score": float(scores[i])}
                for i in top_idx if scores[i] > 0.0
            ]
        except ImportError:
            # sklearn not available — fall back to keyword search
            kw_results = self.search(query, fuzzy=True)
            return [{"key": k, "value": v, "score": 1.0} for k, v in list(kw_results.items())[:top_k]]

    def delete(self, key: str) -> str:
        if key not in self._data:
            return "Not found"
        del self._data[key]
        self._flush()
        return f"Deleted '{key}'"

    def clear(self) -> None:
        """Wipe all in-memory data and remove the backing file."""
        self._data = {}
        Config.MEM_FILE.unlink(missing_ok=True)


# ---------- Diff Applier ----------

class DiffApplier:
    """Apply SEARCH/REPLACE blocks or unified diffs to workspace files."""

    _SR_BLOCK = re.compile(
        r'<{7} SEARCH\r?\n(.*?)\r?\n={7}\r?\n(.*?)\r?\n>{7} REPLACE',
        re.DOTALL,
    )
    _HUNK_RE = re.compile(r'^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@')

    @staticmethod
    def _norm(s: str) -> str:
        """Strip trailing whitespace from every line for comparison."""
        return "\n".join(line.rstrip() for line in s.splitlines())

    def _resolve_path(self, path: str) -> Path | str:
        """Return a resolved workspace ``Path`` or an error string."""
        try:
            p = (Config.FILES_DIR / path).resolve()
            p.relative_to(Config.FILES_DIR.resolve())
            return p
        except ValueError:
            return f"Error: path outside workspace: {path}"

    # ── SEARCH/REPLACE ────────────────────────────────────────────────

    def _apply_search_replace(self, text: str, diff: str) -> tuple[str, int, list[str]]:
        blocks = self._SR_BLOCK.findall(diff)
        if not blocks:
            return text, 0, ["No SEARCH/REPLACE blocks found"]

        errors: list[str] = []
        applied = 0

        for search_raw, replace_raw in blocks:
            search_norm = self._norm(search_raw)
            text_norm = self._norm(text)

            idx = text_norm.find(search_norm)
            if idx == -1:
                errors.append(f"SEARCH block not found:\n{search_raw[:120]}")
                continue

            # Find line-based position in original text
            orig_lines = text.splitlines(keepends=True)
            # Count lines before the match in normalized text
            before_match = text_norm[:idx]
            pre_lines = before_match.count('\n')

            # Calculate byte positions in original text
            start = sum(len(line) for line in orig_lines[:pre_lines])
            # Number of lines in search block (in normalized version)
            search_lines = search_norm.count('\n') + (1 if not search_norm.endswith('\n') else 0)
            end = sum(len(line) for line in orig_lines[:pre_lines + search_lines])

            # Ensure replace ends with newline if search did
            tail = "" if replace_raw.endswith("\n") and search_raw.endswith("\n") else "\n"
            text = text[:start] + replace_raw + tail + text[end:]
            applied += 1

        return text, applied, errors

    # ── Unified diff ──────────────────────────────────────────────────

    def _apply_unified(
        self, lines: list[str], diff: str
    ) -> tuple[list[str], int, list[str]]:
        diff_lines = diff.splitlines(keepends=True)
        result = list(lines)
        errors: list[str] = []
        offset = applied = 0
        i = 0

        while i < len(diff_lines):
            if not (m := self._HUNK_RE.match(diff_lines[i])):
                i += 1
                continue
            orig_start = int(m.group(1)) - 1
            i += 1
            removes: list[str] = []
            adds: list[str] = []
            while i < len(diff_lines) and not self._HUNK_RE.match(diff_lines[i]):
                line = diff_lines[i]
                if line.startswith("-"):
                    removes.append(line[1:])
                elif line.startswith("+"):
                    adds.append(line[1:])
                i += 1

            pos = orig_start + offset
            if self._norm("".join(result[pos: pos + len(removes)])) != self._norm("".join(removes)):
                errors.append(f"Hunk mismatch at line {orig_start + 1}")
                continue
            result[pos: pos + len(removes)] = adds
            offset += len(adds) - len(removes)
            applied += 1

        return result, applied, errors

    # ── Entry point ───────────────────────────────────────────────────

    def apply(self, path: str, diff: str) -> str:
        p = self._resolve_path(path)
        if isinstance(p, str):
            return p
        if not p.exists():
            return f"Error: file not found: {path}"

        try:
            original = p.read_text(encoding="utf-8", errors="replace")
        except Exception as e:
            return f"Error reading file: {e}"

        if "<<<<<<< SEARCH" in diff:
            new_text, n, errors = self._apply_search_replace(original, diff)
            fmt = "search/replace"
        else:
            new_lines, n, errors = self._apply_unified(
                original.splitlines(keepends=True), diff
            )
            new_text = "".join(new_lines)
            fmt = "unified diff"

        if not n:
            return "Error — no blocks applied:\n" + "\n".join(errors)

        try:
            p.write_text(new_text, encoding="utf-8")
            delta = new_text.count("\n") - original.count("\n")
            warnings = f"\n  Warnings: {'; '.join(errors)}" if errors else ""
            return (
                f"✓ Patched {path} [{fmt}]  {n} block(s) applied"
                f"  Δ{delta:+d} lines{warnings}"
            )
        except Exception as e:
            return f"Error writing file: {e}"


# ---------- Secure Code Executor ----------

class SecureCodeExecutor:
    """
    Execute Python snippets in an isolated subprocess with JSON output
    parsing and optional memory resource limits.
    """

    _WRAPPER = textwrap.dedent("""\
        import json, re, math, random, statistics, collections, itertools
        import functools, datetime, string, csv, io
        from pathlib import Path

        try:
        {code}
            if 'result' in dir():
                print(json.dumps({{"result": result, "status": "success"}}, default=str))
            else:
                print(json.dumps({{"result": "OK (no output)", "status": "success"}}))
        except Exception as e:
            import traceback
            print(json.dumps({{"error": str(e), "traceback": traceback.format_exc(), "status": "error"}}))
    """)
    # Subprocess is given slightly more time than the user-facing timeout
    # to allow for process startup overhead.
    _SUBPROCESS_GRACE = 5  # seconds

    @staticmethod
    def _set_resource_limits() -> None:
        """Apply memory cap via ``RLIMIT_AS`` (Linux / macOS only)."""
        try:
            mem_bytes = Config.MAX_PYTHON_MEMORY_MB * 1024 * 1024
            resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
        except Exception:
            pass  # Silently skip on unsupported platforms

    @staticmethod
    def _lint(code: str) -> str | None:
        """
        Quick syntax + compile check before spawning a subprocess.
        Returns an error string on failure, None if code looks valid.
        """
        import ast as _ast
        try:
            _ast.parse(code)
        except SyntaxError as e:
            return f"SyntaxError at line {e.lineno}: {e.msg}\n  {e.text}"
        try:
            compile(code, "<string>", "exec")
        except Exception as e:
            return f"CompileError: {e}"
        return None

    @classmethod
    def run(cls, code: str, timeout: int | None = None, apply_limits: bool = True) -> str:
        # Fast lint pass — catches syntax errors without spawning a subprocess
        if lint_err := cls._lint(code):
            return f"Code rejected (lint failed):\n{lint_err}"
        timeout = timeout or Config.CODE_TIMEOUT
        wrapper = cls._WRAPPER.format(code=textwrap.indent(code, "    "))
        tmp_path: str | None = None
        try:
            with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
                tmp_path = f.name
                f.write(wrapper)
            preexec = cls._set_resource_limits if apply_limits else None
            result = subprocess.run(
                [sys.executable, tmp_path],
                capture_output=True, text=True,
                timeout=timeout + cls._SUBPROCESS_GRACE,
                cwd=str(Config.FILES_DIR),
                preexec_fn=preexec,
            )
            return cls._parse_output(result)
        except subprocess.TimeoutExpired:
            return f"Error: Timed out after {timeout}s"
        except Exception as e:
            return f"Execution failed: {e}"
        finally:
            if tmp_path:
                try:
                    os.unlink(tmp_path)
                except OSError:
                    pass

    @staticmethod
    def _parse_output(result: subprocess.CompletedProcess) -> str:
        try:
            out = json.loads(result.stdout.strip())
            if out.get("status") == "error":
                return f"Execution error: {out.get('error')}\n{out.get('traceback', '')}"
            return str(out.get("result", "No result"))
        except json.JSONDecodeError:
            parts = [result.stdout.strip()]
            if result.stderr.strip():
                parts.append(f"STDERR:\n{result.stderr.strip()}")
            return "\n".join(filter(None, parts)) or "OK (no output)"


# ---------- Web Search ----------

class SecureWebSearch:
    """DuckDuckGo search and URL fetcher with SSRF protection."""

    _BLOCKED_PREFIXES: tuple[str, ...] = (
        "localhost", "127.", "0.0.0.0", "::1",
        "10.", "172.16.", "192.168.", "169.254.",
    )
    _USER_AGENT = "Mozilla/5.0 (compatible; AgenticAI/1.0)"

    @classmethod
    def _valid_url(cls, url: str) -> bool:
        """Return ``True`` only for public http(s) URLs."""
        if not url.startswith(("http://", "https://")):
            return False

        try:
            parsed = urlparse(url)
            hostname = parsed.hostname
            if not hostname:
                return False

            # Check for IP addresses and block private/rfc1918 ranges
            try:
                ip = ipaddress.ip_address(hostname)
                if ip.is_loopback or ip.is_private or ip.is_link_local or ip.is_unspecified:
                    return False
            except ValueError:
                # Not an IP address, check for blocked hostname prefixes
                if any(hostname.startswith(b) for b in cls._BLOCKED_PREFIXES):
                    return False

            return True
        except Exception:
            return False

    @staticmethod
    def _import_requests():
        """Import *requests*, returning ``(module, None)`` or ``(None, error_str)``."""
        try:
            import requests as _r
            return _r, None
        except ImportError:
            return None, "Error: requests library not installed"

    def search(self, query: str, max_results: int = 5) -> list[dict[str, Any]]:
        requests, err = self._import_requests()
        if err:
            return [{"snippet": err, "url": ""}]
        try:
            r = requests.get(
                "https://api.duckduckgo.com/",
                timeout=Config.WEB_TIMEOUT,
                params={
                    "q": query, "format": "json",
                    "no_html": 1, "skip_disambig": 1, "no_redirect": 1,
                },
            ).json()

            results: list[dict[str, Any]] = []
            if r.get("Abstract"):
                results.append({
                    "title": r.get("Heading", query)[:100],
                    "snippet": r["Abstract"][:300],
                    "url": r.get("AbstractURL", ""),
                })
            for topic in r.get("RelatedTopics", [])[:max_results]:
                if isinstance(topic, dict) and topic.get("Text"):
                    results.append({
                        "title": topic["Text"][:80],
                        "snippet": topic["Text"][:300],
                        "url": topic.get("FirstURL", ""),
                    })
            return results[:max_results] or [{"snippet": "No results", "url": ""}]
        except Exception as e:
            return [{"snippet": f"Search error: {e}", "url": ""}]

    def fetch(self, url: str, max_chars: int | None = None) -> str:
        requests, err = self._import_requests()
        if err:
            return err
        max_chars = max_chars or Config.MAX_WEB_CONTENT
        if not self._valid_url(url):
            return "Error: Invalid or blocked URL"
        try:
            resp = requests.get(
                url, timeout=Config.WEB_TIMEOUT, stream=True,
                headers={"User-Agent": self._USER_AGENT},
            )
            resp.raise_for_status()
            if "text/" not in resp.headers.get("Content-Type", ""):
                return "Error: Non-text content"

            chunks: list[str] = []
            length = 0
            for chunk in resp.iter_content(chunk_size=8192, decode_unicode=True):
                if chunk:
                    chunks.append(chunk)
                    length += len(chunk)
                    if length > max_chars:
                        break

            raw = "".join(chunks)
            text = re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', raw)).strip()
            return text[:max_chars] + ("...(truncated)" if len(text) > max_chars else "")
        except Exception as e:
            return f"Fetch error: {e}"


print("✓ Core components ready (Memory, Diff, Code Exec, Web)")

✓ Core components ready (Memory, Diff, Code Exec, Web)


## CELL 8: Sub-tools & Skills Manager

In [32]:
# @title
# ---------- Cached Subtools & Skills ----------

import warnings
import ast
import re
import json
import datetime
import textwrap
import importlib
import traceback
from pathlib import Path
from typing import Any


class CachedSubtools:
    """
    Code-1 sub-tool system (create/run/list/delete dynamic Python tools)
    extended with Code-2's skill validation and Tools.md registration.
    """

    BLOCKED = re.compile(
        r'import\s+(os|sys|subprocess)|__import__|eval\s*\(|exec\s*\('
        r'|open\s*\(|compile\s*\(|getattr\s*\(|setattr\s*\(|delattr\s*\('
        r'|globals\s*\(|locals\s*\(|\.system\s*\(|\.popen\s*\(|subprocess\.',
        re.IGNORECASE,
    )
    _SOFT_PATTERNS: list[tuple[re.Pattern, str]] = [
        (re.compile(p, f), p) for p, f in [
            (r'__\w+__',         re.IGNORECASE),
            (r'ctypes',          re.IGNORECASE),
            (r'socket\.',        0),
            (r'urllib\.request', 0),
            (r'pickle\.',        re.IGNORECASE),
            (r'importlib',       0),
            (r'\.write\s*\(',    0),
            (r'shutil\.',        0),
        ]
    ]
    # Limits used in security review
    _MAX_LINES   = 300
    _MAX_IMPORTS = 20
    # Fields exposed by list()
    _LIST_FIELDS = ("description", "usage_count", "last_used", "created")
    # Fields parsed from Tools.md skill blocks
    _SKILL_ATTRS = ("type", "location", "purpose", "trigger", "input", "output")

    def __init__(self) -> None:
        self._mf: Path              = Config.SUBTOOL_DIR / "manifest.json"
        self._m:  dict[str, Any]    = {}
        self._cache: dict[str, Any] = {}
        if self._mf.exists():
            try:
                self._m = json.loads(self._mf.read_text(encoding="utf-8"))
            except Exception:
                # Corrupt manifest — try backup before starting fresh
                bak = self._mf.with_suffix(".json.bak")
                if bak.exists():
                    try:
                        self._m = json.loads(bak.read_text(encoding="utf-8"))
                        warnings.warn(f"manifest.json corrupt — recovered {len(self._m)} records from .bak")
                    except Exception:
                        pass  # Both corrupt — start fresh

    # ── Persistence ──────────────────────────────────────────────────

    def _save(self) -> None:
        """Atomically persist the manifest via a temp file, keeping a .bak backup."""
        tmp = self._mf.with_suffix(".json.tmp")
        bak = self._mf.with_suffix(".json.bak")
        try:
            tmp.write_text(json.dumps(self._m, indent=2, default=str), encoding="utf-8")
            # Keep backup of previous good manifest before replacing
            if self._mf.exists():
                import shutil as _sh
                _sh.copy2(self._mf, bak)
            tmp.replace(self._mf)
        except Exception:
            tmp.unlink(missing_ok=True)
            raise

    # ── Security ─────────────────────────────────────────────────────

    def _security_review(self, code: str) -> list[str]:
        """Return a list of human-readable security warnings for *code*."""
        warnings_list: list[str] = [
            f"Pattern '{pat}' matched: {m.group()!r}"
            for rx, pat in self._SOFT_PATTERNS
            if (m := rx.search(code))
        ]
        lines = code.splitlines()
        if len(lines) > self._MAX_LINES:
            warnings_list.append(f"Code is large ({len(lines)} lines)")
        if code.count("import") > self._MAX_IMPORTS:
            warnings_list.append("Excessive imports detected")
        return warnings_list

    # ── Core sub-tool CRUD ────────────────────────────────────────────

    def create(self, name: str, description: str, code: str, force: bool = False) -> str:
        name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
        if not name or not name[0].isalpha():
            return "Error: Invalid tool name"
        if m := self.BLOCKED.search(code):
            logger.warning("subtool blocked", name=name, pattern=m.group())
            return f"Error: Code contains blocked pattern: {m.group()!r}"
        if "def run(" not in code:
            return "Error: Code must define run(**kwargs)"

        warnings_list = self._security_review(code)
        if warnings_list and not force:
            logger.warning("subtool security warnings", name=name, warnings=str(warnings_list))
            return (
                "Security review found potential issues:\n"
                + "\n".join(f"  ⚠ {w}" for w in warnings_list)
                + "\nCall again with force=True to create despite warnings."
            )

        p = Config.SUBTOOL_DIR / f"{name}.py"
        p.write_text(textwrap.dedent(code), encoding="utf-8")
        self._m[name] = {
            "description":       description,
            "path":              str(p),
            "created":           datetime.datetime.now().isoformat(),
            "usage_count":       0,
            "last_used":         None,
            "security_warnings": warnings_list,
        }
        self._cache.pop(name, None)
        self._save()

        warn_note = f"  ⚠ {len(warnings_list)} warning(s) recorded." if warnings_list else ""
        logger.info("subtool created", name=name, force=str(force))
        return f"✓ Sub-tool '{name}' created{warn_note}"

    def run(self, name: str, **kwargs: Any) -> Any:
        if name not in self._m:
            return f"Not found: {name}"
        p = Path(self._m[name]["path"])
        if not p.exists():
            return f"Error: File missing for '{name}'"
        try:
            spec = importlib.util.spec_from_file_location(name, p)
            mod  = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            if not hasattr(mod, "run"):
                return f"Error: '{name}' has no run() function"
            result = mod.run(**kwargs)
            self._m[name]["usage_count"] = self._m[name].get("usage_count", 0) + 1
            self._m[name]["last_used"]   = datetime.datetime.now().isoformat()
            self._save()
            return result if isinstance(result, str) else json.dumps(result, default=str)
        except Exception as e:
            return f"Subtool error: {e}\n{traceback.format_exc()}"

    def list(self) -> dict[str, dict[str, Any]]:
        return {
            k: {f: v.get(f) for f in self._LIST_FIELDS}
            for k, v in self._m.items()
        }

    def code(self, name: str) -> str:
        if name not in self._m:
            return "Not found"
        p = Path(self._m[name]["path"])
        return p.read_text(encoding="utf-8") if p.exists() else "File missing"

    def delete(self, name: str) -> str:
        if name not in self._m:
            return "Not found"
        Path(self._m[name]["path"]).unlink(missing_ok=True)
        del self._m[name]
        self._cache.pop(name, None)
        self._save()
        return f"Deleted '{name}'"

    # ── Skills helpers ────────────────────────────────────────────────

    @staticmethod
    def _load_skill_module(name: str, filepath: str | Path) -> tuple[Any, str | None]:
        """
        Import a skill module from *filepath*.
        Returns ``(module, None)`` on success or ``(None, error_message)`` on failure.
        """
        try:
            spec = importlib.util.spec_from_file_location(name, filepath)
            if spec is None:
                return None, "Cannot load skill module"
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod, None
        except Exception as e:
            return None, str(e)

    def _parse_skills_from_tools_md(self, content: str) -> list[dict[str, Any]]:
        """Parse skill blocks from a Tools.md file string."""
        skills: list[dict[str, Any]] = []
        current: dict[str, Any] | None = None

        for raw_line in content.split("\n"):
            line = raw_line.strip()
            if line.startswith("## Tool:"):
                if current and current.get("type") == "skill":
                    skills.append(current)
                current = {"name": line.removeprefix("## Tool:").strip(),
                           **{a: "" for a in self._SKILL_ATTRS}}
            elif current:
                if line.startswith("#"):
                    # Any new heading closes the current block
                    if current.get("type") == "skill":
                        skills.append(current)
                    current = None
                    continue
                low = line.lower()
                for attr in self._SKILL_ATTRS:
                    if low.startswith(f"- {attr}") or low.startswith(f"- **{attr}**"):
                        current[attr] = line.split(":", 1)[1].strip() if ":" in line else ""
                        break

        if current and current.get("type") == "skill":
            skills.append(current)
        return skills

    # ── Skill-manager extensions ──────────────────────────────────────

    def list_skills(self, refresh: bool = False) -> list[dict[str, Any]]:
        """List skills registered in core/Tools.md."""
        tools_content = FileManager.read_file(Config.CORE_DIR / "Tools.md", default="")
        skills = self._parse_skills_from_tools_md(tools_content)

        for skill in skills:
            loc  = skill.get("location", "")
            path = Config.SKILLS_DIR / loc.removeprefix("skills/") if loc else None
            skill["file_exists"] = path is not None and path.exists()
            skill["file_path"]   = str(path) if skill["file_exists"] else None

        return skills

    def validate_skill(self, name: str) -> dict[str, Any]:
        """Validate a named skill."""
        status: dict[str, Any] = {
            "name": name, "valid": False,
            "file_exists": False, "has_run": False, "error": None,
        }
        skill = next((s for s in self.list_skills() if s["name"] == name), None)
        if not skill:
            status["error"] = "Skill not registered in Tools.md"
            return status

        fp = skill.get("file_path")
        if not fp:
            status["error"] = "Skill file not found"
            return status

        status["file_exists"] = True

        try:
            with open(fp, 'r', encoding='utf-8') as f:
                code = f.read()
        except Exception as e:
            status["error"] = f"Error reading file: {e}"
            return status

        # Check for run function using AST (safe, no execution)
        try:
            tree = ast.parse(code)
            for node in tree.body:
                if isinstance(node, ast.FunctionDef) and node.name == 'run':
                    status["has_run"] = True
                    break
            else:
                status["error"] = "Missing run() function"
                return status
        except SyntaxError as e:
            status["error"] = f"Syntax error in skill code: {e}"
            return status

        # Run security review (no execution)
        warnings_list = self._security_review(code)
        if warnings_list:
            status["security_warnings"] = warnings_list

        status["valid"] = status["has_run"]
        return status

    def validate_all_skills(self) -> dict[str, list[dict[str, Any]]]:
        """Validate all registered skills."""
        valid: list[dict[str, Any]]   = []
        invalid: list[dict[str, Any]] = []
        for skill in self.list_skills():
            v = self.validate_skill(skill["name"])
            if v["valid"]:
                valid.append(skill)
            else:
                skill["validation_error"] = v["error"]
                invalid.append(skill)
        return {"valid": valid, "invalid": invalid}

    def create_skill(self, name: str, code: str, purpose: str, trigger: str,
                     input_fmt: str = "", output_fmt: str = "", force: bool = False) -> str:
        """Create a skill file and register it in core/Tools.md."""
        if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', name):
            return f"Error: Invalid skill name '{name}'"

        # Check for duplicate skill name in Tools.md
        existing_skills = self.list_skills()
        if any(s["name"] == name for s in existing_skills):
            return f"Error: Skill '{name}' already registered in Tools.md"

        skill_path = Config.SKILLS_DIR / f"{name}.py"
        if skill_path.exists():
            return f"Error: Skill '{name}' already exists. Delete first."

        if m := self.BLOCKED.search(code):
            logger.warning("skill blocked", name=name, pattern=m.group())
            return f"Error: Code contains blocked pattern: {m.group()!r}"

        if "def run(" not in code:
            return "Error: Code must define run(**kwargs)"

        warnings_list = self._security_review(code)
        if warnings_list and not force:
            logger.warning("skill security warnings", name=name, warnings=str(warnings_list))
            return (
                "Security review found potential issues:\n"
                + "\n".join(f"  ⚠ {w}" for w in warnings_list)
                + "\nCall again with force=True to create despite warnings."
            )

        if not FileManager.write_file(skill_path, code):
            return "Error: Failed to write skill file"

        tools_path    = Config.CORE_DIR / "Tools.md"
        tools_content = FileManager.read_file(tools_path, default="")
        entry = (
            f"\n## Tool: {name}\n- Type     : skill\n"
            f"- Location : skills/{name}.py\n- Purpose  : {purpose}\n"
            f"- Trigger  : {trigger}\n- Input    : {input_fmt}\n"
            f"- Output   : {output_fmt}\n"
        )

        if "## Skill Scripts" in tools_content:
            head, tail  = tools_content.split("## Skill Scripts", 1)
            new_content = head + "## Skill Scripts\n" + entry + tail.lstrip()
        else:
            new_content = tools_content + "\n" + entry

        if not FileManager.write_file(tools_path, new_content):
            skill_path.unlink(missing_ok=True)
            return "Error: Failed to register skill in Tools.md"

        logger.info(f"Created skill: {name}")
        return f"✓ Skill '{name}' created and registered"

    def run_skill(self, name: str, input_data: Any = None) -> Any:
        """Run a skill from the skills directory."""
        skill_file = Config.SKILLS_DIR / f"{name}.py"
        if not skill_file.exists():
            return f"Error: Skill '{name}' not found"

        mod, err = self._load_skill_module(name, skill_file)
        if err:
            return f"Skill error: {err}"
        if not hasattr(mod, "run"):
            return f"Error: Skill '{name}' has no run() function"

        try:
            # Support both run(input_data) and run(**kwargs) signatures
            if isinstance(input_data, dict):
                try:
                    return mod.run(**input_data)
                except TypeError:
                    return mod.run(input_data)
            return mod.run(input_data)
        except Exception as e:
            logger.error(f"Skill {name} execution failed: {e}")
            return f"Skill error: {e}\n{traceback.format_exc()}"


print("✓ Sub-tools & Skills manager ready")

✓ Sub-tools & Skills manager ready


## CELL 9: Shell, Session, Need, Todo Managers

In [33]:
# @title
# ---------- Shell Runner ----------

import re
import subprocess
import shlex
import os
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any


class ShellRunner:
    """Run shell commands with a safety blocklist and configurable timeout."""

    _HARD_BLOCKED = re.compile(
        r'\brm\s+-r[f]?\b'
        r'|\bdd\b|\bmkfs\b'
        r'|:\(\)\s*\{.*:\(\)|:\s*&\s*\}'
        r'|\b(?:shutdown|reboot|halt|poweroff)\b'
        r'|\bchmod\s+-R\b'
        r'|\b(?:iptables|ufw)\b'
        r'|\bcrontab\s+-r\b',
        re.IGNORECASE | re.DOTALL,
    )
    _DISABLED: bool = os.environ.get("AGENT_DISABLE_SHELL", "").lower() in ("1", "true", "yes")

    @classmethod
    def _resolve_cwd(cls, cwd: str) -> str:
        """Map the *cwd* shorthand to an absolute directory string.

        All paths (including '/') are validated against the workspace root
        to prevent path traversal attacks.
        """
        candidate = Config.BASE_DIR if cwd == "/" else Config.FILES_DIR / cwd if cwd else Config.FILES_DIR

        if cwd and cwd != "/" and os.path.isabs(cwd):
            logger.warning(f"Absolute cwd '{cwd}' rejected - using workspace root")
            return str(Config.FILES_DIR)

        p = Path(candidate).resolve()
        try:
            p.relative_to(Config.FILES_DIR.resolve())
        except ValueError:
            logger.warning(f"cwd '{cwd}' resolves outside workspace - using workspace root")
            return str(Config.FILES_DIR)
        return str(p)

    @classmethod
    def run(cls, command: str, timeout: int | None = None, cwd: str = "") -> str:
        timeout = Config.SHELL_TIMEOUT if timeout is None else timeout

        if cls._DISABLED:
            return "Error: shell_run is disabled (AGENT_DISABLE_SHELL=1)."
        if m := cls._HARD_BLOCKED.search(command):
            logger.warning("shell_run blocked", command=command[:400], matched=m.group())
            return f"Error: Command blocked by safety filter (matched: {m.group()!r})."
        try:
            r = subprocess.run(
                shlex.split(command),
                shell=False,
                capture_output=True,
                text=True,
                timeout=timeout,
                cwd=cls._resolve_cwd(cwd),
                env={**os.environ, "PYTHONUNBUFFERED": "1"},
            )
            parts = [s for s in (
                r.stdout.strip(),
                f"[stderr]\n{r.stderr.strip()}" if r.stderr.strip() else "",
            ) if s]
            parts.append(f"[exit code: {r.returncode}]")
            return "\n".join(parts)
        except subprocess.TimeoutExpired:
            return f"Error: Command timed out after {timeout}s"
        except Exception as e:
            return f"Error: {e}"


# ---------- Session Manager ----------

class SessionManager:
    """Daily log files, multi-day memory search, and core-context loading."""

    # Core identity files loaded into every system prompt
    _CONTEXT_FILES: tuple[str, ...] = (
        "core/SOUL.md", "core/USER.md", "core/MEMORY.md",
        "core/HEARTBEAT.md", "core/Need.md", "core/Tools.md",
    )

    @staticmethod
    def get_todays_log_path() -> Path:
        today = datetime.now().strftime("%Y-%m-%d")
        return Config.MEMORY_DIR / f"{today}.md"

    @staticmethod
    def ensure_todays_log() -> Path:
        """Create today's log file if absent and return its path."""
        now  = datetime.now()
        path = Config.MEMORY_DIR / f"{now.strftime('%Y-%m-%d')}.md"
        if not path.exists():
            header = (
                f"# Daily Log - {now.strftime('%Y-%m-%d')}\n\n"
                f"## [{now.strftime('%H:%M:%S')}] Session Start\n"
                f"- Agent initialized\n"
            )
            FileManager.write_file(path, header)
        return path

    @staticmethod
    def save_log(entry: str, category: str = "General") -> bool:
        path = SessionManager.ensure_todays_log()
        ts   = datetime.now().strftime("%H:%M:%S")
        return FileManager.append_file(
            path, f"## [{ts}] {category}\n- {entry}", add_timestamp=False
        )

    @staticmethod
    def search_memory(query: str, days_back: int = 7) -> list[dict[str, Any]]:
        """Search the last *days_back* daily logs for *query* (case-insensitive)."""
        q       = query.lower()
        results: list[dict[str, Any]] = []
        try:
            for i in range(days_back):
                date = (datetime.now() - timedelta(days=i)).strftime("%Y-%m-%d")
                path = Config.MEMORY_DIR / f"{date}.md"
                if not path.exists():
                    continue
                content = FileManager.read_file(path)
                if q not in content.lower():
                    continue
                matches = [
                    {"line_num": n + 1, "text": line.strip()}
                    for n, line in enumerate(content.splitlines())
                    if q in line.lower()
                ][:10]
                results.append({"date": date, "matches": matches})
        except Exception as e:
            logger.error(f"Memory search failed: {e}")
        return results

    @staticmethod
    def load_context() -> dict[str, str]:
        """Load all core identity files into a ``{relative_path: content}`` dict."""
        return {
            rel: FileManager.read_file(Config.BASE_DIR / rel, default=f"# Missing: {rel}")
            for rel in SessionManager._CONTEXT_FILES
        }


# ---------- Need Management ----------

_NEED_FIELDS: tuple[str, ...] = ("Priority", "Needs", "Reason", "Blocking", "Status")
_NEED_REQUIRED: frozenset[str] = frozenset({"priority", "needs", "status"})


def post_need(priority: str, need: str, reason: str, blocking: str) -> bool:
    """Append a structured PENDING request to core/Need.md."""
    path    = Config.CORE_DIR / "Need.md"
    content = FileManager.read_file(path, default="")
    nums    = re.findall(r'Request #(\d+)', content)
    nxt     = max(int(x) for x in nums) + 1 if nums else 1
    now     = datetime.now().strftime("%Y-%m-%d %H:%M")
    entry   = (
        f"\n## [{now}] Request #{nxt}\n"
        f"- **Priority** : {priority}\n"
        f"- **Needs**    : {need}\n"
        f"- **Reason**   : {reason}\n"
        f"- **Blocking** : {blocking}\n"
        f"- **Status**   : PENDING\n"
    )
    return FileManager.append_file(path, entry, add_timestamp=False)


def check_needs() -> list[dict[str, str]]:
    """Return all valid PENDING requests from core/Need.md."""
    content = FileManager.read_file(Config.CORE_DIR / "Need.md", default="")
    pending: list[dict[str, str]] = []
    for block in content.split("## [")[1:]:
        entry: dict[str, str] = {}
        for field in _NEED_FIELDS:
            if m := re.search(rf'\*\*{field}\*\*\s*:\s*(.+)', block):
                entry[field.lower()] = m.group(1).strip()
        if entry.get("status") == "PENDING" and _NEED_REQUIRED.issubset(entry):
            pending.append(entry)
    return pending


# ---------- Todo List Manager ----------

class TodoListManager:
    """Hierarchical task tracker backed by ``EfficientMemory``."""

    _DONE_STATUSES: frozenset[str] = frozenset({"completed", "failed"})

    def __init__(self, memory: EfficientMemory) -> None:
        self.mem  = memory
        self._key = "current_todo_list"

    # ── Persistence ──────────────────────────────────────────────────

    @staticmethod
    def _empty() -> dict[str, Any]:
       return {"task": "", "tasks": [], "current_task_id": None, "completed": False}

    def _load(self) -> dict[str, Any]:
        data = self.mem.get(self._key)
        return data if isinstance(data, dict) else self._empty()

    def _save(self, data: dict[str, Any]) -> None:
        self.mem.save(self._key, data, tag="todo")

    # ── Tree helpers ─────────────────────────────────────────────────

    def _find(self, tasks: list[dict], task_id: str) -> dict | None:
        """Recursively locate a task by ID in the tree."""
        for t in tasks:
            if t["id"] == task_id:
                return t
            if found := self._find(t.get("children", []), task_id):
                return found
        return None

    def _flatten_ids(self, tasks: list[dict]) -> list[str]:
        """Return all task IDs in depth-first order."""
        ids: list[str] = []
        for t in tasks:
            ids.append(t["id"])
            if t.get("children"):
                ids.extend(self._flatten_ids(t["children"]))
        return ids

    def _count(self, tasks: list[dict], status: str | None = None) -> int:
        """Count tasks (and their children) matching *status*, or all if ``None``."""
        total = 0
        for t in tasks:
            total += 1 if status is None or t.get("status") == status else 0
            total += self._count(t.get("children", []), status)
        return total

    def _next_pending(self, tasks: list[dict], after_id: str) -> dict | None:
        """Return the first pending task that appears after *after_id* in DFS order."""
        flat = self._flatten_ids(tasks)
        try:
            start = flat.index(after_id) + 1
        except ValueError:
            return None
        for tid in flat[start:]:
            if (t := self._find(tasks, tid)) and t["status"] == "pending":
                return t
        return None

    @staticmethod
    def _make_task(task_id: str, description: str) -> dict[str, Any]:
        return {
            "id": task_id, "description": description, "status": "pending",
            "result": None, "error": None, "notes": "", "children": [],
        }

    # ── Public API ───────────────────────────────────────────────────

    def create(self, task: str, steps: list[str]) -> str:
        tasks = [self._make_task(str(i), s) for i, s in enumerate(steps, 1)]
        data  = {
            "task":            task,
            "tasks":           tasks,
            "completed":       False,
            "current_task_id": tasks[0]["id"] if tasks else None,
            "created":         datetime.now().isoformat(),
        }
        self._save(data)
        cur = tasks[0]["description"] if tasks else "none"
        return f"✓ Created todo list with {len(steps)} steps. Current: {cur}"

    def status(self) -> dict[str, Any]:
        data = self._load()
        if not data["tasks"]:
            return {"error": "No active todo list"}
        total     = self._count(data["tasks"])
        completed = self._count(data["tasks"], "completed")
        pct       = f"{completed / total * 100:.1f}%" if total else "0%"
        return {
            "task":         data["task"],
            "created":      data.get("created"),
            "completed":    data["completed"],
            "progress":     f"{completed}/{total} completed ({pct})",
            "current_task": (
                self._find(data["tasks"], data["current_task_id"])
                if data["current_task_id"] else None
            ),
            "tasks": data["tasks"],
            "stats": {
                s: self._count(data["tasks"], s)
                for s in ("completed", "failed", "in_progress", "pending")
            },
        }

    def add_subtask(self, parent_id: str, description: str) -> str:
        data   = self._load()
        parent = self._find(data["tasks"], parent_id)
        if not parent:
            return f"Error: Parent task '{parent_id}' not found"
        if parent.get("status") == "completed":
            return f"Error: Cannot add subtask to completed task '{parent_id}'"
        children = parent.setdefault("children", [])
        new_id   = f"{parent_id}.{len(children) + 1}"
        children.append(self._make_task(new_id, description))
        data["current_task_id"] = new_id
        self._save(data)
        return f"✓ Added subtask '{description}' (ID: {new_id}) → parent '{parent_id}'. Now current."

    def _finish_task(
        self, task_id: str, new_status: str,
        result: str | None = None, error: str | None = None,
    ) -> str:
        """Shared implementation for ``complete`` and ``fail``."""
        data = self._load()
        task = self._find(data["tasks"], task_id)
        if not task:
            return f"Error: Task '{task_id}' not found"
        if task["status"] == new_status:
            return f"Task '{task_id}' already {new_status}"

        task["status"] = new_status
        if result is not None:
            task["result"] = result
        if error is not None:
            task["error"] = error

        icon = "✓" if new_status == "completed" else "✗"
        nxt  = self._next_pending(data["tasks"], task_id)
        if nxt:
            data["current_task_id"] = nxt["id"]
            msg = f"{icon} {new_status.title()} '{task_id}'. Next: {nxt['id']} - {nxt['description']}"
        else:
            data["current_task_id"] = None
            data["completed"]       = True
            tail = "ALL TASKS DONE! 🎉" if new_status == "completed" else "NO MORE TASKS."
            msg  = f"{icon} {new_status.title()} '{task_id}'. {tail}"

        self._save(data)
        return msg

    def complete(self, task_id: str, result: str | None = None) -> str:
        return self._finish_task(task_id, "completed", result=result)

    def fail(self, task_id: str, error: str | None = None) -> str:
        return self._finish_task(task_id, "failed", error=error)

    def set_current(self, task_id: str) -> str:
        data = self._load()
        task = self._find(data["tasks"], task_id)
        if not task:
            return f"Error: Task '{task_id}' not found"
        if task["status"] in self._DONE_STATUSES:
            return f"Error: Task '{task_id}' is {task['status']}"
        data["current_task_id"] = task_id
        self._save(data)
        return f"Current task set to {task_id}: {task['description']}"

    def update(self, task_id: str, description: str | None = None, notes: str | None = None) -> str:
        data = self._load()
        task = self._find(data["tasks"], task_id)
        if not task:
            return f"Error: Task '{task_id}' not found"
        if description is not None:
            task["description"] = description
        if notes is not None:
            task["notes"] = notes
        self._save(data)
        return f"✓ Updated task {task_id}"

    def get_current_task(self) -> dict | None:
        data = self._load()
        return (
            self._find(data["tasks"], data["current_task_id"])
            if data["current_task_id"] else None
        )

    def is_completed(self) -> bool:
        data = self._load()
        return bool(data["completed"]) and data["current_task_id"] is None


print("✓ Shell, Session, Need, Todo managers ready")

✓ Shell, Session, Need, Todo managers ready


## CELL 10: Parallel API Handler

In [34]:
# @title
# ---------- Data Classes ----------

@dataclass
class ParallelTask:
    """A single unit of work submitted to ``ParallelAPIHandler``."""
    task_id:    str
    messages:   list[dict[str, Any]]
    system:     str              = ""
    tools:      list[dict]       = field(default_factory=list)
    model:      str              = ""
    # FIX: Use None as sentinel so 0 is never confused with "unset"
    max_tokens: int | None       = None


@dataclass
class ParallelResult:
    """The outcome of a single parallel API call."""
    task_id:    str
    content:    str
    tool_calls: list[dict[str, Any]] = field(default_factory=list)
    error:      str                  = ""
    tokens_in:  int                  = 0
    tokens_out: int                  = 0


# ---------- Parallel API Handler ----------

class ParallelAPIHandler:
    """Execute multiple ``ParallelTask`` objects concurrently via a thread pool.

    Note: ``APICounter`` and ``RateLimiter`` must be thread-safe, as
    ``_single_request`` is invoked from multiple threads simultaneously.
    """

    # ANSI colour shorthands used in progress output
    _DIM   = "\033[90m"
    _THINK = "\033[35m"
    _RESP  = "\033[36m"
    _ERR   = "\033[91m"
    _RESET = "\033[0m"

    def __init__(
        self,
        client:       Any,
        counter:      APICounter,
        rate_limiter: RateLimiter | None = None,
        max_workers:  int | None         = None,
    ) -> None:
        self.client       = client
        self.counter      = counter
        self.rate_limiter = rate_limiter
        self.max_workers  = max_workers or Config.MAX_PARALLEL_REQUESTS

    # ── Internal helpers ─────────────────────────────────────────────

    @staticmethod
    def _ts() -> str:
        """Return a millisecond-precision timestamp string for log lines."""
        return datetime.now().strftime("%H:%M:%S.%f")[:-3]

    @staticmethod
    def _extract_tokens(usage: Any) -> tuple[int, int, int]:
        """
        Return ``(tokens_in, tokens_reasoning, tokens_out)`` from a usage object.
        Mirrors the extraction logic in ``Agent._extract_usage``.
        """
        if not usage:
            return 0, 0, 0
        details    = getattr(usage, "completion_tokens_details", None)
        tokens_in  = getattr(usage, "prompt_tokens",      0) or 0
        reasoning  = getattr(details, "reasoning_tokens", 0) or 0
        tokens_out = max((getattr(usage, "completion_tokens", 0) or 0) - reasoning, 0)
        return tokens_in, reasoning, tokens_out

    @staticmethod
    def _serialize_tool_calls(raw_calls: Any) -> list[dict[str, Any]]:
        """Convert raw tool-call objects to plain dicts, skipping malformed entries."""
        if not raw_calls:
            return []
        result = []
        for tc in raw_calls:
            # FIX: Guard against malformed tool call objects missing expected attrs
            try:
                result.append({
                    "id":   tc.id,
                    "type": "function",
                    "function": {
                        "name":      tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                })
            except AttributeError as e:
                logger.warning(f"Skipping malformed tool call: {e}")
        return result

    # ── Single request ────────────────────────────────────────────────

    @retry_api()
    @retry_api()
    def _call_api(self, task: ParallelTask, msgs: list[dict], model: str, max_tokens: int) -> Any:
        """
        Raw API call with fallback support.
        If the primary model fails with a fallback-eligible error, retries with FALLBACK_MODEL.
        """
        def should_fallback(exc: Exception) -> bool:
            err = str(exc).lower()
            triggers = [
                '404', 'not found', 'model not found', 'deleted', 'unavailable',
                '429', 'rate limit', 'too many', 'quota',
                '500', '502', '503', '504', 'server error', 'overloaded',
                'timeout', 'connection', 'network',
            ]
            return any(t in err for t in triggers)

        # First attempt with the requested model
        if self.rate_limiter:
            self.rate_limiter.wait_if_needed()
        try:
            return self.client.chat.completions.create(
                model=model,
                max_tokens=max_tokens,
                messages=msgs,
                tools=task.tools or None,
                tool_choice="auto" if task.tools else None,
            )
        except Exception as e:
            # Determine if we should fall back and haven't already used a non-primary model
            if should_fallback(e) and (not task.model or task.model == Config.MODEL):
                # Try fallback model
                fallback_model = Config.FALLBACK_MODEL
                if self.rate_limiter:
                    self.rate_limiter.wait_if_needed()
                try:
                    return self.client.chat.completions.create(
                        model=fallback_model,
                        max_tokens=max_tokens,
                        messages=msgs,
                        tools=task.tools or None,
                        tool_choice="auto" if task.tools else None,
                    )
                except Exception as e2:
                    # Combine errors
                    raise Exception(f"Primary model failed: {e}. Fallback model {fallback_model} failed: {e2}")
            else:
                raise
    def _single_request(self, task: ParallelTask) -> ParallelResult:
        model      = task.model or Config.MODEL
        # FIX: Explicit None check — max_tokens=0 would previously fall through
        # to Config.MAX_TOKENS silently; now 0 is treated as a valid value.
        max_tokens = Config.MAX_TOKENS if task.max_tokens is None else task.max_tokens
        msgs       = (
            [{"role": "system", "content": task.system}] if task.system else []
        ) + task.messages

        req_num = self.counter.next_request_number()
        t0      = time.monotonic()
        print(
            f"  {self._DIM}[parallel] → REQ #{req_num} start"
            f"  {task.task_id}  {self._ts()}{self._RESET}"
        )

        try:
            resp                             = self._call_api(task, msgs, model, max_tokens)
            elapsed                          = time.monotonic() - t0
            msg                              = resp.choices[0].message
            tokens_in, reasoning, tokens_out = self._extract_tokens(resp.usage)

            self.counter.record(
                model=model,
                tokens_in=tokens_in,
                tokens_out=tokens_out,
                tokens_reasoning=reasoning,
            )
            print(
                f"  {self._DIM}[parallel] ← REQ #{req_num} done "
                f"  {task.task_id}  {self._ts()}"
                f"  {self._THINK}🧠{reasoning:,}{self._DIM}"
                f"  {self._RESP}💬{tokens_out:,}{self._DIM}"
                f"  {elapsed:.1f}s{self._RESET}"
            )
            return ParallelResult(
                task_id=task.task_id,
                content=msg.content or "",
                tool_calls=self._serialize_tool_calls(msg.tool_calls),
                tokens_in=tokens_in,
                tokens_out=tokens_out,
            )

        except Exception as e:
            elapsed = time.monotonic() - t0
            self.counter.record(model=model, error=True)
            print(
                f"  {self._ERR}[parallel] ✗ REQ #{req_num} error"
                f"  {task.task_id}  {elapsed:.1f}s  {e}{self._RESET}"
            )
            return ParallelResult(task_id=task.task_id, content="", error=str(e))

    # ── Public API ────────────────────────────────────────────────────

    def run_parallel(self, tasks: list[ParallelTask]) -> list[ParallelResult]:
        """
        Submit all *tasks* to the thread pool and return results in the
        same order as the input list, regardless of completion order.
        """
        if not tasks:
            return []

        # FIX: Use a dict to collect results so the return type is clean
        # without needing `type: ignore` on a list[ParallelResult | None].
        results: dict[int, ParallelResult] = {}

        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as pool:
            future_to_index = {
                pool.submit(self._single_request, t): i
                for i, t in enumerate(tasks)
            }
            for future in concurrent.futures.as_completed(future_to_index):
                idx = future_to_index[future]
                try:
                    results[idx] = future.result()
                except Exception as e:
                    results[idx] = ParallelResult(
                        task_id=tasks[idx].task_id, content="", error=str(e)
                    )

        return [results[i] for i in range(len(tasks))]


print("✓ Parallel API handler ready")

✓ Parallel API handler ready


## CELL 11: Heartbeat & Session Managers

In [35]:
# @title
# ---------- Heartbeat Manager ----------

import re
import threading
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any


class HeartbeatManager:
    """Periodic health checks: core files, pending needs, skill inventory, log rotation."""

    CORE_FILES: tuple[str, ...] = (
        "SOUL.md", "USER.md", "MEMORY.md", "HEARTBEAT.md", "Need.md", "Tools.md",
    )

    # ── Health checks ─────────────────────────────────────────────────

    @staticmethod
    def _check_core_files(status: dict[str, Any]) -> None:
        """Verify that all required core identity files exist."""
        missing = [f for f in HeartbeatManager.CORE_FILES
                   if not (Config.CORE_DIR / f).exists()]
        if missing:
            status["checks"]["context_validation"] = "critical"
            status["action_required"] = True
            status["actions"].append(f"Missing core files: {missing}")
        else:
            status["checks"]["context_validation"] = "ok"

    @staticmethod
    def _check_pending_needs(status: dict[str, Any]) -> None:
        """Flag any PENDING entries in Need.md."""
        needs = HeartbeatManager._get_pending_needs()
        if needs:
            status["checks"]["need_review"] = "attention"
            status["action_required"] = True
            status["actions"].append(f"{len(needs)} pending request(s) in Need.md")
        else:
            status["checks"]["need_review"] = "ok"

    @staticmethod
    def _check_skills(status: dict[str, Any], tools_store: CachedSubtools) -> None:
        """Validate registered skills and detect unregistered skill files."""
        # FIX (from Doc 6): Cleaner comprehension for invalid skills
        issues: list[str] = [
            f"{s['name']}: {s.get('validation_error', 'unknown')}"
            for s in tools_store.validate_all_skills().get("invalid", [])
        ]

        if Config.SKILLS_DIR.exists():
            # FIX (from Doc 5): Wrap iterdir() in try/except — permission errors
            # or race conditions on the directory must not crash the heartbeat.
            try:
                registered   = {s["name"] for s in tools_store.list_skills()}
                on_disk      = {f.stem for f in Config.SKILLS_DIR.iterdir() if f.suffix == ".py"}
                unregistered = on_disk - registered
                if unregistered:
                    issues.append(f"Unregistered skills: {sorted(unregistered)}")
            except Exception as e:
                logger.warning(f"Error checking unregistered skills: {e}")

        if issues:
            status["checks"]["skill_inventory"] = "attention"
            status["action_required"] = True
            status["actions"].extend(issues)
        else:
            status["checks"]["skill_inventory"] = "ok"

    # ── Public API ────────────────────────────────────────────────────

    @staticmethod
    def check(tools_store: CachedSubtools | None = None) -> dict[str, Any]:
        """Run all health checks and return a structured status dict."""
        status: dict[str, Any] = {
            "timestamp":        datetime.now().isoformat(),
            "interval_seconds": Config.HEARTBEAT_INTERVAL,
            "checks":           {},
            "action_required":  False,
            "actions":          [],
        }

        HeartbeatManager._check_core_files(status)
        HeartbeatManager._check_pending_needs(status)

        if tools_store is not None:
            HeartbeatManager._check_skills(status, tools_store)

        # FIX (from Doc 5): Guard against logger not having rotate_old_daily_logs —
        # an AttributeError here would silently swallow all heartbeat results.
        if hasattr(logger, "rotate_old_daily_logs"):
            try:
                logger.rotate_old_daily_logs()
            except Exception as e:
                logger.warning(f"Log rotation failed: {e}")
        status["checks"]["log_rotation"] = "ok"

        severity = "ATTENTION" if status["action_required"] else "OK"
        msg      = f"HEARTBEAT {severity}"
        if status["actions"]:
            msg += f": {'; '.join(status['actions'][:3])}"
        logger.info(msg)
        return status

    @staticmethod
    def _get_pending_needs() -> list[dict[str, str]]:
        """
        Parse PENDING request blocks from core/Need.md.
        Mirrors ``check_needs()`` but is private to avoid the circular
        dependency that would arise from calling the module-level function.
        Uses the shared ``_NEED_FIELDS`` and ``_NEED_REQUIRED`` constants
        defined in the managers cell.
        """
        content = FileManager.read_file(Config.CORE_DIR / "Need.md", default="")
        pending: list[dict[str, str]] = []
        for block in content.split("## [")[1:]:
            entry: dict[str, str] = {}
            for field in _NEED_FIELDS:
                if m := re.search(rf'\*\*{field}\*\*\s*:\s*(.+)', block):
                    entry[field.lower()] = m.group(1).strip()
            # FIX: Parse field before checking (not raw text search) — consistent
            # with the check_needs() fix in the managers cell.
            # FIX: Require minimum fields present before appending — rejects
            # partial/malformed blocks, consistent with _NEED_REQUIRED in managers cell.
            if entry.get("status") == "PENDING" and _NEED_REQUIRED.issubset(entry):
                pending.append(entry)
        return pending

    @staticmethod
    def run_loop(
        interval:    int | None             = None,
        stop_event:  threading.Event | None = None,
        tools_store: CachedSubtools | None  = None,
    ) -> None:
        """
        Run ``check()`` repeatedly until *stop_event* is set.
        Designed to be executed on a background daemon thread.
        """
        # FIX: Explicit None check — interval=0 would previously fall through
        # to Config.HEARTBEAT_INTERVAL silently, same pattern as ShellRunner.run.
        interval = Config.HEARTBEAT_INTERVAL if interval is None else interval
        logger.info(f"Heartbeat started (interval={interval}s)")
        while not (stop_event and stop_event.is_set()):
            try:
                HeartbeatManager.check(tools_store)
            except Exception as e:
                logger.error(f"Heartbeat error: {e}")
            time.sleep(interval)
        logger.info("Heartbeat stopped")


print("✓ Heartbeat manager ready")

✓ Heartbeat manager ready


## CELL 12: Main Agent Class

In [36]:
# @title
# Import OpenAI client
try:
    from openai import OpenAI
except ImportError:
    print("ERROR: openai not installed. Run: pip install openai")
    raise


class Agent:
    """Universal autonomous AI agent backed by the NVIDIA NIM API."""

    # Log colours / levels
    _LOG_LEVELS: dict[str, str] = {
        "error": "error", "system": "info", "user": "info",
        "assistant": "info", "tool": "info", "result": "debug",
    }
    _LOG_COLORS: dict[str, str] = {
        "system":    "\033[90m", "user":   "\033[94m",
        "assistant": "\033[92m", "tool":   "\033[93m",
        "result":    "\033[96m", "error":  "\033[91m",
    }
    _CACHEABLE_TOOLS = frozenset({"web_search", "web_fetch"})

    # Chat commands
    _CHAT_COMMANDS: dict[str, str] = {
        "exit/quit/q":   "Exit the agent",
        "status":        "Show full agent status",
        "stats":         "Show API usage statistics",
        "history":       "Show past session stats",
        "progress":      "Show current plan progress",
        "todo":          "Show current todo list",
        "logs [N]":      "Show last N log records (default 20)",
        "reset":         "Clear conversation history",
        "clear":         "Clear persistent memory",
        "stream":        "Toggle streaming on/off",
        "heartbeat":     "Run system health check",
        "help":          "Show this help",
    }

    # Tools by category — evaluated lazily so TOOLS is always up to date
    _TOOLS_BY_CATEGORY: dict[str, Callable[[], list[str]]] = {
        "Memory":    lambda: [t["name"] for t in TOOLS if t["name"].startswith("memory_")],
        "Files":     lambda: [t["name"] for t in TOOLS if t["name"].startswith("file_")],
        "Shell":     lambda: ["shell_run"],
        "Code":      lambda: ["execute_python", "diff_apply"],
        "Web":       lambda: ["web_search", "web_fetch"],
        "Sub-tools": lambda: [t["name"] for t in TOOLS if t["name"].startswith("subtool_")],
        "Skills":    lambda: [t["name"] for t in TOOLS if t["name"].startswith("skill_")],
        "Todo":      lambda: [t["name"] for t in TOOLS if t["name"].startswith("todo_")],
        "Needs":     lambda: ["post_need", "check_needs"],
        "Session":   lambda: ["save_session_log", "search_memory", "check_heartbeat",
                              "read_file", "write_file", "append_file", "delete_file",
                              "list_directory", "search_files"],
        "Parallel":  lambda: ["parallel_generate"],
        "Git":       lambda: ["git_status","git_diff","git_commit","git_push","git_log"],
        "Database":  lambda: ["sql_query","sql_tables"],
        "Browser":   lambda: ["browser_fetch"],
        "Email":     lambda: ["send_email"],
        "Other":     lambda: ["task_plan","agent_status","api_stats",
                              "memory_semantic_search","export_conversation",
                              "structured_query","workspace_diff","cancel_task"],
    }

    _STATUS_ICONS: dict[str, str] = {
        "pending": "○", "in_progress": "▶", "completed": "✓", "failed": "✗",
    }

    # Extension hints for parallel file generation
    _EXT_HINTS: dict[str, str] = {
        ".py":   "senior Python engineer. Clean, type-hinted, documented code",
        ".js":   "senior JavaScript engineer. Modern ES2022+ with JSDoc",
        ".ts":   "senior TypeScript engineer. Strict types, interfaces, JSDoc",
        ".jsx":  "senior React engineer. Functional components with hooks",
        ".tsx":  "senior React/TS engineer. Strict typing, functional components",
        ".html": "senior frontend engineer. Semantic, accessible HTML5",
        ".css":  "senior CSS engineer. Clean, modern CSS with BEM",
        ".md":   "technical writer. Clear, well-structured Markdown",
        ".json": "Output valid, well-formatted JSON only — no prose, no fences",
        ".yaml": "Output valid YAML only — no prose, no fences",
        ".sh":   "senior DevOps engineer. Robust, portable bash with error handling",
        ".sql":  "senior DB engineer. Clean, optimised SQL",
        ".rs":   "senior Rust engineer. Safe, idiomatic Rust",
        ".go":   "senior Go engineer. Idiomatic, well-documented Go",
        ".java": "senior Java engineer. Clean, well-documented Java",
        ".rb":   "senior Ruby engineer. Idiomatic, well-documented Ruby",
    }
    _ROLE_PREFIX    = "You are a "
    _NO_FENCE       = (
        "\nOutput ONLY the file content — no preamble, "
        "no markdown fences unless the file IS markdown."
    )
    _DEFAULT_SYSTEM = (
        "You are an expert software engineer and technical writer. "
        "Produce complete, high-quality output for the requested file." + _NO_FENCE
    )

    # Context-window overflow detection keywords
    _OVERFLOW_KEYS: tuple[str, ...] = (
        "context", "token", "length", "limit", "too long", "maximum",
    )

    # Embedded agent instructions (formatted in _build_system_prompt)
    _SYSTEM_INSTRUCTIONS = """\
## 🎯 TODO LIST TASK MANAGEMENT SYSTEM
Always think before you act. For complex multi-step tasks use the todo list:

### Core Tools
• **todo_create(task, steps)** — Create a new todo list with main steps
• **todo_add_subtask(parent_id, description)** — Add subtask to any parent
• **todo_complete(task_id, result?)** — Mark completed with optional result
• **todo_fail(task_id, error?)** — Mark failed with error info
• **todo_status()** — Get full todo tree with progress
• **todo_set_current(task_id)** — Set which task to work on next
• **todo_update(task_id, ...)** — Update task description/notes

### Need Management
• **post_need(priority, need, reason, blocking)** — Request something from the user
• **check_needs()** — List all PENDING user-requests

### Session & Memory
• **save_session_log(entry, category)** — Log an action to today's daily log
• **search_memory(query, days_back)** — Search historical daily logs
• **check_heartbeat()** — Run full system health check

### Workflow Pattern
1. **Plan First**: For complex tasks call todo_create()
2. **Work Sequentially**: Check todo_status() before each action
3. **Create Subtasks**: Use todo_add_subtask() for complex steps
4. **Complete Tasks**: Call todo_complete() / todo_fail() after each step
5. **Parallel Execution**: Use parallel_generate() for 3+ independent files
6. **Track Progress**: The system maintains full hierarchy and status

## ReAct Pattern — ALWAYS follow:
1. **Thought**: Check todo_status()
2. **Plan**: If no list exists, call todo_create()
3. **Act**: Work on current task
4. **Complete**: Call todo_complete()
5. **Repeat**: System auto-advances to next task
6. **Answer**: Give final response when all tasks done

## When NOT to use tools
- Greetings, casual chat, simple questions you already know
- Explaining concepts, writing short text, poems, outlines

## Path rules
- ALL file_* tool paths are RELATIVE to {files_dir}
- NEVER pass absolute paths like '{base_dir}' to file_* tools
- For read_file / write_file / append_file you may use absolute paths

## Core workflow
1. Complex tasks → todo_create first
2. shell_run to install, run, test, lint, git
3. diff_apply for targeted edits (not full rewrites)
4. parallel_generate for 3+ independent files
5. Verify with shell_run after generating code
6. Use start_line/end_line in file_read for large files

## Tool selection cheat-sheet
- Edit existing  : diff_apply
- New file (1)   : file_write
- New files (3+) : file_write_many  OR  parallel_generate
- Run anything   : shell_run
- Maths/logic    : execute_python
- Research       : web_search → web_fetch
- Needs post     : post_need

## Rules
- NEVER call file_write_many with fewer than 3 files
- Always read shell_run output and react to errors
- Context window: 262 000 tokens (budget ~{ctx_budget:,} usable)
- Max output: {max_tokens:,} tokens per call. Max iterations: {max_iters}
"""

    # Core identity file templates written on first run
    _CORE_TEMPLATES: dict[str, str] = {
        "SOUL.md":      "# SOUL.md\n## Identity\nI am an autonomous AI agent.\n",
        "USER.md":      "# USER.md\n## Profile\nUser profile and preferences.\n",
        "HEARTBEAT.md": "# HEARTBEAT.md\n## Periodic Checklist\n- Check pending needs\n- Validate skills\n",
        "MEMORY.md":    "# MEMORY.md\n## Long-term Memory\n*(empty)*\n",
        "Need.md":      "# Need.md\n## Agent Requests to User\n*(none yet)*\n",
        "Tools.md":     "# Tools.md\n## Tool and Skill Registry\n\n## Skill Scripts\n*(none yet)*\n",
    }

    # ── Construction ──────────────────────────────────────────────────

    def __init__(self, verbose: bool = True, stream: bool = True) -> None:
        if not setup_api_key():
            raise RuntimeError("Failed to setup API key")

        self.client      = OpenAI(api_key=os.environ["NVIDIA_API_KEY"], base_url=Config.BASE_URL)
        self.mem         = EfficientMemory()
        self.tools_store = CachedSubtools()
        self.files       = SecureFiles()
        self.code        = SecureCodeExecutor()
        self.web         = SecureWebSearch()
        self.shell       = ShellRunner()
        self.differ      = DiffApplier()
        self.rate_limiter = RateLimiter()
        self.counter     = APICounter()
        self.tool_cache  = ToolResultCache()
        self.parallel    = ParallelAPIHandler(
            self.client, self.counter, rate_limiter=self.rate_limiter
        )
        self.todo        = TodoListManager(self.mem)

        self.verbose:  bool                    = verbose
        self.stream:   bool                    = stream
        self.history:  list[dict[str, Any]]    = []
        self._current_plan_step:  int          = 0
        self._current_plan_total: int          = 0
        self.initialized:         bool         = False
        self._cancel_flag:        bool         = False  # set externally to abort run()

        self._heartbeat_thread: threading.Thread | None = None
        self._stop_heartbeat:   threading.Event         = threading.Event()

        self._register_shutdown()
        self._initialize_core_files()
        self.initialized = True
        self._log(
            f"Agent ready | model={Config.MODEL} | workspace={Config.BASE_DIR} | stream={stream}"
        )

    # ── Core file initialisation ──────────────────────────────────────

    def _initialize_core_files(self) -> None:
        """Write default core identity files if they don't yet exist."""
        for filename, content in self._CORE_TEMPLATES.items():
            path = Config.CORE_DIR / filename
            if not path.exists():
                FileManager.write_file(path, content)
        SessionManager.ensure_todays_log()

    # ── Heartbeat control ─────────────────────────────────────────────

    def start_background_heartbeat(self, interval: int | None = None) -> None:
        if self._heartbeat_thread and self._heartbeat_thread.is_alive():
            logger.warning("Heartbeat already running")
            return
        self._stop_heartbeat.clear()
        effective_interval = interval or Config.HEARTBEAT_INTERVAL
        self._heartbeat_thread = threading.Thread(
            target=HeartbeatManager.run_loop,
            args=(effective_interval, self._stop_heartbeat, self.tools_store),
            daemon=True,
        )
        self._heartbeat_thread.start()
        logger.info(f"Background heartbeat started (interval={effective_interval}s)")

    def stop_background_heartbeat(self) -> None:
        if self._heartbeat_thread:
            self._stop_heartbeat.set()
            self._heartbeat_thread.join(timeout=5)
            if self._heartbeat_thread.is_alive():
                logger.warning("Heartbeat thread did not stop cleanly")
            else:
                logger.info("Heartbeat stopped")

    # ── Internal helpers ──────────────────────────────────────────────

    def _register_shutdown(self) -> None:
        """Register SIGINT/SIGTERM handlers to persist stats before exit."""
        def _handler(sig, frame):
            print("\n\033[93m[AGENT] Shutting down… saving stats.\033[0m")
            self.counter.persist_stats()
            self.stop_background_heartbeat()
            sys.exit(0)
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, _handler)
            except (OSError, ValueError):
                pass  # Signal registration not supported in this context (e.g. threads)

    def _log(self, msg: str, role: str = "system") -> None:
        getattr(logger, self._LOG_LEVELS.get(role, "info"))(str(msg)[:10000], role=role)
        if not self.verbose and role != "error":
            return
        color = self._LOG_COLORS.get(role, "\033[97m")
        print(f"{color}[{role.upper():<9}]\033[0m {str(msg)[:4000]}")

    # ── Tool dispatch ─────────────────────────────────────────────────

    def _build_dispatch(self, inp: dict[str, Any]) -> dict[str, Callable[[], Any]]:
        """
        Build a name → callable map for all supported tools.
        Lambdas close over *inp* so each is evaluated only when called.
        """
        m, t, f = self.mem, self.tools_store, self.files
        return {
            # Memory
            "memory_save":       lambda: m.save(inp["key"], inp["value"], inp.get("tag", "")),
            "memory_get":        lambda: m.get(inp["key"]),
            "memory_list":       lambda: m.list_keys(inp.get("tag", "")),
            "memory_search":     lambda: m.search(inp["query"]),
            "memory_delete":     lambda: m.delete(inp["key"]),
            # Files (sandboxed)
            "file_read":         lambda: f.read(inp["path"],
                                     int(inp.get("start_line") or 0),
                                     int(inp.get("end_line")   or 0)),
            "file_write":        lambda: f.write(inp["path"], inp["content"],
                                     inp.get("append", False), inp.get("backup", False)),
            "file_write_many":   lambda: f.write_many(inp["writes"]),
            "file_list":         lambda: f.list(inp.get("directory", ""), inp.get("metadata", False)),
            "file_delete":       lambda: f.delete(inp["path"]),
            "file_search":       lambda: f.search(inp["query"], inp.get("directory", "")),
            "file_rename":       lambda: f.rename(inp["src"], inp["dst"]),
            # Shell
            "shell_run":         lambda: self.shell.run(
                                     inp["command"],
                                     timeout=min(int(inp.get("timeout", Config.SHELL_TIMEOUT)), Config.SHELL_TIMEOUT * 5),
                                     cwd=inp.get("cwd", "")),
            # Code
            "execute_python":    lambda: (
                                     self.code.run(inp["code"], inp.get("timeout", Config.CODE_TIMEOUT))
                                     if inp.get("code") else "Missing parameter 'code'"),
            # Diff
            "diff_apply":        lambda: self.differ.apply(inp["path"], inp["diff"]),
            # Web
            "web_search":        lambda: self.web.search(inp["query"], inp.get("max_results", 5)),
            "web_fetch":         lambda: self.web.fetch(inp["url"], inp.get("max_chars", Config.MAX_WEB_CONTENT)),
            # Sub-tools
            "subtool_create":    lambda: t.create(inp["name"], inp["description"],
                                     inp["code"], inp.get("force", False)),
            "subtool_run":       lambda: t.run(
                                     inp["name"],
                                     **(inp.get("kwargs") if isinstance(inp.get("kwargs"), dict) else {})),
            "subtool_list":      lambda: t.list(),
            "subtool_delete":    lambda: t.delete(inp["name"]),
            # Skills
            "skill_list":        lambda: t.list_skills(),
            "skill_create":      lambda: t.create_skill(
                                     inp["name"], inp["code"], inp["purpose"], inp["trigger"],
                                     inp.get("input_fmt", ""), inp.get("output_fmt", "")),
            "skill_run":         lambda: t.run_skill(inp["name"], inp.get("input_data")),
            "skill_validate":    lambda: (
                                     t.validate_skill(inp["name"]) if inp.get("name")
                                     else t.validate_all_skills()),
            # Todo
            "todo_create":       lambda: self.todo.create(inp["task"], inp["steps"]),
            "todo_status":       lambda: self.todo.status(),
            "todo_add_subtask":  lambda: self.todo.add_subtask(inp["parent_id"], inp["description"]),
            "todo_complete":     lambda: self.todo.complete(inp["task_id"], inp.get("result")),
            "todo_fail":         lambda: self.todo.fail(inp["task_id"], inp.get("error")),
            "todo_set_current":  lambda: self.todo.set_current(inp["task_id"]),
            "todo_update":       lambda: self.todo.update(
                                     inp["task_id"], inp.get("description"), inp.get("notes")),
            # Needs
            "post_need":         lambda: post_need(
                                     inp["priority"], inp["need"], inp["reason"], inp["blocking"]),
            "check_needs":       lambda: check_needs(),
            # Session
            "save_session_log":  lambda: SessionManager.save_log(
                                     inp["entry"], inp.get("category", "General")),
            "search_memory":     lambda: SessionManager.search_memory(
                                     inp["query"], inp.get("days_back", 7)),
            "check_heartbeat":   lambda: HeartbeatManager.check(self.tools_store),
            # General file ops (unrestricted paths)
            "read_file":         lambda: FileManager.read_file(
                                     Path(inp["filepath"]), encoding=inp.get("encoding", "utf-8")),
            "write_file":        lambda: FileManager.write_file(
                                     Path(inp["filepath"]), inp["content"]),
            "append_file":       lambda: FileManager.append_file(
                                     Path(inp["filepath"]), inp["content"],
                                     inp.get("add_timestamp", True)),
            "delete_file":       lambda: FileManager.delete_path(Path(inp["filepath"])),
            "list_directory":    lambda: FileManager.list_directory(
                                     Path(inp.get("path", ".")), inp.get("pattern", "*")),
            "search_files":      lambda: self._search_files(
                                     inp["query"],
                                     inp.get("directory", "."),
                                     inp.get("file_pattern", "*.md")),
            # Planning / status
            "task_plan":         lambda: self._save_plan(inp["task"], inp["steps"]),
            "agent_status":      lambda: self._agent_status(),
            "api_stats":         lambda: {
                                     **self.counter.summary(),
                                     "historical_sessions": len(self.counter.load_historical()),
                                 },
            "parallel_generate":      lambda: self._parallel_generate(
                                         inp["items"], inp.get("system_prompt", "")),
            # Git
            "git_status":             lambda: self._git_run("status --short", inp.get("directory","")),
            "git_diff":               lambda: self._git_run(
                                         ("diff --cached" if inp.get("staged") else "diff"),
                                         inp.get("directory","")),
            "git_commit":             lambda: self._git_run(
                                         f'add -A && git commit -m {json.dumps(inp["message"])}',
                                         inp.get("directory","")),
            "git_push":               lambda: self._git_run(
                                         f'push origin {inp.get("branch","HEAD")}',
                                         inp.get("directory","")),
            "git_log":                lambda: self._git_run(
                                         f'log --oneline -{inp.get("n",10)}',
                                         inp.get("directory","")),
            # Database
            "sql_query":              lambda: self._sql_query(
                                         inp["db_path"], inp["sql"],
                                         inp.get("params",[])),
            "sql_tables":             lambda: self._sql_tables(inp["db_path"]),
            # Browser
            "browser_fetch":          lambda: self._browser_fetch(
                                         inp["url"],
                                         inp.get("wait_for",""),
                                         inp.get("max_chars", 8000)),
            # Email
            "send_email":             lambda: self._send_email(
                                         inp["to"], inp["subject"], inp["body"],
                                         inp.get("cc",""), inp.get("html", False)),
            # Semantic memory
            "memory_semantic_search": lambda: self.mem.semantic_search(
                                         inp["query"], inp.get("top_k",5)),
            # Conversation export
            "export_conversation":    lambda: self.export_conversation(
                                         inp.get("fmt","md"), inp.get("path")),
            # Structured query
            "structured_query":       lambda: self.structured_query(
                                         inp["prompt"],
                                         inp.get("schema_desc","Return valid JSON only."),
                                         inp.get("max_tokens",2000)),
            # Workspace diff
            "workspace_diff":         lambda: self._workspace_diff(),
            # Cancel running task
            "cancel_task":            lambda: self._cancel_running_task(),
        }

    def _dispatch(self, name: str, inp: dict[str, Any]) -> Any:
        """Look up and invoke a tool by name, with caching and error handling."""
        try:
            self.rate_limiter.wait_if_needed()
            if name in self._CACHEABLE_TOOLS:
                if cached := self.tool_cache.get(name, inp):
                    self._log(f"  \033[90m[cache hit] {name}\033[0m", "system")
                    return cached
            fn = self._build_dispatch(inp).get(name)
            if fn is None:
                return f"Unknown tool: {name}"
            result = fn()
            if name in self._CACHEABLE_TOOLS and result and not str(result).startswith("Error"):
                self.tool_cache.set(name, inp, result)
            return result if isinstance(result, str) else json.dumps(result, default=str, indent=2)
        except Exception as e:
            msg = f"Tool error [{name}]: {e}\n{traceback.format_exc()}"
            self._log(msg, "error")
            return msg

    # ── File search ───────────────────────────────────────────────────

    def _search_files(
        self, query: str, directory: str = ".", file_pattern: str = "*.md"
    ) -> list[dict[str, Any]]:
        q      = query.lower()
        target = Path(directory) if Path(directory).is_absolute() else Config.BASE_DIR / directory
        results: list[dict[str, Any]] = []
        for filepath in target.rglob(file_pattern):
            try:
                content = FileManager.read_file(filepath)
                if q not in content.lower():
                    continue
                matches = [
                    {"line_num": n + 1, "text": line.strip()}
                    for n, line in enumerate(content.splitlines())
                    if q in line.lower()
                ][:10]
                results.append({"file": str(filepath), "matches": matches})
            except Exception:
                continue
        return results

    # ── Agent status ──────────────────────────────────────────────────

    def _agent_status(self) -> dict[str, Any]:
        dirs = {
            name: path.exists()
            for name, path in (
                ("core",   Config.CORE_DIR),
                ("memory", Config.MEMORY_DIR),
                ("skills", Config.SKILLS_DIR),
                ("work",   Config.WORK_DIR),
                ("files",  Config.FILES_DIR),
            )
        }
        skills_count = (
            len(list(Config.SKILLS_DIR.glob("*.py")))
            if Config.SKILLS_DIR.exists() else 0
        )
        return {
            "model":            Config.MODEL,
            "workspace":        str(Config.BASE_DIR),
            "initialized":      self.initialized,
            "directories":      dirs,
            "core_files":       {f: (Config.CORE_DIR / f).exists() for f in HeartbeatManager.CORE_FILES},
            "memory_keys":      self.mem.list_keys(),
            "subtools":         list(self.tools_store.list().keys()),
            "skills_count":     skills_count,
            "files":            self.files.list(),
            "api_stats":        self.counter.summary(),
            "tool_cache_size":  self.tool_cache.size,
            "plan_progress":    (
                f"{self._current_plan_step}/{self._current_plan_total}"
                if self._current_plan_total else "no active plan"
            ),
            "todo_status":      self.todo.status(),
            "heartbeat_active": (
                self._heartbeat_thread is not None and self._heartbeat_thread.is_alive()
            ),
        }

    # ── Plan tracking ─────────────────────────────────────────────────

    def _save_plan(self, task: str, steps: list[str]) -> str:
        self._current_plan_step, self._current_plan_total = 0, len(steps)
        self.mem.save("current_plan", {"task": task, "steps": steps, "current_step": 0}, tag="plan")
        return f"Plan saved ({len(steps)} steps). Use agent_status to track progress."

    def _advance_plan_step(self) -> None:
        if self._current_plan_total <= 0:
            return
        self._current_plan_step = min(self._current_plan_step + 1, self._current_plan_total)
        plan = self.mem.get("current_plan")
        if isinstance(plan, dict):
            plan["current_step"] = self._current_plan_step
            self.mem.save("current_plan", plan, tag="plan")

    # ── System prompt ─────────────────────────────────────────────────

    def _build_system_prompt(self) -> str:
        mem_keys      = self.mem.list_keys()
        subtool_names = list(self.tools_store.list().keys())
        context       = SessionManager.load_context()
        has_identity  = any(len(v) > 50 for v in context.values())

        header = (
            f"You are a universal autonomous AI agent powered by {Config.MODEL} via NVIDIA NIM.\n"
            f"Workspace: {Config.BASE_DIR} | Files dir: {Config.FILES_DIR} | "
            f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n"
            f"Memory keys: {mem_keys[:Config.SYSTEM_PROMPT_MEM_LIMIT]} | "
            f"Sub-tools: {subtool_names[:Config.SYSTEM_PROMPT_TOOL_LIMIT]}\n\n"
        )
        instructions = self._SYSTEM_INSTRUCTIONS.format(
            files_dir=Config.FILES_DIR, base_dir=Config.BASE_DIR,
            ctx_budget=Config.CONTEXT_TOKEN_BUDGET,
            max_tokens=Config.MAX_TOKENS, max_iters=Config.MAX_ITERS,
        )
        if has_identity:
            tools_str        = "\n".join(f"- {t['name']}: {t['description']}" for t in TOOLS)
            identity_section = build_system_prompt_from_context(context, tools_str)
            return identity_section + "\n---\n\n" + header + self._todo_context() + instructions
        return header + self._todo_context() + instructions

    def _todo_context(self) -> str:
        ts = self.todo.status()
        if "error" in ts or not ts.get("tasks"):
            return ""
        current = ts.get("current_task")
        lines   = [
            "## 📋 ACTIVE TODO LIST\n",
            f"Overall Task: {ts['task']}\n",
            f"Progress: {ts['progress']}\n",
        ]
        if current:
            lines.append(
                f"CURRENT TASK: {current['id']} - {current['description']} [{current['status']}]\n"
            )
        lines.append("Use todo_status() to see full tree. Always work on CURRENT TASK.\n\n")
        return "".join(lines)

    # ── Token-aware history management ────────────────────────────────

    @staticmethod
    def _est_tokens(text: str) -> int:
        """
        Conservative token-count approximation (1 token ≈ 3 chars) for history
        budget management.  Intentionally overestimates to provide a safety margin
        against context-window overflows.
        Note: ``LiveStatusBar._approx`` uses ``// 4`` (more accurate) for display only.
        """
        return max(1, len(text) // 3)

    def _history_tokens(self) -> int:
        total = 0
        for msg in self.history:
            content = msg.get("content") or ""
            total  += self._est_tokens(
                content if isinstance(content, str) else json.dumps(content)
            )
            if tc := msg.get("tool_calls"):
                total += self._est_tokens(json.dumps(tc))
        return total

    def _cap_result(self, result: str, _tool_name: str) -> str:
        """Truncate *result* to the configured character cap, preserving head and tail."""
        cap = Config.TOOL_RESULT_CHARS_MAX
        if len(result) <= cap:
            return result
        h = cap // 2
        return (
            f"{result[:h]}\n\n... [TRUNCATED: {len(result):,} chars total, "
            f"showing first+last {h} chars] ...\n\n{result[-h:]}"
        )

    def _summarize_messages(self, messages: list[dict[str, Any]]) -> str:
        """Summarise conversation history to reduce token usage."""
        def _msg_line(m: dict[str, Any]) -> str:
            content = m.get("content") or ""
            if isinstance(content, list):
                content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
            return f"[{m.get('role', '?')}]: {str(content)[:4000]}"

        dump   = "\n".join(map(_msg_line, messages))
        prompt = (
            "Summarise the following conversation history in 3-5 sentences, "
            "preserving all key facts, decisions, file names, and errors:\n\n" + dump
        )

        @retry_api(max_retries=3, backoff=1.5)
        def _call(client, model, content):
            return client.chat.completions.create(
                model=model, max_tokens=600,
                messages=[{"role": "user", "content": content}],
            )

        try:
            self.rate_limiter.wait_if_needed()
            # Determine model with fallback
            model_used = Config.MODEL
            try:
                resp = _call(self.client, model_used, prompt)
            except Exception as e:
                if self._should_fallback(e):
                    model_used = Config.FALLBACK_MODEL
                    resp = _call(self.client, model_used, prompt)
                else:
                    raise
            return resp.choices[0].message.content or dump[:1000]
        except Exception as exc:
            logger.warning("summarise failed, using truncated dump", error=str(exc))
            return dump[:1000]
        def _msg_line(m: dict[str, Any]) -> str:
            content = m.get("content") or ""
            if isinstance(content, list):
                content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
            return f"[{m.get('role', '?')}]: {str(content)[:4000]}"

        dump   = "\n".join(map(_msg_line, messages))
        prompt = (
            "Summarise the following conversation history in 3-5 sentences, "
            "preserving all key facts, decisions, file names, and errors:\n\n" + dump
        )

        @retry_api(max_retries=3, backoff=1.5)
        def _call(client, model, content):
            return client.chat.completions.create(
                model=model, max_tokens=600,
                messages=[{"role": "user", "content": content}],
            )

        try:
            self.rate_limiter.wait_if_needed()
            resp = _call(self.client, Config.MODEL, prompt)
            return resp.choices[0].message.content or dump[:1000]
        except Exception as exc:
            logger.warning("summarise failed, using truncated dump", error=str(exc))
            return dump[:1000]

    def _trim_history_to_budget(self, budget: int | None = None) -> None:
        """Summarise and drop old history messages to stay within *budget* tokens."""
        budget = budget or Config.CONTEXT_TOKEN_BUDGET
        if self._history_tokens() <= budget or len(self.history) <= 4:
            return
        head   = [self.history[0]]
        tail   = self.history[-12:]
        middle = self.history[1: max(1, len(self.history) - 12)]
        if middle:
            summary = self._summarize_messages(middle)
            head.append({
                "role":    "assistant",
                "content": f"[Context summary — earlier conversation]\n{summary}",
            })
        self.history = head + tail
        while self._history_tokens() > budget and len(self.history) > 2:
            self.history = [self.history[0]] + self.history[2:]

    # ── API call helpers ──────────────────────────────────────────────

    def _extract_usage(self, usage: Any) -> tuple[int, int, int]:
        """
        Return ``(tokens_in, tokens_reasoning, tokens_out)`` from a usage object.
        Model-agnostic: gracefully handles models that omit reasoning_tokens
        or completion_tokens (e.g. StepFun, non-OpenAI providers).
        """
        if not usage:
            return 0, 0, 0
        details   = getattr(usage, "completion_tokens_details", None)
        prompt    = getattr(usage, "prompt_tokens",      0) or 0
        reasoning = getattr(details, "reasoning_tokens", 0) or 0
        completion = getattr(usage, "completion_tokens", 0) or 0
        # Fallback: some models only expose total_tokens
        if prompt == 0 and completion == 0:
            total = getattr(usage, "total_tokens", 0) or 0
            prompt = total  # conservative: attribute all to prompt
        response  = max(completion - reasoning, 0)
        return prompt, reasoning, response

    def _serialize_tool_calls(self, raw_calls: Any) -> list[dict[str, Any]]:
        if not raw_calls:
            return []
        return [
            {
                "id": tc.id, "type": "function",
                "function": {"name": tc.function.name, "arguments": tc.function.arguments},
            }
            for tc in raw_calls
        ]

    def _api_call_with_overflow_retry(
        self, system: str
    ) -> tuple[str, list[dict] | None]:
        """Call the API, automatically trimming history on context-overflow errors."""
        msgs   = [{"role": "system", "content": system}] + self.history
        caller = self._call_api_streaming if self.stream else self._call_api_normal
        try:
            text, tool_calls = caller(msgs)
            if isinstance(text, str) and text.startswith("API Error"):
                return text, None
            return text, tool_calls
        except Exception as e:
            if not any(k in str(e).lower() for k in self._OVERFLOW_KEYS):
                self.counter.record(model=Config.MODEL, error=True)
                return f"API Error: {e}", None
            self._log("Context overflow — trimming history and retrying…", "system")
            self._trim_history_to_budget(budget=Config.CONTEXT_TOKEN_BUDGET // 3)
            try:
                msgs             = [{"role": "system", "content": system}] + self.history
                text, tool_calls = caller(msgs)
                return text, tool_calls
            except Exception as e2:
                self.counter.record(model=Config.MODEL, error=True)
                return f"API Error (after trim): {e2}", None

    # ── Streaming API call ────────────────────────────────────────────

    @retry_api()
    def _call_api_streaming(self, messages: list[dict]) -> tuple[str, list]:
        req_num   = self.counter.next_request_number()
        print(f"\033[92m[ASSISTANT]\033[0m  \033[90mstreaming REQ #{req_num}…\033[0m")
        bar       = LiveStatusBar(req_number=req_num, last_n_words=8)
        full_text = ""
        tc_buf:   dict[int, dict] = {}
        success   = True
        t0        = time.monotonic()

        try:
            # Determine model with fallback support
            model_used = Config.MODEL
            if self.rate_limiter:
                self.rate_limiter.wait_if_needed()
            try:
                stream = self.client.chat.completions.create(
                    model=model_used, max_tokens=Config.MAX_TOKENS,
                    messages=messages, tools=OAI_TOOLS, tool_choice="auto",
                    stream=True, stream_options={"include_usage": True},
                )
            except Exception as e:
                if self._should_fallback(e):
                    model_used = Config.FALLBACK_MODEL
                    if self.rate_limiter:
                        self.rate_limiter.wait_if_needed()
                    stream = self.client.chat.completions.create(
                        model=model_used, max_tokens=Config.MAX_TOKENS,
                        messages=messages, tools=OAI_TOOLS, tool_choice="auto",
                        stream=True, stream_options={"include_usage": True},
                    )
                else:
                    raise

            for chunk in stream:
                if Config.DEBUG_STREAM_RAW:
                    try:
                        chunk_repr = repr(chunk)
                        if len(chunk_repr) > 500:
                            chunk_repr = chunk_repr[:500] + '...'
                        logger.debug(f"[stream] raw chunk: {chunk_repr}")
                    except Exception as e:
                        logger.debug(f"[stream] raw chunk error: {e}")
                if hasattr(chunk, "usage") and chunk.usage is not None:
                    self._record_stream_usage(chunk.usage, bar, req_num)
                    continue
                if not getattr(chunk, "choices", None):
                    continue
                delta = chunk.choices[0].delta
                if r := getattr(delta, "reasoning_content", None):
                    bar.add_reasoning(r)
                if delta.content:
                    full_text += delta.content
                    bar.add_response(delta.content)
                if delta.tool_calls:
                    self._accumulate_tool_chunks(delta.tool_calls, tc_buf)
        except Exception:
            success = False
            self.counter.record(model=model_used, error=True)
            raise
        finally:
            bar.finalize(success=success, elapsed=time.monotonic() - t0)
            if success and self.counter.total < req_num:
                self.counter.record(
                    model=model_used,
                    tokens_out=bar.response_tokens,
                    tokens_reasoning=bar.reasoning_tokens,
                )
        return full_text, [tc_buf[i] for i in sorted(tc_buf)]
    def _record_stream_usage(self, usage: Any, bar: LiveStatusBar, req_num: int) -> None:
        prompt, api_reasoning, api_response = self._extract_usage(usage)
        reasoning = max(bar.reasoning_tokens, api_reasoning)
        response  = api_response or bar.response_tokens
        bar.set_actual_tokens(reasoning=reasoning, response=response)
        self.counter.record(
            model=Config.MODEL,
            tokens_in=prompt, tokens_out=response, tokens_reasoning=reasoning,
        )

    @staticmethod
    def _accumulate_tool_chunks(deltas: Any, buf: dict[int, dict]) -> None:
        for tc in deltas:
            entry = buf.setdefault(tc.index, {
                "id": "", "type": "function",
                "function": {"name": "", "arguments": ""},
            })
            if tc.id:                 entry["id"]                        = tc.id
            if tc.function.name:      entry["function"]["name"]         += tc.function.name
            if tc.function.arguments: entry["function"]["arguments"]    += tc.function.arguments

    # ── Non-streaming API call ────────────────────────────────────────

    @retry_api()
    def _call_api_normal(self, messages: list[dict]) -> tuple[str, list]:
        """Call API with primary model and fallback on failure."""
        def attempt(model):
            resp = self.client.chat.completions.create(
                model=model, max_tokens=Config.MAX_TOKENS,
                messages=messages, tools=OAI_TOOLS, tool_choice="auto",
            )
            prompt, reasoning, response = self._extract_usage(getattr(resp, "usage", None))
            self.counter.record(
                model=model,
                tokens_in=prompt, tokens_out=response, tokens_reasoning=reasoning,
            )
            msg = resp.choices[0].message
            tcs = self._serialize_tool_calls(msg.tool_calls) if msg.tool_calls else []
            return msg.content or "", tcs

        try:
            return attempt(Config.MODEL)
        except Exception as e:
            if self._should_fallback(e):
                logger.warning(f"Primary model {Config.MODEL} failed: {e}. Falling back to {Config.FALLBACK_MODEL}")
                try:
                    return attempt(Config.FALLBACK_MODEL)
                except Exception as e2:
                    raise Exception(f"Both models failed. Primary: {e}. Fallback: {e2}")
            else:
                raise

    def _should_fallback(self, exc: Exception) -> bool:
        """Determine if an exception should trigger model fallback."""
        err = str(exc).lower()
        triggers = [
            '404', 'not found', 'model not found', 'deleted', 'unavailable',
            '429', 'rate limit', 'too many', 'quota',
            '500', '502', '503', '504', 'server error', 'overloaded',
            'timeout', 'connection', 'network',
        ]
        return any(t in err for t in triggers)
    def run(self, task: str) -> str:
        """Process *task* and return the final response string."""
        self._log(task, "user")
        self.history.append({"role": "user", "content": task})
        SessionManager.save_log(f"User: {task}", category="Input")
        system = self._build_system_prompt()

        for _ in range(Config.MAX_ITERS):
            # Check cancellation flag (set by TaskQueue.cancel_current or external code)
            if self._cancel_flag:
                self._cancel_flag = False
                self._log("🛑 Task cancelled by external request.", "system")
                SessionManager.save_log("Task cancelled.", category="System")
                return "Task was cancelled."

            self._trim_history_to_budget()
            self._log(
                f"context ~{self._history_tokens():,} est. tokens  |  "
                f"{len(self.history)} history msgs", "system"
            )
            self._log_current_todo()

            text, tool_calls = self._api_call_with_overflow_retry(system)
            if isinstance(text, str) and text.startswith("API Error"):
                SessionManager.save_log(f"API error: {text}", category="Error")
                return text

            asst: dict[str, Any] = {"role": "assistant", "content": text}
            if tool_calls:
                asst["tool_calls"] = tool_calls
            self.history.append(asst)

            if not tool_calls:
                if self.todo.is_completed():
                    self._log("✅ ALL TODO TASKS COMPLETED!", "system")
                    evolve_now(agent) # Evolve and update core .md files
                    # show_core_files()
                    self._log("✅ ALL core_files updated!", "system")
                    result = text or "All planned tasks completed successfully."
                else:
                    result = text or "Task completed."
                SessionManager.save_log(
                    f"Agent: {result[:10000]}{'...' if len(result) > 10000 else ''}",
                    category="Output",
                )
                return result

            self._process_tool_calls(tool_calls)

        return "Max iterations reached. Break task into smaller steps."

    def _log_current_todo(self) -> None:
        ct = self.todo.get_current_task()
        if ct and not self.todo.is_completed():
            self._log(f"📋 TODO: {ct['id']} - {ct['description']} [{ct['status']}]", "system")

    def _process_tool_calls(self, tool_calls: list[dict]) -> None:
        for tc in tool_calls:
            name = tc["function"]["name"]
            try:
                inp = json.loads(tc["function"]["arguments"])
            except (json.JSONDecodeError, KeyError):
                inp = {}
            self._log(f"→ {name}({json.dumps(inp, default=str)[:120]})", "tool")
            res               = self._dispatch(name, inp)
            self._advance_plan_step()
            rs                = res if isinstance(res, str) else json.dumps(res, default=str)
            rs_stored         = self._cap_result(rs, name)
            truncated         = len(rs_stored) < len(rs)
            suffix            = (
                f"  \033[90m[stored {len(rs_stored):,}/{len(rs):,} chars]\033[0m"
                if truncated else ""
            )
            self._log(f"← {name}: {rs[:200]}{suffix}", "result")
            self.history.append({"role": "tool", "tool_call_id": tc["id"], "content": rs_stored})

    # ── Parallel task runner ──────────────────────────────────────────

    def run_parallel_tasks(self, tasks: list[str], system_override: str = "") -> list[dict[str, Any]]:
        """Run multiple independent user tasks in parallel and return their results."""
        system = system_override or (
            f"You are a secure autonomous AI agent ({Config.MODEL} via NVIDIA NIM).\n"
            f"Workspace: {Config.BASE_DIR} | Memory keys: {self.mem.list_keys()[:10]}"
        )
        results = self.parallel.run_parallel([
            ParallelTask(
                task_id=f"task_{i}",
                messages=[{"role": "user", "content": t}],
                system=system, tools=OAI_TOOLS,
            )
            for i, t in enumerate(tasks)
        ])
        output: list[dict[str, Any]] = []
        for i, r in enumerate(results):
            output.append({"task_id": r.task_id, "task": tasks[i],
                           "result": r.content, "error": r.error})
            self._log(
                f"{'✓' if not r.error else '✗'} parallel[{r.task_id}]: "
                f"{(r.content or r.error)[:120]}", "result"
            )
        return output

    # ── Parallel generation ───────────────────────────────────────────

    def _infer_system(self, paths: list[str], override: str) -> str:
        """Infer a system prompt from file extensions, or fall back to the default."""
        if override:
            return override
        exts  = {Path(p).suffix.lower() for p in paths}
        hints = [self._EXT_HINTS[e] for e in exts if e in self._EXT_HINTS]
        if len(hints) == 1:
            h    = hints[0]
            base = h if h.startswith("Output") else self._ROLE_PREFIX + h
            return base + self._NO_FENCE
        return self._DEFAULT_SYSTEM

    def _parallel_generate(self, items: list[dict], system_prompt: str = "") -> str:
        if not items:
            return "Error: no items provided"
        waves      = self._topo_waves(items)
        sys_inst   = self._infer_system([it["path"] for it in items], system_prompt)
        all_writes: list[str] = []
        all_errors: list[str] = []
        has_deps   = len(waves) > 1 and any(it.get("deps") for it in items)
        t0         = time.monotonic()

        for wi, wave in enumerate(waves):
            if has_deps:
                self._log(
                    f"⚡ parallel_generate wave {wi + 1}/{len(waves)}: "
                    f"{len(wave)} items — {[it['path'] for it in wave]}", "system"
                )
            results = self.parallel.run_parallel([
                ParallelTask(
                    task_id=it["path"],
                    messages=[{"role": "user", "content": it.get("prompt", "")}],
                    system=sys_inst, tools=[],
                    model=Config.MODEL, max_tokens=Config.MAX_TOKENS,
                )
                for it in wave
            ])
            writes, errors = self._partition_results(wave, results)
            all_writes.extend(self._save_writes(writes))
            all_errors.extend(errors)

        return self._format_gen_summary(all_writes, all_errors, items, waves, has_deps,
                                        time.monotonic() - t0)

    def _topo_waves(self, items: list[dict]) -> list[list[dict]]:
        """Topologically sort *items* by their deps into parallel execution waves."""
        by_path   = {it["path"]: it for it in items}
        remaining = set(by_path)
        completed: set[str] = set()
        waves:     list[list[dict]] = []

        while remaining:
            ready = [
                by_path[p] for p in remaining
                if all(d in completed for d in (by_path[p].get("deps") or []))
            ]
            if not ready:
                self._log(
                    f"⚠ circular/missing deps — running {len(remaining)} items unsorted", "system"
                )
                waves.append([by_path[p] for p in remaining])
                break
            waves.append(ready)
            done       = {it["path"] for it in ready}
            remaining -= done
            completed |= done
        return waves

    @staticmethod
    def _partition_results(
        wave: list[dict], results: list[ParallelResult]
    ) -> tuple[list[dict], list[str]]:
        writes: list[dict] = []
        errors: list[str]  = []
        for item, res in zip(wave, results):
            p = item["path"]
            if res.error:
                errors.append(f"✗ {p}: {res.error}")
            elif res.content:
                writes.append({"path": p, "content": res.content})
            else:
                errors.append(f"✗ {p}: empty response")
        return writes, errors

    def _save_writes(self, writes: list[dict]) -> list[str]:
        if not writes:
            return []
        if len(writes) >= 3:
            return [f"✓ {r}" for r in self.files.write_many(writes)]
        return [f"✓ {self.files.write(w['path'], w['content'])}" for w in writes]

    def _format_gen_summary(
        self,
        writes:   list[str],
        errors:   list[str],
        items:    list[dict],
        waves:    list[list[dict]],
        has_deps: bool,
        elapsed:  float,
    ) -> str:
        wave_note = f"  ({len(waves)} waves)" if has_deps else ""
        lines     = writes + errors + [
            f"\033[90m⏱  {elapsed:.1f}s total for {len(items)} files{wave_note}  "
            f"(sequential est. ≈ {elapsed * len(items):.0f}s)\033[0m"
        ]
        summary = "\n".join(lines)
        self._log(summary, "result")
        return summary

    # ── Display helpers ───────────────────────────────────────────────

    def _print_todo_tree(self, tasks: list[dict], indent: str = "    ") -> None:
        current_id = self.todo._load().get("current_task_id")
        for t in tasks:
            icon = self._STATUS_ICONS.get(t["status"], "?")
            tag  = " 👈 CURRENT" if t.get("id") == current_id else ""
            print(f"{indent}{icon} {t['id']}: {t['description']}{tag}")
            for key in ("result", "error"):
                if val := t.get(key):
                    print(f"{indent}   {key.capitalize()}: {val[:100]}{'...' if len(val) > 100 else ''}")
            if t.get("children"):
                self._print_todo_tree(t["children"], indent + "    ")

    def _print_stats(self) -> None:
        s    = self.counter.summary()
        sep  = "─" * 48
        print("\n".join(("", sep,
            f"  Total requests : {s['total_requests']}  (✓{s['success']} ✗{s['errors']})",
            f"  Prompt tokens  : {s['tokens_prompt']:,}",
            f"  Think tokens   : {s['tokens_reasoning']:,}",
            f"  Resp  tokens   : {s['tokens_response']:,}",
            f"  Total tokens   : {s['tokens_total']:,}",
            f"  Uptime         : {s['uptime_seconds']}s",
            f"  Avg req/min    : {s['avg_requests_per_min']}",
            f"  By model       : {s['by_model']}", sep, "",
        )))

    def _print_progress(self) -> None:
        plan = self.mem.get("current_plan")
        if not plan or not isinstance(plan, dict):
            print("  No active plan. Use task_plan tool to create one.")
            return
        steps, cur = plan.get("steps", []), plan.get("current_step", 0)
        print(f"\n  Task: {plan.get('task', '')}")
        for i, step in enumerate(steps):
            icon = "✓" if i < cur else ("▶" if i == cur else "○")
            print(f"  {icon} [{i + 1}/{len(steps)}] {step}")

    def _print_todo(self) -> None:
        ts = self.todo.status()
        if "error" in ts:
            print("  No active todo list.")
            return
        current = ts.get("current_task")
        lines   = [
            f"\n  📋 Task: {ts['task']}",
            f"  Progress: {ts['progress']}",
            *(
                [f"  Current: {current['id']} - {current['description']} [{current['status']}]"]
                if current else []
            ),
            f"  Stats: {ts['stats']}", "", "  Full tree:",
        ]
        print("\n".join(lines))
        self._print_todo_tree(ts["tasks"])
        print()

    def _print_logs(self, n: int = 20) -> None:
        records = logger.read_recent(n)
        if not records:
            print("  No log records found.")
            return
        sep = "  " + "─" * 72
        print(f"\n  Last {len(records)} log records from {logger.LOG_FILE}:")
        print(sep)
        for r in records:
            print("  " + logger.format_record(r))
        print(sep + "\n")

    def _print_help(self) -> None:
        Y, G, R = "\033[93m", "\033[90m", "\033[0m"
        lines   = ["", f"  {Y}Chat Commands{R}"]
        lines  += [f"    {G}•{R} {k:<24} {v}" for k, v in self._CHAT_COMMANDS.items()]
        lines  += ["", f"  {Y}Agent Tools{R}"]
        for cat, fn in self._TOOLS_BY_CATEGORY.items():
            lines.append(f"  {G}{cat}{R}")
            for tool_name in fn():
                desc = next((t["description"] for t in TOOLS if t["name"] == tool_name), "")
                lines.append(f"    {G}•{R} {tool_name:<28} {str(desc)[:80]}")
            lines.append("")
        print("\n".join(lines))

    # ── Chat loop ─────────────────────────────────────────────────────

    def chat(self) -> None:
        """Start the interactive REPL."""
        thin, thick = "─" * 65, "═" * 65
        env  = "☁️  Colab" if IS_COLAB else "💻 Local"
        hb   = (
            f"✓ active ({Config.HEARTBEAT_INTERVAL}s)"
            if self._heartbeat_thread and self._heartbeat_thread.is_alive()
            else "✗ stopped"
        )
        print("\n".join(("", thick,
            f"  🤖  Universal Agentic AI  │  {Config.MODEL}",
            f"  📁  Workspace : {Config.BASE_DIR}",
            f"  📡  Streaming : {'ON' if self.stream else 'OFF'}   {env}",
            f"  💓  Heartbeat : {hb}",
            f"  📋  Logs      : {logger.LOG_FILE}", thin,
            "  Commands: exit | status | stats | history | progress | todo",
            "            logs [N] | reset | clear | stream | heartbeat | help",
            thick, "",
        )))

        while True:
            try:
                task = input("\n[YOU] ").strip()
                if not task:
                    continue
                cmd = task.lower()

                if cmd in ("exit", "quit", "q"):
                    self.counter.persist_stats()
                    self.stop_background_heartbeat()
                    print(f"👋  Goodbye!\n{self.counter}")
                    break
                elif cmd == "status":
                    print(json.dumps(self._agent_status(), indent=2, default=str))
                elif cmd == "stats":
                    self._print_stats()
                elif cmd == "history":
                    for session in self.counter.load_historical():
                        print(json.dumps(session, indent=2, default=str))
                elif cmd == "stream":
                    self.stream = not self.stream
                    print(f"  Streaming {'ON ✓' if self.stream else 'OFF ✗'}")
                elif cmd == "reset":
                    self.history = []
                    print("  ✓ Conversation history cleared")
                elif cmd == "clear":
                    self.mem.clear()
                    print("  ✓ Persistent memory cleared")
                elif cmd == "progress":
                    self._print_progress()
                elif cmd == "todo":
                    self._print_todo()
                elif cmd.startswith("logs"):
                    parts = cmd.split()
                    self._print_logs(int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 20)
                elif cmd == "heartbeat":
                    print(json.dumps(HeartbeatManager.check(self.tools_store), indent=2, default=str))
                elif cmd == "help":
                    self._print_help()
                else:
                    result = self.run(task)
                    if not self.stream:
                        print(f"\n[RESULT]\n{result}\n")
                    print(f"\n\033[90m{self.counter}\033[0m")

            except KeyboardInterrupt:
                print("\n  Interrupted — type 'exit' to quit.")
            except Exception as e:
                self._log(f"Chat error: {e}", "error")

    # ── Git helper ───────────────────────────────────────────────────────────
    def _git_run(self, git_args: str, directory: str = "") -> str:
        """Run a git sub-command in the workspace or specified directory."""
        cwd = str(Config.FILES_DIR / directory) if directory else str(Config.BASE_DIR)
        result = subprocess.run(
            f"git {git_args}", shell=True, capture_output=True, text=True,
            cwd=cwd, timeout=30,
        )
        parts = [s for s in (result.stdout.strip(), result.stderr.strip()) if s]
        parts.append(f"[exit: {result.returncode}]")
        return "\n".join(parts)

    # ── Database helpers ──────────────────────────────────────────────────────
    def _sql_query(self, db_path: str, sql: str, params: list | None = None) -> str:
        """Execute *sql* against a SQLite file and return results as JSON."""
        import sqlite3 as _sqlite3
        p = (Config.FILES_DIR / db_path).resolve()
        try:
            con = _sqlite3.connect(str(p))
            con.row_factory = _sqlite3.Row
            cur = con.execute(sql, params or [])
            rows = [dict(r) for r in cur.fetchall()]
            con.close()
            return json.dumps(rows, default=str, indent=2)
        except Exception as e:
            return f"SQL error: {e}"

    def _sql_tables(self, db_path: str) -> str:
        """List all tables in a SQLite database."""
        return self._sql_query(
            db_path,
            "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY name"
        )

    # ── Browser fetch ─────────────────────────────────────────────────────────
    def _browser_fetch(self, url: str, wait_for: str = "", max_chars: int = 8000) -> str:
        """Fetch a JS-rendered page using playwright (auto-installs if missing)."""
        try:
            from playwright.sync_api import sync_playwright
        except ImportError:
            self.shell.run("pip install -q playwright && playwright install chromium")
            try:
                from playwright.sync_api import sync_playwright
            except ImportError:
                return "Error: playwright installation failed. Run: pip install playwright && playwright install chromium"
        try:
            with sync_playwright() as pw:
                browser = pw.chromium.launch(headless=True)
                page    = browser.new_page()
                page.goto(url, timeout=Config.tool_timeout("browser_fetch") * 1000)
                if wait_for:
                    page.wait_for_selector(wait_for, timeout=10000)
                content = page.inner_text("body")
                browser.close()
            text = re.sub(r'\s+', ' ', content).strip()
            return text[:max_chars] + ("…(truncated)" if len(text) > max_chars else "")
        except Exception as e:
            return f"Browser fetch error: {e}"

    # ── Email ─────────────────────────────────────────────────────────────────
    def _send_email(self, to: str, subject: str, body: str,
                    cc: str = "", html: bool = False) -> str:
        """Send email via SMTP. Reads credentials from env/Colab Secrets."""
        import smtplib
        from email.mime.text import MIMEText
        from email.mime.multipart import MIMEMultipart

        smtp_host = os.environ.get("SMTP_HOST", "smtp.gmail.com")
        smtp_port = int(os.environ.get("SMTP_PORT", "587"))
        smtp_user = os.environ.get("SMTP_USER", "")
        smtp_pass = os.environ.get("SMTP_PASSWORD", "")

        if not smtp_user or not smtp_pass:
            return ("Error: SMTP credentials not configured.\n"
                    "Set SMTP_HOST, SMTP_PORT, SMTP_USER, SMTP_PASSWORD in env/Colab Secrets.")
        try:
            msg = MIMEMultipart("alternative")
            msg["From"]    = smtp_user
            msg["To"]      = to
            msg["Subject"] = subject
            if cc:
                msg["Cc"] = cc
            msg.attach(MIMEText(body, "html" if html else "plain"))
            with smtplib.SMTP(smtp_host, smtp_port) as s:
                s.starttls()
                s.login(smtp_user, smtp_pass)
                recipients = [to] + ([cc] if cc else [])
                s.sendmail(smtp_user, recipients, msg.as_string())
            return f"✓ Email sent to {to}"
        except Exception as e:
            return f"Email error: {e}"

    # ── Workspace diff ────────────────────────────────────────────────────────
    def _workspace_diff(self) -> str:
        """Show files changed since this agent session started."""
        try:
            snap_key = "_workspace_snapshot"
            current: dict[str, str] = {}
            for f in Config.FILES_DIR.rglob("*"):
                if f.is_file() and not f.name.startswith("."):
                    try:
                        rel = str(f.relative_to(Config.FILES_DIR))
                        current[rel] = f"{f.stat().st_size}:{f.stat().st_mtime:.0f}"
                    except Exception:
                        pass

            snapshot = self.mem.get(snap_key) or {}
            if not snapshot:
                self.mem.save(snap_key, current, tag="system")
                return "Workspace snapshot taken. Run again after changes to see diff."

            added    = [k for k in current if k not in snapshot]
            deleted  = [k for k in snapshot if k not in current]
            modified = [k for k in current if k in snapshot and current[k] != snapshot[k]]

            lines = [f"📊 Workspace diff since session start:"]
            if added:    lines.append(f"  ➕ Added ({len(added)}):    " + ", ".join(added[:20]))
            if modified: lines.append(f"  ✏️  Modified ({len(modified)}): " + ", ".join(modified[:20]))
            if deleted:  lines.append(f"  ❌ Deleted ({len(deleted)}):  " + ", ".join(deleted[:20]))
            if not (added or modified or deleted):
                lines.append("  No changes detected.")

            # Update snapshot
            self.mem.save(snap_key, current, tag="system")
            return "\n".join(lines)
        except Exception as e:
            return f"Workspace diff error: {e}"

    # ── Cancel running task ───────────────────────────────────────────────────
    def _cancel_running_task(self) -> str:
        """Set the cancellation flag so the next run() iteration aborts."""
        self._cancel_flag = True
        return "🛑 Cancellation flag set — current task will stop at next iteration boundary."

    def reset(self) -> None:
        """Clear conversation history."""
        self.history = []

    def export_conversation(
        self,
        fmt: str = "md",
        path: str | None = None,
    ) -> str:
        """
        Export the full conversation history to a readable file.

        Args:
            fmt:  'md' (Markdown), 'json' (raw JSON), or 'html'
            path: output path relative to FILES_DIR. Auto-generated if None.

        Returns the path of the written file.
        """
        ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
        ext  = {"md": ".md", "json": ".json", "html": ".html"}.get(fmt, ".md")
        dest = Path(path) if path else Config.FILES_DIR / f"conversation_{ts}{ext}"
        dest.parent.mkdir(parents=True, exist_ok=True)

        if fmt == "json":
            dest.write_text(json.dumps(self.history, indent=2, default=str), encoding="utf-8")

        elif fmt == "html":
            rows = []
            for msg in self.history:
                role    = msg.get("role", "")
                content = msg.get("content") or ""
                if isinstance(content, list):
                    content = " ".join(p.get("text","") for p in content if isinstance(p,dict))
                color = {"user":"#1a3a6a","assistant":"#1a4a2a","tool":"#3a2a0a"}.get(role,"#222")
                rows.append(
                    f'<div style="background:{color};padding:10px;margin:4px 0;border-radius:6px;'
                    f'font-family:monospace;white-space:pre-wrap">'
                    f'<b style="color:#aaa">{role.upper()}</b><br>{str(content)[:5000]}</div>'
                )
            html = (
                f'<!DOCTYPE html><html><head><meta charset="utf-8">'
                f'<title>Conversation {ts}</title>'
                f'<style>body{{background:#0d0f12;color:#d4dae8;margin:20px}}</style>'
                f'</head><body>{"".join(rows)}</body></html>'
            )
            dest.write_text(html, encoding="utf-8")

        else:  # Markdown
            lines = [f"# Conversation Export  —  {ts}\n "]
            for msg in self.history:
                role    = msg.get("role","")
                content = msg.get("content") or ""
                if isinstance(content, list):
                    content = " ".join(p.get("text","") for p in content if isinstance(p,dict))
                tcs = msg.get("tool_calls", [])
                lines.append(f"## {role.upper()}")
                lines.append(str(content)[:5000])
                if tcs:
                    lines.append(f"*Tool calls: {', '.join(tc['function']['name'] for tc in tcs)}*")
                lines.append("")
            dest.write_text("".join(lines), encoding="utf-8")

        self._log(f"Conversation exported → {dest}", "system")
        return str(dest)

    def structured_query(
        self,
        prompt: str,
        schema_desc: str = "Return valid JSON only.",
        max_tokens: int = 2000,
    ) -> dict | list | None:
        """
        Ask the model a question and force a structured JSON response.
        Uses response_format=json_object when supported, falls back to
        prompt-level instruction otherwise.
        """
        full_prompt = f"{prompt}\n\n{schema_desc}\nRespond with ONLY valid JSON."

        def attempt(model: str):
            """Try the query with a given model, handling JSON/plain fallback internally."""
            if self.rate_limiter:
                self.rate_limiter.wait_if_needed()
            try:
                # Try JSON mode first (if supported)
                resp = self.client.chat.completions.create(
                    model=model,
                    max_tokens=max_tokens,
                    response_format={"type": "json_object"},
                    messages=[{"role": "user", "content": full_prompt}],
                )
            except Exception:
                # Fall back to plain call with JSON instruction in prompt
                resp = self.client.chat.completions.create(
                    model=model,
                    max_tokens=max_tokens,
                    messages=[{"role": "user", "content": full_prompt}],
                )
            raw = (resp.choices[0].message.content or "").strip()
            raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.MULTILINE).strip()
            return json.loads(raw)

        try:
            return attempt(Config.MODEL)
        except Exception as e:
            if self._should_fallback(e):
                try:
                    return attempt(Config.FALLBACK_MODEL)
                except Exception as e2:
                    raise Exception(f"structured_query failed with primary and fallback: {e}; {e2}")
            else:
                raise


## CELL 13: Tools Schema & System Prompt Builder

In [37]:
# @title
# ---------- Tools Schema ----------
def _props(**fields: Any) -> dict:
    """Convert keyword fields into JSON-schema property dicts."""
    return {k: v if isinstance(v, dict) else {"type": v} for k, v in fields.items()}

def _params(required: list[str] | None = None, **fields: Any) -> dict:
    """Build a JSON-schema 'parameters' object."""
    p: dict[str, Any] = {"type": "object", "properties": _props(**fields)}
    if required:
        p["required"] = required
    return p

def _tool(name: str, desc: str, params: dict | None = None) -> dict:
    """Construct a single tool definition dict."""
    return {
        "name": name,
        "description": desc,
        "parameters": params or {"type": "object", "properties": {}},
    }

_S, _I, _B = "string", "integer", "boolean"
_KEY  = {"key": _S}
_PATH = {"path": _S}
_TASK = {"task_id": {"type": _S, "description": "Task ID (from todo_status)"}}

TOOLS: list[dict] = [
    # Memory
    _tool("memory_save",   "Save a value to persistent memory.",
          _params(["key", "value"], key=_S, value={}, tag=_S)),
    _tool("memory_get",    "Retrieve a memory value by key.",
          _params(["key"], **_KEY)),
    _tool("memory_list",   "List all memory keys, optionally filtered by tag.",
          _params(tag=_S)),
    _tool("memory_search", "Search memory by query string.",
          _params(["query"], query=_S)),
    _tool("memory_delete", "Delete a memory key.",
          _params(["key"], **_KEY)),

    # Files (workspace-sandboxed)
    _tool("file_read",
          "Read a file. Use start_line/end_line for large files (1-indexed, inclusive).",
          _params(["path"], path=_S,
                  start_line={"type": _I, "description": "First line (1-indexed)"},
                  end_line={"type":   _I, "description": "Last line (inclusive)"})),
    _tool("file_write",
          "Write or append to a file. backup=true saves .bak before overwriting.",
          _params(["path", "content"], path=_S, content=_S,
                  append={"type": _B, "default": False},
                  backup={"type": _B, "default": False})),
    _tool("file_write_many",
          "Write multiple files at once in parallel (use for 3+ files).",
          _params(["writes"], writes={
              "type": "array",
              "items": {"type": "object", "required": ["path", "content"],
                        "properties": _props(path=_S, content=_S, append=_B)}})),
    _tool("file_list",
          "List files in workspace. metadata=true for size/date/type.",
          _params(directory=_S,
                  metadata={"type": _B, "default": False})),
    _tool("file_delete", "Delete a file.",
          _params(["path"], **_PATH)),
    _tool("file_search", "Search files by name or content.",
          _params(["query"], query=_S, directory=_S)),
    _tool("file_rename", "Rename or move a file within the workspace.",
          _params(["src", "dst"], src=_S, dst=_S)),

    # Shell
    _tool("shell_run",
          f"Run any shell command. Default cwd = {Config.FILES_DIR}. stdout+stderr captured.",
          _params(["command"], command=_S,
                  timeout={"type": _I, "default": Config.SHELL_TIMEOUT},
                  cwd={"type": _S, "default": "",
                       "description": "'' = files dir  '/' = workspace root  'src' = files/src/"})),

    # Code
    _tool("execute_python",
          "Execute Python snippet in sandbox. result variable is returned.",
          _params(["code"], code=_S, timeout={"type": _I, "default": Config.CODE_TIMEOUT})),

    # Diff
    _tool("diff_apply",
          "Apply unified diff or SEARCH/REPLACE blocks to a file.",
          _params(["path", "diff"], path=_S, diff=_S)),

    # Web
    _tool("web_search", "Search the web via DuckDuckGo.",
          _params(["query"], query=_S,
                  max_results={"type": _I, "default": 5, "minimum": 1, "maximum": 10})),
    _tool("web_fetch", "Fetch and extract text from any http/https URL.",
          _params(["url"], url=_S,
                  max_chars={"type": _I, "default": 5000, "minimum": 100, "maximum": 10000})),

    # Sub-tools
    _tool("subtool_create",
          "Create reusable Python sub-tool. Must define run(**kwargs). force=true overrides warnings.",
          _params(["name", "description", "code"],
                  name=_S, description=_S, code=_S,
                  force={"type": _B, "default": False})),
    _tool("subtool_run",  "Run a saved sub-tool by name.",
          _params(["name"], name=_S, kwargs={"type": "object"})),
    _tool("subtool_list", "List all saved sub-tools."),
    _tool("subtool_delete", "Delete a sub-tool.",
          _params(["name"], name=_S)),

    # Skills
    _tool("skill_list",   "List registered skills from core/Tools.md."),
    _tool("skill_create",
          "Create a new skill file and register it in core/Tools.md.",
          _params(["name", "code", "purpose", "trigger"],
                  name=_S, code=_S, purpose=_S, trigger=_S,
                  input_fmt=_S, output_fmt=_S)),
    _tool("skill_run",
          "Run a registered skill by name.",
          _params(["name"], name=_S,
                  input_data={"type": "object", "description": "Input data for the skill"})),
    _tool("skill_validate",
          "Validate one or all registered skills.",
          _params(name=_S)),

    # Todo
    _tool("todo_create",
          "Create todo list for a complex multi-step task.",
          _params(["task", "steps"],
                  task={"type": _S, "description": "Overall task description"},
                  steps={"type": "array", "items": {"type": _S},
                         "description": "List of main steps"})),
    _tool("todo_status", "Get complete todo tree with all tasks, status, and progress."),
    _tool("todo_add_subtask", "Add a subtask to any existing parent task.",
          _params(["parent_id", "description"],
                  parent_id={"type": _S}, description={"type": _S})),
    _tool("todo_complete", "Mark task completed. Auto-advances to next task.",
          _params(["task_id"], **_TASK,
                  result={"type": _S, "description": "Optional result/outcome"})),
    _tool("todo_fail", "Mark task failed. Auto-advances to next task.",
          _params(["task_id"], **_TASK,
                  error={"type": _S, "description": "Error message/reason"})),
    _tool("todo_set_current", "Manually set which task to work on next.",
          _params(["task_id"], **_TASK)),
    _tool("todo_update", "Update task description or add notes.",
          _params(["task_id"], **_TASK,
                  description={"type": _S}, notes={"type": _S})),

    # Needs
    _tool("post_need",
          "Post a structured request to Need.md for the user.",
          _params(["priority", "need", "reason", "blocking"],
                  priority=_S, need=_S, reason=_S, blocking=_S)),
    _tool("check_needs", "Return all PENDING requests from Need.md."),

    # Session & heartbeat
    _tool("save_session_log",
          "Append an entry to today's daily log.",
          _params(["entry"], entry=_S, category=_S)),
    _tool("search_memory",
          "Search through recent daily memory logs.",
          _params(["query"], query=_S,
                  days_back={"type": _I, "default": 7})),
    _tool("check_heartbeat", "Run a full system health check (heartbeat)."),

    # General file ops
    _tool("read_file",
          "Read any file by absolute or relative path (not workspace-sandboxed).",
          _params(["filepath"], filepath=_S, encoding=_S)),
    _tool("write_file",
          "Write to any file by path (creates parent dirs).",
          _params(["filepath", "content"], filepath=_S, content=_S)),
    _tool("append_file",
          "Append content to a file, optionally with timestamp.",
          _params(["filepath", "content"], filepath=_S, content=_S,
                  add_timestamp={"type": _B, "default": True})),
    _tool("delete_file", "Delete a file or directory recursively.",
          _params(["filepath"], filepath=_S)),
    _tool("list_directory",
          "List files/dirs with metadata.",
          _params(["path"], path=_S, pattern=_S)),
    _tool("search_files",
          "Search file contents by query in a directory.",
          _params(["query"], query=_S, directory=_S,
                  file_pattern={"type": _S, "default": "*.md"})),

    # Planning / status
    _tool("task_plan", "Save a step-by-step plan to memory.",
          _params(["task", "steps"], task=_S,
                  steps={"type": "array", "items": {"type": _S}})),
    _tool("agent_status", "Get full agent status: memory, subtools, files, API stats."),
    _tool("api_stats",    "Get detailed API request and token usage statistics."),

    # ── New tools ────────────────────────────────────────────────────────────
    # Git
    _tool("git_status",  "Get git status of the workspace.",
          _params(directory=_S)),
    _tool("git_diff",    "Show git diff (staged or unstaged).",
          _params(directory=_S, staged={"type": _B, "default": False})),
    _tool("git_commit",  "Stage all changes and commit with a message.",
          _params(["message"], message=_S, directory=_S)),
    _tool("git_push",    "Push commits to remote origin.",
          _params(directory=_S, branch=_S)),
    _tool("git_log",     "Show last N git commit messages.",
          _params(n={"type": _I, "default": 10}, directory=_S)),

    # Database (SQLite)
    _tool("sql_query",   "Execute a SQL query against a SQLite database file.",
          _params(["db_path", "sql"], db_path=_S, sql=_S,
                  params={"type": "array", "items": {}, "description": "Query parameters"})),
    _tool("sql_tables",  "List all tables in a SQLite database.",
          _params(["db_path"], db_path=_S)),

    # Browser (JS-rendered pages)
    _tool("browser_fetch", "Fetch a JS-rendered web page using a headless browser.",
          _params(["url"], url=_S,
                  wait_for={"type": _S, "description": "CSS selector to wait for"},
                  max_chars={"type": _I, "default": 8000})),

    # Email
    _tool("send_email",  "Send an email via SMTP (reads credentials from env/secrets).",
          _params(["to", "subject", "body"],
                  to=_S, subject=_S, body=_S,
                  cc=_S, html={"type": _B, "default": False})),

    # Semantic memory
    _tool("memory_semantic_search",
          "Semantic (TF-IDF) search over memory — finds conceptually related entries.",
          _params(["query"], query=_S, top_k={"type": _I, "default": 5})),

    # Conversation export
    _tool("export_conversation",
          "Export full conversation history to a file (md/json/html).",
          _params(fmt={"type": _S, "default": "md", "enum": ["md","json","html"]},
                  path=_S)),

    # Structured query
    _tool("structured_query",
          "Ask the model a question and get back a structured JSON response.",
          _params(["prompt"], prompt=_S, schema_desc=_S,
                  max_tokens={"type": _I, "default": 2000})),

    # Workspace diff
    _tool("workspace_diff",
          "Show files changed since session start (added/modified/deleted).",
          _params()),

    # Cancel current task
    _tool("cancel_task",
          "Cancel the currently-running queued task.",
          _params()),

    # Parallel generation
    _tool("parallel_generate",
          "Generate multiple files IN PARALLEL. Much faster than sequential for 3+ files.",
          _params(["items"],
                  items={"type": "array",
                         "items": {"type": "object", "required": ["path", "prompt"],
                                   "properties": _props(
                                       path={"type": _S},
                                       prompt={"type": _S},
                                       deps={"type": "array", "items": {"type": _S}})}},
                  system_prompt={"type": _S})),
]

OAI_TOOLS: list[dict] = [{"type": "function", "function": t} for t in TOOLS]

# ---------- System Prompt Builder ----------
def _extract_section(text: str, heading_prefix: str = "##") -> str:
    """
    Extract content after the first Markdown heading that starts with
    `heading_prefix`, stopping before the next heading of any level.
    Falls back to the full text if no matching heading is found.
    """
    lines = text.splitlines()
    capturing = False
    collected: list[str] = []

    for line in lines:
        stripped = line.lstrip()
        if not capturing:
            # Start capturing after the first matching heading
            if stripped.startswith(heading_prefix):
                capturing = True
        else:
            # Stop at the next Markdown heading (any level)
            if stripped.startswith("#"):
                break
            collected.append(line)

    result = "\n".join(collected).strip()
    return result if result else text.strip()


def build_system_prompt_from_context(context: dict[str, str], tools_str: str) -> str:
    """Build a rich system prompt from Code-2 identity files."""
    soul  = _extract_section(context.get("core/SOUL.md",   ""))
    user  = _extract_section(context.get("core/USER.md",   ""))
    mem   = _extract_section(context.get("core/MEMORY.md", ""))
    needs = _extract_section(context.get("core/Need.md",   ""))

    today = datetime.now().strftime("%Y-%m-%d")

    return (
        f"You are an autonomous AI agent powered by {Config.MODEL} via NVIDIA NIM.\n\n"
        f"=== AGENT IDENTITY ===\n{soul}\n\n"
        f"=== USER PROFILE ===\n{user}\n\n"
        f"=== LONG-TERM MEMORY ===\n{mem}\n\n"
        f"=== CURRENT NEEDS ===\n{needs}\n\n"
        f"=== AVAILABLE TOOLS ===\n{tools_str}\n\n"
        f"=== TODAY ===\n{today}\n\n"
        "Begin by checking for pending needs, then respond to the user.\n"
    )

print("✓ Tools schema and system prompt builder ready")

✓ Tools schema and system prompt builder ready


## CELL 14: API Key Setup & Entry Points

In [38]:
# @title
from __future__ import annotations


# ---------- API Key Setup ----------

# Error messages are defined as constants so they're easy to update and not
# duplicated across the Colab and local branches of setup_api_key().
_KEY_URL = "https://build.nvidia.com"

_MSG_NO_KEY_COLAB = (
    "\n❌ No NVIDIA API key found.\n"
    "   ➜  Colab Secrets: Left sidebar → 🔑 → Add NVIDIA_API_KEY\n"
    "   ➜  Or paste into NVIDIA_API_KEY = \"\" at top of file.\n"
    f"   ➜  Free key: {_KEY_URL}"
)
_MSG_NO_KEY_LOCAL = (
    "\n❌ No NVIDIA API key found.\n"
    "   ➜  Paste into NVIDIA_API_KEY = \"\" at top, or\n"
    "   ➜  export NVIDIA_API_KEY=nvapi-xxxx...\n"
    f"   ➜  Free key: {_KEY_URL}"
)


def setup_api_key() -> bool:
    """
    Resolve the NVIDIA API key using the following priority order:

    1. Already present in ``os.environ["NVIDIA_API_KEY"]``
    2. Colab Secrets (``google.colab.userdata``) — Colab only
    3. Interactive hidden prompt — local TTY only

    Returns ``True`` if a key was found and set, ``False`` otherwise.
    """
    # 1. Already set in the environment
    if os.environ.get("NVIDIA_API_KEY", "").strip():
        return True

    # 2. Colab Secrets — the primary Colab key source
    if IS_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("NVIDIA_API_KEY")
            if key and key.strip():
                os.environ["NVIDIA_API_KEY"] = key.strip()
                return True
        except Exception:
            pass  # Secret not set or userdata unavailable — fall through

    # 3. Interactive local prompt (hidden input)
    if not IS_COLAB and sys.stdin.isatty():
        try:
            import getpass
            key = getpass.getpass("\n🔑 Enter your NVIDIA API key (input hidden): ").strip()
            if key:
                os.environ["NVIDIA_API_KEY"] = key
                return True
        except Exception:
            pass  # Fall through to the error message below

    # 4. No key found — print environment-appropriate guidance
    print(_MSG_NO_KEY_COLAB if IS_COLAB else _MSG_NO_KEY_LOCAL)
    return False


# ---------- Entry Points ----------

def start_chat(
    auto_heartbeat:     bool      = True,
    heartbeat_interval: int | None = None,
) -> None:
    """Launch the interactive chat interface."""
    sep = "=" * 65
    print(f"{sep}\n  UNIVERSAL AUTONOMOUS AI AGENT — Interactive Chat\n{sep}")
    agent = Agent(stream=True)
    if auto_heartbeat:
        agent.start_background_heartbeat(interval=heartbeat_interval)
    agent.chat()


def setup_colab() -> bool:
    """
    Colab-specific setup helper: install dependencies, mount Drive,
    and prompt for the API key if absent.
    """
    print("[Colab Setup]")

    # Ensure openai is installed
    try:
        import openai  # noqa: F401
    except ImportError:
        print("Installing openai…")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
        print("  - installed")

    # Mount Google Drive (non-fatal if unavailable)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("  - Drive mounted")
    except Exception as e:
        print(f"  - Drive mount skipped ({e})")

    # Prompt for the API key using hidden input
    if not os.getenv("NVIDIA_API_KEY"):
        print("Please set NVIDIA_API_KEY in Colab secrets or enter below.")
        try:
            import getpass
            key = getpass.getpass("Enter NVIDIA_API_KEY (or press Enter to skip): ").strip()
        except Exception:
            key = ""
        if key:
            os.environ["NVIDIA_API_KEY"] = key
            print("  - key set for this session")

    print(
        f"\n[Colab] Workspace root: {Config.BASE_DIR}\n"
        "Start with:\n"
        "  agent = Agent()\n"
        "  agent.start_background_heartbeat()\n"
        "  agent.run('Your task here')\n"
        "  # or: start_chat()"
    )
    return True


def demo() -> None:
    """Run a quick demonstration of the agent across several representative tasks."""
    sep = "=" * 65
    print(f"\n{sep}\n  AUTONOMOUS AGENT — Demonstration\n{sep}\n")

    agent = Agent(stream=True)
    agent.start_background_heartbeat(interval=3600)

    tasks = [
        "What is your current status?",
        "List your available sub-tools and skills.",
        "Check Need.md for any pending requests.",
        "Read MEMORY.md and summarise key points.",
    ]
    for task in tasks:
        print(f"\n[User] {task}")
        response = agent.run(task)
        print(f"\n[Agent] {response}\n")
        time.sleep(1)

    agent.stop_background_heartbeat()
    print("[Demo] Complete.")


# ---------- Initialise ----------

print("✓ Entry points ready (setup_api_key, start_chat, setup_colab, demo)")
setup_api_key()

✓ Entry points ready (setup_api_key, start_chat, setup_colab, demo)


True

## CELL 16: Core Evolution

In [39]:
# @title
from __future__ import annotations

from typing import Any

# ══════════════════════════════════════════════════════════════════════════════
#  CELL 16 — Core File Evolution System
#
#  Problem this solves:
#    SOUL.md, USER.md, MEMORY.md, HEARTBEAT.md are written once at init with
#    placeholder content and then NEVER updated again — they stay static no
#    matter how much you use the agent.
#
#  What this cell adds:
#    CoreEvolutionManager — a lightweight manager that hooks into the Agent
#    after each run() call and after each heartbeat, using a small LLM call
#    to extract and persist what matters into the right file.
#
#  File responsibilities (what each file SHOULD contain):
#    SOUL.md      — Agent's learned identity: capabilities, preferred patterns,
#                   tools it uses most, lessons from past failures
#    USER.md      — User profile: name/handle, tech stack, preferences,
#                   common task types, communication style
#    MEMORY.md    — Long-term facts: key decisions, project names, file
#                   structures, recurring errors & fixes, important outputs
#    HEARTBEAT.md — Checklist + log of actual health-check results over time
#    Need.md      — Agent → User requests (already active)
#    Tools.md     — Skill registry (already active via skill_create)
# ══════════════════════════════════════════════════════════════════════════════


class CoreEvolutionManager:
    """
    Actively evolves the core identity files after every conversation turn
    and every heartbeat check, so they grow richer the more you use the agent.

    Integration (add to Agent.__init__ after self.todo = TodoListManager(...)):

        self.core_evo = CoreEvolutionManager(
            client=self.client,
            counter=self.counter,
            rate_limiter=self.rate_limiter,
        )

    Then at the end of Agent.run(), just before ``return result``:

        self.core_evo.on_turn(task, result, self.history)

    And in HeartbeatManager.check(), after the status dict is built:

        CoreEvolutionManager.on_heartbeat(status)
    """

    # Separate token budget for evolution LLM calls (prevents runaway costs).
    # Set to 0 to disable evolution entirely. Tracked independently from agent budget.
    EVOLUTION_TOKEN_BUDGET: int = int(os.environ.get("AGENT_EVO_TOKEN_BUDGET", "50000"))
    _evo_tokens_used: int = 0  # class-level counter shared across instances

    # How many conversation turns between full MEMORY.md consolidations.
    # Lower = more API calls but fresher memory. Raise if costs matter.
    CONSOLIDATE_EVERY = 10

    # Maximum characters kept in MEMORY.md before consolidation trims it.
    MEMORY_MAX_CHARS = 8_000

    # Sections written/managed inside each file
    _SOUL_SECTION       = "## Learned Behaviours"
    _USER_SECTION       = "## Observed Preferences"
    _MEMORY_SECTION     = "## Long-term Memory"
    _HEARTBEAT_SECTION  = "## Health Log"

    def __init__(
        self,
        client:       Any,
        counter:      APICounter,
        rate_limiter: RateLimiter | None = None,
    ) -> None:
        self._client       = client
        self._counter      = counter
        self._rate_limiter = rate_limiter
        self._turn_count   = 0

    # ── Public hooks ─────────────────────────────────────────────────

    def on_turn(self, task: str, result: str, history: list[dict]) -> None:
        """
        Called after every ``Agent.run()`` completes.
        Extracts facts from the exchange and updates MEMORY.md and USER.md.
        Every ``CONSOLIDATE_EVERY`` turns it also re-summarises MEMORY.md
        and refreshes SOUL.md with observed behavioural patterns.
        """
        self._turn_count += 1
        try:
            self._update_memory(task, result)
            self._update_user_profile(task, result)
            if self._turn_count % self.CONSOLIDATE_EVERY == 0:
                self._consolidate_memory()
                self._update_soul(history)
        except Exception as e:
            logger.warning(f"CoreEvolution.on_turn failed: {e}")

    @staticmethod
    def on_heartbeat(status: dict[str, Any]) -> None:
        """
        Called after every ``HeartbeatManager.check()`` completes.
        Appends a timestamped health snapshot to HEARTBEAT.md.
        No LLM call — pure structured write.
        """
        try:
            CoreEvolutionManager._append_heartbeat_log(status)
        except Exception as e:
            logger.warning(f"CoreEvolution.on_heartbeat failed: {e}")

    # ── MEMORY.md ─────────────────────────────────────────────────────

    def _update_memory(self, task: str, result: str) -> None:
        """
        Ask the LLM to extract up to 5 bullet-point facts worth remembering
        from this exchange and append them to MEMORY.md.
        Skips trivial exchanges (greetings, status checks, very short results).
        """
        if len(result) < 80 or task.lower() in ("status", "stats", "todo", "progress"):
            return  # Nothing worth persisting

        prompt = (
            "Extract up to 5 concrete facts worth remembering long-term from this "
            "exchange. Focus on: file names/paths created or modified, key decisions "
            "made, errors encountered and how they were fixed, project names, "
            "important configurations. Ignore pleasantries and status checks.\n\n"
            f"USER TASK:\n{task[:600]}\n\n"
            f"AGENT RESULT (first 800 chars):\n{result[:800]}\n\n"
            "Reply with ONLY a markdown bullet list (- fact). "
            "If nothing is worth remembering, reply with exactly: SKIP"
        )
        response = self._llm(prompt, max_tokens=300)
        if not response or response.strip() == "SKIP":
            return

        path    = Config.CORE_DIR / "MEMORY.md"
        content = FileManager.read_file(path, default="")
        ts      = datetime.now().strftime("%Y-%m-%d %H:%M")
        entry   = f"\n### [{ts}] Turn {self._turn_count}\n{response.strip()}\n"

        if self._MEMORY_SECTION in content:
            updated = content.replace(
                self._MEMORY_SECTION,
                self._MEMORY_SECTION + entry,
                1,
            )
        else:
            updated = content + f"\n{self._MEMORY_SECTION}\n{entry}"

        FileManager.write_file(path, updated)
        logger.info(f"MEMORY.md updated (turn {self._turn_count})")

    def _consolidate_memory(self) -> None:
        """
        When MEMORY.md exceeds ``MEMORY_MAX_CHARS``, use the LLM to produce
        a condensed summary, replacing the verbose history with a tight digest.
        Keeps the last 2 turns verbatim so recent context is never lost.
        """
        path    = Config.CORE_DIR / "MEMORY.md"
        content = FileManager.read_file(path, default="")

        if len(content) < self.MEMORY_MAX_CHARS:
            return  # Not large enough to need consolidation yet

        prompt = (
            "The following is a long-term memory log for an AI agent. "
            "Consolidate it into a tight, structured summary (max 600 words) "
            "preserving all unique facts: project names, file paths, key decisions, "
            "recurring errors, user preferences. Discard duplicates and vague entries.\n\n"
            f"MEMORY LOG:\n{content[:6000]}\n\n"
            "Reply with ONLY the consolidated markdown content under the heading "
            f"'{self._MEMORY_SECTION}'."
        )
        consolidated = self._llm(prompt, max_tokens=700)
        if not consolidated:
            return

        ts      = datetime.now().strftime("%Y-%m-%d %H:%M")
        updated = (
            f"# MEMORY.md\n"
            f"*Last consolidated: {ts} (turn {self._turn_count})*\n\n"
            + consolidated.strip() + "\n"
        )
        FileManager.write_file(path, updated)
        logger.info(f"MEMORY.md consolidated at turn {self._turn_count}")

    # ── USER.md ───────────────────────────────────────────────────────

    def _update_user_profile(self, task: str, result: str) -> None:
        """
        Infer user preferences and profile signals from the task phrasing
        and update USER.md. Only writes when something genuinely new is
        detected — avoids redundant rewrites.
        """
        prompt = (
            "From this single agent exchange, identify any NEW signals about the user: "
            "their tech stack, preferred language/framework, working style, project type, "
            "communication preferences (terse vs verbose), expertise level. "
            "Only report things not already obvious from context.\n\n"
            f"TASK: {task[:400]}\n"
            f"RESULT (first 300 chars): {result[:300]}\n\n"
            "Reply with ONLY a markdown bullet list (- observation). "
            "If nothing new is detectable, reply exactly: SKIP"
        )
        response = self._llm(prompt, max_tokens=200)
        if not response or response.strip() == "SKIP":
            return

        path    = Config.CORE_DIR / "USER.md"
        content = FileManager.read_file(path, default="")
        ts      = datetime.now().strftime("%Y-%m-%d")
        entry   = f"\n#### [{ts}]\n{response.strip()}\n"

        if self._USER_SECTION in content:
            updated = content.replace(
                self._USER_SECTION,
                self._USER_SECTION + entry,
                1,
            )
        else:
            updated = content + f"\n{self._USER_SECTION}\n{entry}"

        FileManager.write_file(path, updated)
        logger.info("USER.md updated with new profile signals")

    # ── SOUL.md ───────────────────────────────────────────────────────

    def _update_soul(self, history: list[dict]) -> None:
        """
        Every ``CONSOLIDATE_EVERY`` turns, reflect on recent conversation
        history and update SOUL.md with observed behavioural patterns —
        which tools work well, recurring task types, successful strategies.
        """
        if not history:
            return

        # Build a compact transcript of the recent history
        lines: list[str] = []
        for msg in history[-30:]:
            role    = msg.get("role", "")
            content = msg.get("content") or ""
            if isinstance(content, list):
                content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
            if role in ("user", "assistant") and content:
                lines.append(f"{role.upper()}: {str(content)[:200]}")
            elif role == "tool" and content:
                lines.append(f"TOOL RESULT: {str(content)[:120]}")

        transcript = "\n".join(lines)

        prompt = (
            "Based on this recent conversation history, identify 3-5 behavioural "
            "patterns for the AI agent: which tools it uses most effectively, "
            "what task types it handles well, any patterns in how it breaks down "
            "problems, recurring strategies that work. Be specific and actionable.\n\n"
            f"HISTORY:\n{transcript}\n\n"
            "Reply with ONLY a markdown bullet list under the heading "
            f"'{self._SOUL_SECTION}'."
        )
        response = self._llm(prompt, max_tokens=350)
        if not response:
            return

        path    = Config.CORE_DIR / "SOUL.md"
        content = FileManager.read_file(path, default="")
        ts      = datetime.now().strftime("%Y-%m-%d %H:%M")
        entry   = f"\n#### Observed at {ts} (turn {self._turn_count})\n{response.strip()}\n"

        if self._SOUL_SECTION in content:
            updated = content.replace(
                self._SOUL_SECTION,
                self._SOUL_SECTION + entry,
                1,
            )
        else:
            updated = content + f"\n{self._SOUL_SECTION}\n{entry}"

        FileManager.write_file(path, updated)
        logger.info(f"SOUL.md updated at turn {self._turn_count}")

    # ── HEARTBEAT.md ──────────────────────────────────────────────────

    @staticmethod
    def _append_heartbeat_log(status: dict[str, Any]) -> None:
        """
        Write a concise structured snapshot of the heartbeat result into
        HEARTBEAT.md so there is a running log of system health over time.
        """
        path    = Config.CORE_DIR / "HEARTBEAT.md"
        content = FileManager.read_file(path, default="")
        ts      = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        checks  = status.get("checks", {})
        actions = status.get("actions", [])
        overall = "⚠ ATTENTION" if status.get("action_required") else "✓ OK"

        check_lines = "\n".join(
            f"  - {k}: {v}" for k, v in checks.items()
        )
        action_lines = (
            "\n".join(f"  - {a}" for a in actions)
            if actions else "  - none"
        )
        entry = (
            f"\n### [{ts}] {overall}\n"
            f"**Checks:**\n{check_lines}\n"
            f"**Actions required:**\n{action_lines}\n"
        )

        section = "## Health Log"
        if section in content:
            updated = content.replace(section, section + entry, 1)
        else:
            updated = content + f"\n{section}\n{entry}"

        FileManager.write_file(path, updated)

    # ── LLM helper ───────────────────────────────────────────────────

    def _llm(self, prompt: str, max_tokens: int = 300) -> str | None:
        """
        Make a lightweight, non-streaming LLM call for extraction tasks.
        Enforces a separate EVOLUTION_TOKEN_BUDGET to prevent runaway API costs.
        Uses a separate counter record so it doesn't inflate agent request stats.
        """
        # Check evolution token budget before calling
        if (self.EVOLUTION_TOKEN_BUDGET > 0 and
                CoreEvolutionManager._evo_tokens_used >= self.EVOLUTION_TOKEN_BUDGET):
            logger.warning(
                f"CoreEvolution: token budget exhausted "
                f"({CoreEvolutionManager._evo_tokens_used:,} / "
                f"{self.EVOLUTION_TOKEN_BUDGET:,}). "
                f"Skipping. Set AGENT_EVO_TOKEN_BUDGET env var to increase."
            )
            return None

        try:
            # Inner attempt with fallback
            def should_fallback(exc: Exception) -> bool:
                err = str(exc).lower()
                triggers = [
                    '404', 'not found', 'model not found', 'deleted', 'unavailable',
                    '429', 'rate limit', 'too many', 'quota',
                    '500', '502', '503', '504', 'server error', 'overloaded',
                    'timeout', 'connection', 'network',
                ]
                return any(t in err for t in triggers)

            if self._rate_limiter:
                self._rate_limiter.wait_if_needed()

            model_used = Config.MODEL
            try:
                resp = self._client.chat.completions.create(
                    model=Config.MODEL,
                    max_tokens=max_tokens,
                    messages=[{"role": "user", "content": prompt}],
                )
            except Exception as e:
                if should_fallback(e):
                    if self._rate_limiter:
                        self._rate_limiter.wait_if_needed()
                    try:
                        resp = self._client.chat.completions.create(
                            model=Config.FALLBACK_MODEL,
                            max_tokens=max_tokens,
                            messages=[{"role": "user", "content": prompt}],
                        )
                        model_used = Config.FALLBACK_MODEL
                    except Exception as e2:
                        raise Exception(f"Primary failed: {e}. Fallback failed: {e2}")
                else:
                    raise

            text = resp.choices[0].message.content or ""

            # Track evolution token usage against separate budget
            usage = getattr(resp, "usage", None)
            if usage:
                tokens_in  = getattr(usage, "prompt_tokens",    0) or 0
                tokens_out = getattr(usage, "completion_tokens", 0) or 0
                CoreEvolutionManager._evo_tokens_used += tokens_in + tokens_out
                self._counter.record(
                    model=model_used,
                    tokens_in=tokens_in,
                    tokens_out=tokens_out,
                )
            return text.strip() if text.strip() else None

        except Exception as e:
            logger.warning(f"CoreEvolution LLM call failed: {e}")
            return None


# ══════════════════════════════════════════════════════════════════════════════
#  Patch Agent and HeartbeatManager to wire in CoreEvolutionManager
#
#  We patch rather than rewrite so this cell is self-contained and can be
#  dropped into any existing notebook without touching cells 9–12.
# ══════════════════════════════════════════════════════════════════════════════

# ── Sentinel stored on the classes survives cell reruns ──────────────────────
# Using a function attribute as sentinel fails because each rerun captures the
# already-patched function as _original, building an infinitely recursive chain.
# Storing the sentinel on Agent / HeartbeatManager persists in the notebook
# global namespace, so re-running this cell is always safe.

# ── 1. Patch Agent.__init__ to create self.core_evo ──────────────────────────
if not getattr(Agent, "_evo_init_patched", False):
    _original_agent_init = Agent.__init__

    def _patched_agent_init(self, verbose: bool = True, stream: bool = True) -> None:
        _original_agent_init(self, verbose=verbose, stream=stream)
        self.core_evo = CoreEvolutionManager(
            client=self.client,
            counter=self.counter,
            rate_limiter=self.rate_limiter,
        )
        logger.info("CoreEvolutionManager attached to agent")

    Agent.__init__ = _patched_agent_init
    Agent._evo_init_patched = True
    print("  ✓ Agent.__init__ patched")
else:
    print("  ℹ  Agent.__init__ already patched — skipped")


# ── 2. Patch Agent.run to call core_evo.on_turn after each response ──────────
if not getattr(Agent, "_evo_run_patched", False):
    _original_agent_run = Agent.run

    def _patched_agent_run(self, task: str) -> str:
        result = _original_agent_run(self, task)
        if hasattr(self, "core_evo"):
            self.core_evo.on_turn(task, result, self.history)
        return result

    Agent.run = _patched_agent_run
    Agent._evo_run_patched = True
    print("  ✓ Agent.run patched")
else:
    print("  ℹ  Agent.run already patched — skipped")


# ── 3. Patch HeartbeatManager.check to call on_heartbeat after each check ────
if not getattr(HeartbeatManager, "_evo_hb_patched", False):
    _original_hb_check = HeartbeatManager.check

    def _patched_hb_check(tools_store=None):
        status = _original_hb_check(tools_store)
        CoreEvolutionManager.on_heartbeat(status)
        return status

    HeartbeatManager.check = staticmethod(_patched_hb_check)
    HeartbeatManager._evo_hb_patched = True
    print("  ✓ HeartbeatManager.check patched")
else:
    print("  ℹ  HeartbeatManager.check already patched — skipped")


# ══════════════════════════════════════════════════════════════════════════════
#  Manual helpers — call these directly if you want immediate updates
# ══════════════════════════════════════════════════════════════════════════════

def evolve_now(agent: "Agent") -> None:
    """
    Force an immediate full evolution cycle:
    consolidate MEMORY.md, update SOUL.md, update HEARTBEAT.md.
    Useful to call at the end of a long work session.

    Usage:  evolve_now(agent)
    """
    print("🧬 Running full evolution cycle…")
    evo = agent.core_evo if hasattr(agent, "core_evo") else CoreEvolutionManager(
        client=agent.client, counter=agent.counter, rate_limiter=agent.rate_limiter,
    )
    evo._consolidate_memory()
    evo._update_soul(agent.history)
    CoreEvolutionManager.on_heartbeat(HeartbeatManager.check(agent.tools_store))
    print("✓ MEMORY.md consolidated")
    print("✓ SOUL.md updated")
    print("✓ HEARTBEAT.md logged")


def show_core_files(agent: "Agent" | None = None) -> None:
    """
    Pretty-print all 6 core files so you can see their current state.

    Usage:  show_core_files()          # reads files directly
            show_core_files(agent)     # same, just for convenience
    """
    files = ["SOUL.md", "USER.md", "MEMORY.md", "HEARTBEAT.md", "Need.md", "Tools.md"]
    sep   = "─" * 60
    for fname in files:
        path    = Config.CORE_DIR / fname
        content = FileManager.read_file(path, default="*(file missing)*")
        print(f"\n{sep}\n📄  {fname}\n{sep}")
        print(content[:2000])
        if len(content) > 2000:
            print(f"  … [{len(content) - 2000:,} more chars — open the file to see all]")


print("✓ CoreEvolutionManager ready")
print("  Core files will now evolve automatically as you use the agent.")
print("  Call evolve_now(agent) at any time to force a full cycle.")
print("  Call show_core_files() to inspect the current state of all files.")

  ✓ Agent.__init__ patched
  ✓ Agent.run patched
  ✓ HeartbeatManager.check patched
✓ CoreEvolutionManager ready
  Core files will now evolve automatically as you use the agent.
  Call evolve_now(agent) at any time to force a full cycle.
  Call show_core_files() to inspect the current state of all files.


## CELL 17: Persistance

In [40]:
# @title
from __future__ import annotations

import shutil
import threading
import zipfile
from pathlib import Path
from typing import Any

# ══════════════════════════════════════════════════════════════════════════════
#  CELL 17 — Session Persistence
#
#  Problem: Colab wipes /content/ on every session reset. All core files,
#  memory, skills, daily logs and the agent manifest are lost.
#
#  Solution: Three-layer persistence backed by Google Drive:
#    1. AUTO-SAVE  — background thread saves a ZIP to Drive every N minutes
#    2. MANUAL     — call save_session() at any time
#    3. SHUTDOWN   — save_session() is called automatically on SIGTERM/SIGINT
#
#  On a new session:
#    - Run this cell → it finds the latest backup and restores everything
#
#  Drive layout:
#    MyDrive/
#      agent_workspace/
#        latest.zip          ← always the most recent full backup
#        backups/
#          2024-01-15_14-30.zip
#          2024-01-15_16-00.zip
#          ...               ← rolling window of last N_BACKUPS snapshots
# ══════════════════════════════════════════════════════════════════════════════


class SessionPersistence:
    """
    Manages save/restore of the entire agent workspace to Google Drive.
    Works in Colab (Drive-backed) and degrades gracefully locally.
    """

    # ── Configuration ─────────────────────────────────────────────────
    DRIVE_ROOT     = Path(os.getenv("AGENT_DRIVE_ROOT", "/content/drive/MyDrive/agent_workspace"))
    LATEST_ZIP     = DRIVE_ROOT / "latest.zip"
    BACKUPS_DIR    = DRIVE_ROOT / "backups"
    N_BACKUPS      = 10          # rolling window kept in backups/
    AUTOSAVE_EVERY = 15 * 60     # seconds between auto-saves (15 min)

    # What to include in the ZIP (relative to Config.BASE_DIR)
    INCLUDE_DIRS = ("core", "memory", "skills", "work", "files", "subtools")
    INCLUDE_ROOT_FILES = ("agent_stats.jsonl",)  # files in workspace root

    def __init__(self) -> None:
        self._autosave_thread: threading.Thread | None = None
        self._stop_autosave:   threading.Event         = threading.Event()
        self._drive_available: bool                    = self._check_drive()

    # ── Drive availability ────────────────────────────────────────────

    @staticmethod
    def _check_drive() -> bool:
        """Return True if Google Drive is mounted and writable."""
        drive = Path("/content/drive/MyDrive")
        return drive.exists() and drive.is_dir()

    @staticmethod
    def mount_drive() -> bool:
        """Mount Google Drive if running in Colab. Returns True on success."""
        if not IS_COLAB:
            print("ℹ  Not in Colab — Drive mount skipped.")
            return False
        if Path("/content/drive/MyDrive").exists():
            print("✓ Drive already mounted.")
            return True
        try:
            from google.colab import drive as _drive
            _drive.mount("/content/drive")
            print("✓ Google Drive mounted.")
            return True
        except Exception as e:
            print(f"✗ Drive mount failed: {e}")
            return False

    # ── Save ──────────────────────────────────────────────────────────

    def save(self, label: str = "manual") -> Path | None:
        """
        Bundle the workspace into a ZIP and write it to Drive.
        Also rotates the backups/ rolling window.
        Returns the path of the written ZIP, or None on failure.
        """
        if not self._drive_available:
            self._drive_available = self._check_drive()
        if not self._drive_available:
            print("✗ Drive not available. Mount Drive first with: persistence.mount_drive()")
            return None

        try:
            self.DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
            self.BACKUPS_DIR.mkdir(parents=True, exist_ok=True)

            # Build zip in a temp file first, then atomically move it
            ts      = datetime.now().strftime("%Y-%m-%d_%H-%M")
            tmp_zip = self.DRIVE_ROOT / f"_tmp_{ts}.zip"
            self._build_zip(tmp_zip)

            # Rotate: copy current latest → backups/ before overwriting
            if self.LATEST_ZIP.exists():
                dest = self.BACKUPS_DIR / f"{ts}.zip"
                shutil.copy2(self.LATEST_ZIP, dest)
                self._prune_backups()

            # Promote tmp → latest
            tmp_zip.replace(self.LATEST_ZIP)

            size_kb = self.LATEST_ZIP.stat().st_size // 1024
            print(f"💾 Saved [{label}] → {self.LATEST_ZIP}  ({size_kb} KB)  {ts}")
            logger.info(f"Session saved to Drive [{label}]", size_kb=str(size_kb))
            return self.LATEST_ZIP

        except Exception as e:
            print(f"✗ Save failed: {e}")
            logger.error(f"Session save failed: {e}")
            return None

    def _build_zip(self, dest: Path) -> None:
        """Write all workspace content into *dest* as a ZIP archive."""
        base = Config.BASE_DIR
        with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            # Include configured subdirectories
            for dir_name in self.INCLUDE_DIRS:
                src_dir = base / dir_name
                if not src_dir.exists():
                    continue
                for file_path in src_dir.rglob("*"):
                    if file_path.is_file():
                        zf.write(file_path, file_path.relative_to(base))

            # Include specific root-level files
            for fname in self.INCLUDE_ROOT_FILES:
                src = base / fname
                if src.exists():
                    zf.write(src, src.relative_to(base))

    def _prune_backups(self) -> None:
        """Keep only the N_BACKUPS most recent files in backups/."""
        zips = sorted(self.BACKUPS_DIR.glob("*.zip"), key=lambda p: p.stat().st_mtime)
        for old in zips[: max(0, len(zips) - self.N_BACKUPS)]:
            old.unlink(missing_ok=True)

    # ── Restore ───────────────────────────────────────────────────────

    def restore(self, from_backup: str | None = None) -> bool:
        """
        Restore the workspace from Drive.

        Args:
            from_backup: filename inside backups/ to restore (e.g. '2024-01-15_14-30.zip').
                         Defaults to latest.zip if not given.

        Returns True on success, False on failure.
        """
        if not self._check_drive():
            print("✗ Drive not mounted. Call persistence.mount_drive() first.")
            return False

        if from_backup:
            src = self.BACKUPS_DIR / from_backup
        else:
            src = self.LATEST_ZIP

        if not src.exists():
            print(f"✗ Backup not found: {src}")
            print("  Nothing to restore — this may be a fresh workspace.")
            return False

        try:
            base = Config.BASE_DIR
            base.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(src, "r") as zf:
                zf.extractall(base)

            size_kb = src.stat().st_size // 1024
            ts      = datetime.fromtimestamp(src.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
            print(f"✅ Restored from {src.name}  ({size_kb} KB)  saved at {ts}")
            logger.info("Session restored from Drive", source=str(src))
            return True

        except Exception as e:
            print(f"✗ Restore failed: {e}")
            logger.error(f"Session restore failed: {e}")
            return False

    def list_backups(self) -> list[dict[str, Any]]:
        """Print and return a list of all available backups."""
        if not self._check_drive():
            print("Drive not mounted.")
            return []

        entries: list[dict[str, Any]] = []

        # latest.zip first
        if self.LATEST_ZIP.exists():
            st = self.LATEST_ZIP.stat()
            entries.append({
                "name":    "latest.zip  ← most recent",
                "size_kb": st.st_size // 1024,
                "saved":   datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M"),
            })

        # Rolling backups newest first
        for p in sorted(
            self.BACKUPS_DIR.glob("*.zip") if self.BACKUPS_DIR.exists() else [],
            key=lambda f: f.stat().st_mtime,
            reverse=True,
        ):
            st = p.stat()
            entries.append({
                "name":    p.name,
                "size_kb": st.st_size // 1024,
                "saved":   datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M"),
            })

        if not entries:
            print("No backups found on Drive yet.")
            return []

        print(f"\n{'NAME':<40} {'SIZE':>8}  SAVED AT")
        print("─" * 65)
        for e in entries:
            print(f"  {e['name']:<38} {e['size_kb']:>6} KB  {e['saved']}")
        print()
        return entries

    # ── Auto-save ─────────────────────────────────────────────────────

    def start_autosave(self, interval: int | None = None) -> None:
        """Start background auto-save thread."""
        if self._autosave_thread and self._autosave_thread.is_alive():
            print("ℹ  Auto-save already running.")
            return
        interval = interval or self.AUTOSAVE_EVERY
        self._stop_autosave.clear()
        self._autosave_thread = threading.Thread(
            target=self._autosave_loop,
            args=(interval,),
            daemon=True,
            name="autosave",
        )
        self._autosave_thread.start()
        mins = interval // 60
        print(f"⏱  Auto-save started — saving every {mins} min to Drive.")
        logger.info(f"Auto-save started (interval={interval}s)")

    def stop_autosave(self) -> None:
        """Stop the background auto-save thread."""
        self._stop_autosave.set()
        if self._autosave_thread:
            self._autosave_thread.join(timeout=5)
        print("⏹  Auto-save stopped.")

    def _autosave_loop(self, interval: int) -> None:
        while not self._stop_autosave.wait(timeout=interval):
            self.save(label="auto")

    # ── Shutdown hook ─────────────────────────────────────────────────

    def register_shutdown_hook(self) -> None:
        """
        Register a save on SIGTERM/SIGINT so the workspace is preserved even
        if the Colab runtime is interrupted rather than gracefully closed.
        Adds to (not replaces) Agent's existing signal handlers.
        """
        import signal as _signal

        def _hook(sig, frame):
            print("\n💾 Saving workspace before shutdown…")
            self.save(label="shutdown")

        for sig in (_signal.SIGTERM, _signal.SIGINT):
            try:
                prev = _signal.getsignal(sig)
                def _chained(s, f, _prev=prev, _hook=_hook):
                    _hook(s, f)
                    if callable(_prev):
                        _prev(s, f)
                _signal.signal(sig, _chained)
            except (OSError, ValueError):
                pass

        print("✓ Shutdown hook registered — workspace saves on interrupt/termination.")


# ══════════════════════════════════════════════════════════════════════════════
#  Session start — auto-restore on every new Colab session
# ══════════════════════════════════════════════════════════════════════════════

def session_start(
    autosave:          bool      = True,
    autosave_interval: int | None = None,
    start_agent:       bool      = False,
) -> tuple["SessionPersistence", "Agent | None"]:
    """
    One-call setup for a new Colab session:
      1. Mounts Drive
      2. Restores previous workspace from latest.zip
      3. Starts auto-save
      4. Registers shutdown hook
      5. Optionally initialises and returns a ready Agent

    Usage (first cell after imports):
        persistence, agent = session_start(start_agent=True)

    Or if you want to initialise the agent yourself:
        persistence, _ = session_start()
        agent = Agent()
    """
    print("=" * 60)
    print("  SESSION START")
    print("=" * 60)

    p = SessionPersistence()

    # 1. Mount Drive
    mounted = p.mount_drive()

    # 2. Restore workspace
    if mounted:
        p.restore()
    else:
        print("ℹ  Skipping restore — Drive not mounted.")

    # 3. Auto-save
    if autosave and mounted:
        p.start_autosave(interval=autosave_interval)

    # 4. Shutdown hook
    p.register_shutdown_hook()

    # 5. Optional agent init
    agent = None
    if start_agent:
        try:
            setup_api_key()
            agent = Agent(stream=True)
            agent.start_background_heartbeat()
            print(f"✅ Agent ready · {Config.MODEL}")
        except Exception as e:
            import traceback
            print(f"✗ Agent init failed: {e}")
            traceback.print_exc()

    print("=" * 60)
    return p, agent


# ══════════════════════════════════════════════════════════════════════════════
#  Module-level instance + convenience shortcuts
# ══════════════════════════════════════════════════════════════════════════════

persistence = SessionPersistence()

def save_session()      -> None: persistence.save(label="manual")
def restore_session()   -> None: persistence.restore()
def list_backups()      -> None: persistence.list_backups()


# ── Auto-run on cell execution ────────────────────────────────────────────────
print("✓ SessionPersistence ready")
print()
print("  Quick-start for a NEW session:")
print("    persistence, agent = session_start(start_agent=True)")
print()
print("  Or step by step:")
print("    persistence.mount_drive()")
print("    persistence.restore()         # load previous workspace")
print("    persistence.start_autosave()  # save every 15 min")
print("    save_session()                # save right now")
print("    list_backups()                # see all available backups")
print("    persistence.restore('2024-01-15_14-30.zip')  # restore specific backup")

✓ SessionPersistence ready

  Quick-start for a NEW session:
    persistence, agent = session_start(start_agent=True)

  Or step by step:
    persistence.mount_drive()
    persistence.restore()         # load previous workspace
    persistence.start_autosave()  # save every 15 min
    save_session()                # save right now
    list_backups()                # see all available backups
    persistence.restore('2024-01-15_14-30.zip')  # restore specific backup


In [41]:
#save_session()
persistence.restore()

✅ Restored from latest.zip  (14850 KB)  saved at 2026-03-24 00:07


True

## CELL 18: Automation

In [42]:
# @title
from __future__ import annotations

import json
import queue
import threading
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Callable

# ══════════════════════════════════════════════════════════════════════════════
#  CELL 18 — Full Automation
#
#  What this adds on top of everything already running:
#
#  ┌─────────────────────────────────────────────────────────────────────┐
#  │  ALREADY AUTOMATED (cells 11/16/17)                                 │
#  │    ✓ Heartbeat health checks (every 30 min)                         │
#  │    ✓ Core file evolution after every turn                           │
#  │    ✓ Auto-save to Drive (every 15 min)                              │
#  │    ✓ Shutdown hook saves on crash                                   │
#  ├─────────────────────────────────────────────────────────────────────┤
#  │  NEWLY AUTOMATED (this cell)                                        │
#  │    ✓ Task queue   — submit tasks; agent works through them headlessly│
#  │    ✓ Express lane — small tasks run in parallel, skip the queue     │
#  │    ✓ Scheduler    — run tasks at fixed times or intervals           │
#  │    ✓ Need poller  — auto-resolves PENDING needs without user input  │
#  │    ✓ Watchdog     — restarts agent if it crashes                    │
#  │    ✓ One-call     — autorun() wires everything up in a single call  │
#  └─────────────────────────────────────────────────────────────────────┘
# ══════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────────────────────────────────────
#  1. TaskQueue — submit tasks anytime; worker thread drains them
# ─────────────────────────────────────────────────────────────────────────────

class TaskQueue:
    """
    Thread-safe FIFO task queue.  Submit tasks from any cell; the worker
    thread processes them one at a time without blocking the notebook.

    Usage:
        tq.submit("Write a readme for /content/agent/files/project/")
        tq.submit("Analyse the logs and summarise errors")
        tq.status()   # see what's pending / done

    For quick tasks that should not wait in line, use the ExpressLane:
        hub.express.run("What time is it?")
        hub.express.run("How many files are in /content/agent/files/?")
    """

    def __init__(self, agent: "Agent") -> None:
        self._agent         = agent
        self._q:    queue.Queue          = queue.Queue()
        self._done: list[dict[str, Any]] = []
        self._lock          = threading.Lock()
        self._stop          = threading.Event()
        self._thread:       threading.Thread | None = None
        self._current:      str | None = None
        self._paused:       bool       = False
        self._cancel_current: bool     = False

    # ── Public API ────────────────────────────────────────────────────

    def submit(self, task: str, priority: bool = False) -> None:
        """
        Add *task* to the queue.
        Set ``priority=True`` to jump ahead of existing items.
        """
        item = {"task": task, "submitted": datetime.now().isoformat(), "status": "pending"}
        # FIX: Rebuild under lock — original had a TOCTOU race where the worker
        # could dequeue between the empty-check and get_nowait() calls.
        with self._lock:
            if priority:
                tmp: list[dict] = [item]
                while True:
                    try:
                        tmp.append(self._q.get_nowait())
                    except queue.Empty:
                        break
                for t in tmp:
                    self._q.put(t)
            else:
                self._q.put(item)
        print(f"📥 Queued [{self._q.qsize()} pending]: {task[:80]}")

    def submit_many(self, tasks: list[str]) -> None:
        """Submit multiple tasks at once."""
        for t in tasks:
            self.submit(t)

    def status(self) -> None:
        """Print queue state: pending, current, and last 5 completed."""
        pending = list(self._q.queue)
        print(f"\n{'─'*55}")
        print(f"  TASK QUEUE  —  pending: {len(pending)}  done: {len(self._done)}")
        print(f"{'─'*55}")
        if self._current:
            print(f"  ▶ RUNNING : {self._current[:70]}")
        for i, item in enumerate(pending):
            print(f"  {i+1:>2}. {item['task'][:70]}")
        if self._done:
            print(f"\n  Last {min(5, len(self._done))} completed:")
            for item in self._done[-5:]:
                ts = item.get("finished", "?")[:16]
                ok = "✓" if not item.get("error") else "✗"
                print(f"    {ok} [{ts}]  {item['task'][:60]}")
        print()

    def clear(self) -> None:
        """Discard all pending (not running) tasks."""
        with self._lock:
            while not self._q.empty():
                try:
                    self._q.get_nowait()
                except queue.Empty:
                    break
        print("🗑  Queue cleared.")

    def cancel_current(self) -> str:
        """
        Signal the currently-running task to be interrupted at the next
        Agent.run() iteration boundary. Returns a status string.
        """
        if self._current is None:
            return "No task currently running."
        self._cancel_current = True
        return f"⏹ Cancellation requested for: {self._current[:80]}"

    # ── Worker ────────────────────────────────────────────────────────

    def start(self) -> None:
        if self._thread and self._thread.is_alive():
            print("ℹ  Task queue already running.")
            return
        self._stop.clear()
        self._thread = threading.Thread(
            target=self._worker, daemon=True, name="task-queue"
        )
        self._thread.start()
        print("▶  Task queue worker started.")

    def stop(self) -> None:
        self._stop.set()
        print("⏹  Task queue worker stopping (finishes current task first).")

    def _worker(self) -> None:
        while not self._stop.is_set():
            if self._paused:
                time.sleep(1)
                continue
            try:
                item = self._q.get(timeout=2)
            except queue.Empty:
                continue

            self._current         = item["task"]
            self._cancel_current  = False          # FIX: reset flag for each new task
            item["started"]       = datetime.now().isoformat()
            item["status"]        = "running"
            try:
                print(f"\n🤖 [QUEUE] Starting: {item['task'][:80]}")
                result = self._agent.run(item["task"])

                # FIX: check cancel flag *after* run returns — agent.run() is
                # synchronous; the flag is the signal it saw a cancellation request.
                if self._cancel_current:
                    item["status"] = "cancelled"
                    item["result"] = result[:500] if result else ""
                    print(f"⏹  [QUEUE] Cancelled: {item['task'][:60]}")
                else:
                    item["result"] = result[:500]
                    item["status"] = "done"
                    print(f"✓  [QUEUE] Done:     {item['task'][:60]}")
            except Exception as e:
                item["error"]  = str(e)
                item["status"] = "error"
                print(f"✗  [QUEUE] Error:    {item['task'][:60]}  →  {e}")
            finally:
                item["finished"] = datetime.now().isoformat()
                with self._lock:
                    self._done.append(item)
                    if len(self._done) > 100:
                        self._done = self._done[-100:]
                self._current        = None
                self._cancel_current = False
                self._q.task_done()


# ─────────────────────────────────────────────────────────────────────────────
#  2. ExpressLane — run small tasks in parallel without touching the queue
# ─────────────────────────────────────────────────────────────────────────────

class ExpressLane:
    """
    Run quick, stateless tasks in parallel — completely bypassing the main
    TaskQueue. Results are returned immediately (non-blocking) or you can
    wait for them.

    The express lane uses direct single-turn API calls (no full agent loop,
    no conversation history) so it never conflicts with the queue worker,
    even if both are running simultaneously.

    Design rationale:
      • TaskQueue uses agent.run() — full multi-step agent loop, one at a time.
      • ExpressLane uses a one-shot API call — lightweight, concurrent, no state.
      • The two never share the same agent state, so there is no race condition.

    Usage:
        # Fire-and-forget (callback receives the result when ready):
        hub.express.run("How many .py files are in /content/agent/files/?",
                        callback=lambda r: print("Express result:", r))

        # Block until done (for notebook cells):
        result = hub.express.run_sync("What is today's date?")
        print(result)

        # Run several in parallel and collect all results:
        results = hub.express.run_all([
            "Summarise the last 5 log lines",
            "Count files in /content/agent/files/",
            "What is the current MEMORY.md word count?",
        ])
    """

    # Maximum parallel express workers (keep small — these are *quick* tasks)
    MAX_WORKERS = 3
    # Token budget for express responses — keep them brief
    MAX_TOKENS  = 512

    def __init__(self, agent: "Agent") -> None:
        self._agent   = agent
        self._results: dict[str, dict[str, Any]] = {}
        self._lock    = threading.Lock()
        self._pool    = None   # created lazily in start()

    def start(self) -> None:
        import concurrent.futures
        if self._pool is None:
            self._pool = concurrent.futures.ThreadPoolExecutor(
                max_workers=self.MAX_WORKERS,
                thread_name_prefix="express",
            )
        print(f"⚡  Express lane started ({self.MAX_WORKERS} parallel workers).")

    def stop(self) -> None:
        if self._pool:
            self._pool.shutdown(wait=False)
            self._pool = None
        print("⏹  Express lane stopped.")

    def _call(self, task: str) -> str:
        """
        Single-turn API call — no agent loop, no history, no tool use.
        Uses the agent's underlying client directly so it shares the same
        model config and counter without touching agent conversation state.
        """
        try:
            resp = self._agent.client.chat.completions.create(
                model      = Config.FAST_MODEL,
                max_tokens = self.MAX_TOKENS,
                messages   = [
                    {
                        "role":    "system",
                        "content": (
                            "You are a quick-answer assistant. "
                            "Answer concisely in 1-3 sentences. "
                            "No preamble, no markdown unless essential."
                        ),
                    },
                    {"role": "user", "content": task},
                ],
            )
            return resp.choices[0].message.content or ""
        except Exception as e:
            return f"[express error] {e}"

    def _run_tracked(self, task_id: str, task: str, callback: Callable | None) -> str:
        """Internal wrapper that tracks result and fires callback."""
        with self._lock:
            self._results[task_id] = {
                "task":      task,
                "status":    "running",
                "started":   datetime.now().isoformat(),
                "result":    None,
            }
        result = self._call(task)
        with self._lock:
            self._results[task_id].update({
                "status":   "done",
                "result":   result,
                "finished": datetime.now().isoformat(),
            })
        if callback:
            try:
                callback(result)
            except Exception as e:
                print(f"⚡  Express callback error: {e}")
        return result

    def run(
        self,
        task:     str,
        callback: Callable[[str], None] | None = None,
    ) -> "concurrent.futures.Future[str]":
        """
        Submit *task* to the express lane. Returns a Future immediately.
        Optionally fires *callback(result)* when done.

        The main queue keeps running — this call does NOT block it.

        Args:
            task:     what to ask (keep it short and self-contained)
            callback: optional function called with the result string when ready

        Returns:
            Future — call .result() to block until done, or ignore for fire-and-forget.

        Example:
            f = hub.express.run("List top 3 files by size in /content/agent/files/")
            # ... do other things ...
            print(f.result())   # blocks here until answer arrives
        """
        if self._pool is None:
            self.start()
        task_id = f"{time.monotonic():.6f}"
        print(f"⚡  [EXPRESS] → {task[:80]}")
        return self._pool.submit(self._run_tracked, task_id, task, callback)

    def run_sync(self, task: str) -> str:
        """
        Submit *task* and block until the result arrives.
        Useful for notebook cells where you want the answer immediately
        without disturbing the queue.

        Example:
            answer = hub.express.run_sync("How many pending tasks are in Need.md?")
            print(answer)
        """
        future = self.run(task)
        result = future.result()
        print(f"⚡  [EXPRESS] ✓ {result[:120]}")
        return result

    def run_all(self, tasks: list[str]) -> list[str]:
        """
        Submit all *tasks* in parallel and return results in the same order
        once ALL have completed. Blocks until every task finishes.

        Example:
            results = hub.express.run_all([
                "Summarise the last 5 log lines",
                "Count .py files in the workspace",
                "What is the agent's uptime?",
            ])
            for t, r in zip(tasks, results):
                print(f"Q: {t}\nA: {r}\n")
        """
        import concurrent.futures
        if self._pool is None:
            self.start()
        futures = [self.run(t) for t in tasks]
        return [f.result() for f in futures]

    def status(self) -> None:
        """Print recent express results."""
        with self._lock:
            results = dict(self._results)
        if not results:
            print("  No express results yet.")
            return
        print(f"\n{'─'*55}")
        print(f"  EXPRESS LANE  —  {len(results)} result(s)")
        print(f"{'─'*55}")
        for r in sorted(results.values(), key=lambda x: x.get("started", ""))[-10:]:
            ts  = r.get("finished", r.get("started", "?"))[:16]
            ok  = "✓" if r["status"] == "done" else "▶"
            res = (r.get("result") or "")[:80]
            print(f"  {ok} [{ts}]  {r['task'][:40]}")
            if res:
                print(f"       → {res}")
        print()


# ─────────────────────────────────────────────────────────────────────────────
#  3. Scheduler — run tasks at fixed times or intervals
# ─────────────────────────────────────────────────────────────────────────────

class Scheduler:
    """
    Lightweight cron-style scheduler.  All scheduled tasks are fed into
    the TaskQueue so they're sequenced correctly.

    Usage:
        scheduler.every(30, "Summarise today's log and update MEMORY.md")
        scheduler.every(60, "Check for new files in /content/agent/files/inbox/")
        scheduler.at("09:00", "Generate daily status report")
        scheduler.cancel("Summarise today's log...")
    """

    def __init__(self, task_queue: TaskQueue) -> None:
        self._tq      = task_queue
        self._jobs:   list[dict[str, Any]] = []
        self._stop    = threading.Event()
        self._thread: threading.Thread | None = None
        self._lock    = threading.Lock()

    # ── Public API ────────────────────────────────────────────────────

    def every(self, minutes: int, task: str) -> str:
        """Run *task* every *minutes* minutes. Returns job id."""
        job_id = f"every_{minutes}m_{task[:20]}"
        job    = {
            "id":        job_id,
            "type":      "interval",
            "minutes":   minutes,
            "task":      task,
            "next_run":  datetime.now() + timedelta(minutes=minutes),
            "run_count": 0,
        }
        with self._lock:
            self._jobs = [j for j in self._jobs if j["id"] != job_id]
            self._jobs.append(job)
        print(f"⏱  Scheduled every {minutes} min: {task[:60]}")
        return job_id

    def at(self, time_str: str, task: str, daily: bool = True) -> str:
        """
        Run *task* at a specific time (HH:MM, 24-hour).
        Set ``daily=True`` (default) to repeat every day.
        """
        job_id  = f"at_{time_str}_{task[:20]}"
        h, m    = map(int, time_str.split(":"))
        now     = datetime.now()
        next_dt = now.replace(hour=h, minute=m, second=0, microsecond=0)
        if next_dt <= now:
            next_dt += timedelta(days=1)
        job = {
            "id":        job_id,
            "type":      "daily" if daily else "once",
            "time_str":  time_str,
            "task":      task,
            "next_run":  next_dt,
            "run_count": 0,
        }
        with self._lock:
            self._jobs = [j for j in self._jobs if j["id"] != job_id]
            self._jobs.append(job)
        print(f"🕐  Scheduled at {time_str}: {task[:60]}")
        return job_id

    def cancel(self, job_id_or_task: str) -> None:
        """Cancel a job by its id or task substring."""
        with self._lock:
            before     = len(self._jobs)
            self._jobs = [
                j for j in self._jobs
                if job_id_or_task not in j["id"] and job_id_or_task not in j["task"]
            ]
            removed = before - len(self._jobs)
        print(f"❌  Cancelled {removed} job(s) matching '{job_id_or_task[:40]}'")

    def status(self) -> None:
        """Print all scheduled jobs and their next run time."""
        with self._lock:
            jobs = list(self._jobs)
        if not jobs:
            print("  No scheduled jobs.")
            return
        print(f"\n{'─'*60}")
        print(f"  SCHEDULER  —  {len(jobs)} job(s)")
        print(f"{'─'*60}")
        for j in sorted(jobs, key=lambda x: x["next_run"]):
            delta = j["next_run"] - datetime.now()
            mins  = int(delta.total_seconds() / 60)
            print(f"  [{j['type']:<8}] in {mins:>4} min  —  {j['task'][:55]}")
        print()

    # ── Runner ────────────────────────────────────────────────────────

    def start(self) -> None:
        if self._thread and self._thread.is_alive():
            print("ℹ  Scheduler already running.")
            return
        self._stop.clear()
        self._thread = threading.Thread(
            target=self._loop, daemon=True, name="scheduler"
        )
        self._thread.start()
        print("▶  Scheduler started.")

    def stop(self) -> None:
        self._stop.set()
        print("⏹  Scheduler stopped.")

    def _loop(self) -> None:
        while not self._stop.wait(timeout=30):
            now = datetime.now()
            # FIX: Take a snapshot of due jobs under the lock, then mutate
            # outside it. Original iterated self._jobs and mutated job dicts
            # (next_run, run_count) while potentially racing with cancel().
            with self._lock:
                due = [j for j in self._jobs if j["next_run"] <= now]

            for job in due:
                self._tq.submit(job["task"])
                # FIX: Mutate job fields under lock to avoid race with cancel()
                with self._lock:
                    job["run_count"] += 1
                    if job["type"] == "interval":
                        job["next_run"] = now + timedelta(minutes=job["minutes"])
                    elif job["type"] == "daily":
                        job["next_run"] = now + timedelta(days=1)
                    else:   # "once" — remove after firing
                        self._jobs = [j for j in self._jobs if j["id"] != job["id"]]
                logger.info(f"Scheduler fired: {job['task'][:60]}")


# ─────────────────────────────────────────────────────────────────────────────
#  4. NeedPoller — auto-resolve PENDING needs without user input
# ─────────────────────────────────────────────────────────────────────────────

class NeedPoller:
    """
    Polls Need.md every N minutes and feeds any PENDING request back to
    the agent as a task so it can try to resolve it autonomously.

    Usage:
        need_poller.start()
        need_poller.stop()
    """

    def __init__(self, task_queue: TaskQueue, interval_minutes: int = 10) -> None:
        self._tq       = task_queue
        self._interval = interval_minutes * 60
        self._stop     = threading.Event()
        self._thread:  threading.Thread | None = None
        # FIX: _seen must expire — if a need is re-posted after resolution it
        # should be picked up again. Store (key, first_seen_ts) and forget after
        # 24 hours so re-posted needs aren't silently dropped forever.
        self._seen:    dict[str, datetime] = {}
        self._seen_lock = threading.Lock()

    def start(self) -> None:
        if self._thread and self._thread.is_alive():
            print("ℹ  Need poller already running.")
            return
        self._stop.clear()
        self._thread = threading.Thread(
            target=self._loop, daemon=True, name="need-poller"
        )
        self._thread.start()
        print(f"👀  Need poller started (every {self._interval // 60} min).")

    def stop(self) -> None:
        self._stop.set()
        print("⏹  Need poller stopped.")

    def _loop(self) -> None:
        while not self._stop.wait(timeout=self._interval):
            try:
                self._evict_old_seen()
                needs = check_needs()
                for need in needs:
                    key = f"{need.get('priority','')}{need.get('needs','')}"
                    with self._seen_lock:
                        if key in self._seen:
                            continue
                        self._seen[key] = datetime.now()
                    task = (
                        f"A pending need was found in Need.md. "
                        f"Priority: {need.get('priority','?')}. "
                        f"Need: {need.get('needs','?')}. "
                        f"Reason: {need.get('reason','?')}. "
                        f"Blocking: {need.get('blocking','?')}. "
                        f"Please attempt to resolve this autonomously using your tools. "
                        f"If you cannot resolve it without human input, explain clearly "
                        f"what is needed and why."
                    )
                    self._tq.submit(task, priority=True)
                    print(f"🔔  Need auto-queued: {need.get('needs','?')[:60]}")
            except Exception as e:
                logger.warning(f"NeedPoller error: {e}")

    def _evict_old_seen(self) -> None:
        """Remove seen-set entries older than 24 h so re-posted needs are retried."""
        cutoff = datetime.now() - timedelta(hours=24)
        with self._seen_lock:
            self._seen = {k: v for k, v in self._seen.items() if v > cutoff}


# ─────────────────────────────────────────────────────────────────────────────
#  5. Watchdog — restarts the agent if it crashes
# ─────────────────────────────────────────────────────────────────────────────

class Watchdog:
    """
    Monitors the agent and its supporting threads. If any critical thread
    dies unexpectedly it logs the failure and attempts to restart.

    Threads monitored:
        - task_queue worker
        - scheduler
        - need_poller
        - express lane pool (checked for replacement, not restart)
        - heartbeat (agent._heartbeat_thread)
    """

    CHECK_INTERVAL = 60   # seconds

    def __init__(
        self,
        agent:       "Agent",
        task_queue:  TaskQueue,
        scheduler:   Scheduler,
        need_poller: NeedPoller,
        express:     ExpressLane,
    ) -> None:
        self._agent   = agent
        self._tq      = task_queue
        self._sched   = scheduler
        self._poller  = need_poller
        self._express = express
        self._stop    = threading.Event()
        self._thread: threading.Thread | None = None

    def start(self) -> None:
        if self._thread and self._thread.is_alive():
            print("ℹ  Watchdog already running.")
            return
        self._stop.clear()
        self._thread = threading.Thread(
            target=self._loop, daemon=True, name="watchdog"
        )
        self._thread.start()
        print(f"🐕  Watchdog started (checks every {self.CHECK_INTERVAL}s).")

    def stop(self) -> None:
        self._stop.set()
        print("⏹  Watchdog stopped.")

    def _loop(self) -> None:
        while not self._stop.wait(timeout=self.CHECK_INTERVAL):
            self._check_and_restart("task_queue",  self._tq._thread,     self._tq.start)
            self._check_and_restart("scheduler",   self._sched._thread,  self._sched.start)
            self._check_and_restart("need_poller", self._poller._thread, self._poller.start)
            # Express lane uses a thread pool — check if it was shut down and recreate
            if self._express._pool is None and not self._stop.is_set():
                logger.warning("Watchdog: express lane pool missing — restarting")
                print("🐕  Watchdog: restarting express lane…")
                self._express.start()
            # Heartbeat lives on the agent
            hb = getattr(self._agent, "_heartbeat_thread", None)
            if hb and not hb.is_alive() and not self._stop.is_set():
                logger.warning("Watchdog: heartbeat died — restarting")
                print("🐕  Watchdog: restarting heartbeat…")
                self._agent.start_background_heartbeat()

    @staticmethod
    def _check_and_restart(
        name:    str,
        thread:  threading.Thread | None,
        restart: Callable[[], None],
    ) -> None:
        if thread is not None and not thread.is_alive():
            logger.warning(f"Watchdog: {name} thread died — restarting")
            print(f"🐕  Watchdog: restarting {name}…")
            try:
                restart()
            except Exception as e:
                logger.error(f"Watchdog: failed to restart {name}: {e}")


# ─────────────────────────────────────────────────────────────────────────────
#  6. AutomationHub — single object holding every automation component
# ─────────────────────────────────────────────────────────────────────────────

class AutomationHub:
    """
    Holds and exposes every automation component.
    Returned by ``autorun()`` so you have one object to interact with.

    Attributes:
        agent        — the Agent instance
        queue        — TaskQueue  (long-running sequential tasks)
        express      — ExpressLane  (quick parallel tasks, skip the queue)
        scheduler    — Scheduler
        need_poller  — NeedPoller
        watchdog     — Watchdog
        persistence  — SessionPersistence

    Quick-reference:
        hub.queue.submit("do something heavy")
        hub.express.run("quick question?")            # non-blocking, parallel
        hub.express.run_sync("quick question?")       # blocks until answered
        hub.express.run_all(["q1", "q2", "q3"])       # all parallel, wait for all
        hub.scheduler.every(60, "summarise today's work")
        hub.scheduler.at("18:00", "generate end-of-day report")
        hub.status()
        hub.stop_all()
    """

    def __init__(
        self,
        agent:       "Agent",
        persistence: "SessionPersistence",
    ) -> None:
        self.agent       = agent
        self.persistence = persistence
        self.queue       = TaskQueue(agent)
        self.express     = ExpressLane(agent)
        self.scheduler   = Scheduler(self.queue)
        self.need_poller = NeedPoller(self.queue)
        self.watchdog    = Watchdog(
            agent, self.queue, self.scheduler, self.need_poller, self.express
        )

    def start_all(self) -> None:
        self.queue.start()
        self.express.start()
        self.scheduler.start()
        self.need_poller.start()
        self.watchdog.start()
        print("\n✅ All automation components running.")

    def stop_all(self) -> None:
        self.watchdog.stop()
        self.need_poller.stop()
        self.scheduler.stop()
        self.queue.stop()
        self.express.stop()
        self.persistence.stop_autosave()
        self.agent.stop_background_heartbeat()
        print("⏹  All automation stopped.")

    def status(self) -> None:
        """Print a full status dashboard."""
        print(f"\n{'═'*60}")
        print(f"  AUTOMATION HUB STATUS  —  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'═'*60}")

        def _t(thread):
            return "\033[92m● running\033[0m" if thread and thread.is_alive() \
                   else "\033[91m● stopped\033[0m"

        express_alive = self.express._pool is not None
        print(f"  heartbeat   {_t(getattr(self.agent, '_heartbeat_thread', None))}")
        print(f"  task_queue  {_t(self.queue._thread)}")
        print(f"  express     {'  \033[92m● running\033[0m' if express_alive else '  \033[91m● stopped\033[0m'}  "
              f"({ExpressLane.MAX_WORKERS} workers)")
        print(f"  scheduler   {_t(self.scheduler._thread)}")
        print(f"  need_poller {_t(self.need_poller._thread)}")
        print(f"  watchdog    {_t(self.watchdog._thread)}")
        print(f"  autosave    {_t(self.persistence._autosave_thread)}")
        print()
        self.queue.status()
        self.express.status()
        self.scheduler.status()

    # ── Shorthands ────────────────────────────────────────────────────

    def submit(self, task: str) -> None:
        """Shorthand for hub.queue.submit(task)."""
        self.queue.submit(task)

    def every(self, minutes: int, task: str) -> str:
        """Shorthand for hub.scheduler.every(minutes, task)."""
        return self.scheduler.every(minutes, task)

    def at(self, time_str: str, task: str) -> str:
        """Shorthand for hub.scheduler.at(time_str, task)."""
        return self.scheduler.at(time_str, task)


# ─────────────────────────────────────────────────────────────────────────────
#  7. autorun() — one call to rule them all
# ─────────────────────────────────────────────────────────────────────────────

def autorun(
    tasks:             list[str] | None = None,
    schedule:          list[tuple]      | None = None,
    autosave_minutes:  int               = 15,
    need_poll_minutes: int               = 10,
    startup_task:      str               = "Check heartbeat, review Need.md, summarise MEMORY.md",
) -> AutomationHub:
    """
    One-call full automation setup. Does everything:

      1. Mounts Drive + restores previous workspace
      2. Sets up API key
      3. Initialises Agent with heartbeat + core evolution
      4. Starts auto-save to Drive
      5. Starts task queue worker
      6. Starts express lane (parallel quick tasks)
      7. Starts scheduler
      8. Starts need poller
      9. Starts watchdog
      10. Runs startup_task immediately
      11. Submits any tasks you pass in
      12. Registers shutdown hook

    Args:
        tasks:             list of task strings to queue immediately
        schedule:          list of (minutes_or_timestr, task) tuples,
                           e.g. [(30, "summarise logs"), ("09:00", "daily report")]
        autosave_minutes:  how often to save to Drive (default 15)
        need_poll_minutes: how often to poll Need.md (default 10)
        startup_task:      first task run after init

    Returns:
        AutomationHub

    Example:
        hub = autorun(
            tasks=["Analyse files in /content/agent/files/"],
            schedule=[(60, "Summarise today's logs"), ("18:00", "Daily report")],
        )

        # While a long task runs in the queue, ask something quick in parallel:
        hub.express.run("What files were modified in the last hour?",
                        callback=lambda r: print("Answer:", r))
        result = hub.express.run_sync("Summarise the last 3 log entries")
    """
    print("═" * 60)
    print("  🤖  FULL AUTOMATION STARTING")
    print("═" * 60)

    # 1. Drive + restore
    p = SessionPersistence()
    if p.mount_drive():
        p.restore()

    # 2. API key
    if not setup_api_key():
        raise RuntimeError(
            "No NVIDIA API key found.\n"
            "  Colab: Left sidebar → 🔑 → add NVIDIA_API_KEY\n"
            "  Local: export NVIDIA_API_KEY=nvapi-xxxx"
        )

    # 3. Agent
    print("\n  Initialising agent…")
    agent = Agent(stream=False)
    agent.start_background_heartbeat()

    # 4. Auto-save
    p.start_autosave(interval=autosave_minutes * 60)
    p.register_shutdown_hook()

    # 5–9. Build hub and start all components
    hub = AutomationHub(agent=agent, persistence=p)
    hub.need_poller._interval = need_poll_minutes * 60
    hub.start_all()

    # 10. Startup task
    hub.queue.submit(startup_task, priority=True)

    # 11. User-provided tasks
    if tasks:
        for t in tasks:
            hub.queue.submit(t)

    # 12. Scheduled jobs
    if schedule:
        for entry in schedule:
            time_or_mins, task = entry
            if isinstance(time_or_mins, int):
                hub.scheduler.every(time_or_mins, task)
            else:
                hub.scheduler.at(str(time_or_mins), task)

    print("\n" + "═" * 60)
    print("  ✅  FULL AUTOMATION RUNNING")
    print("     hub.status()                         — live dashboard")
    print("     hub.submit('task...')                — queue a heavy task")
    print("     hub.express.run('quick q?')          — parallel, non-blocking")
    print("     hub.express.run_sync('quick q?')     — parallel, wait for answer")
    print("     hub.express.run_all(['q1','q2'])     — many in parallel")
    print("     hub.every(N, 'task...')              — recurring scheduled task")
    print("     hub.at('HH:MM', 'task')              — daily scheduled task")
    print("     hub.stop_all()                       — graceful shutdown")
    print("═" * 60 + "\n")

    return hub


print("✓ Automation cell ready")
print()
print("  Minimal usage:")
print("    hub = autorun()")
print()
print("  Full usage:")
print("    hub = autorun(")
print("        tasks=['Analyse logs', 'Update readme'],")
print("        schedule=[(60, 'Summarise work'), ('18:00', 'Daily report')],")
print("    )")
print()
print("  Express lane (parallel, doesn't touch the queue):")
print("    hub.express.run('quick question?')          # fire-and-forget")
print("    hub.express.run_sync('quick question?')     # wait for answer")
print("    hub.express.run_all(['q1', 'q2', 'q3'])     # all parallel")

✓ Automation cell ready

  Minimal usage:
    hub = autorun()

  Full usage:
    hub = autorun(
        tasks=['Analyse logs', 'Update readme'],
        schedule=[(60, 'Summarise work'), ('18:00', 'Daily report')],
    )

  Express lane (parallel, doesn't touch the queue):
    hub.express.run('quick question?')          # fire-and-forget
    hub.express.run_sync('quick question?')     # wait for answer
    hub.express.run_all(['q1', 'q2', 'q3'])     # all parallel


## CELL 19: Streaming logs

In [43]:
# @title
from __future__ import annotations

import threading
import time
from typing import Any

# ══════════════════════════════════════════════════════════════════════════════
#  CELL 19 — Live Log Streamer
#
#  Three ways to use:
#    1. LogStreamer widget  — rich auto-refreshing ipywidgets panel
#    2. tail_logs()        — simple print-to-cell tail (no widget needed)
#    3. watch_logs()       — blocking live tail like `tail -f` in terminal
# ══════════════════════════════════════════════════════════════════════════════

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    _WIDGETS_OK = True
except ImportError:
    _WIDGETS_OK = False


# ─────────────────────────────────────────────────────────────────────────────
#  Colour map — log level → HTML colour
# ─────────────────────────────────────────────────────────────────────────────

_LEVEL_COLOR: dict[str, str] = {
    "debug":   "#5a6278",
    "info":    "#d4dae8",
    "warning": "#f0a04a",
    "error":   "#f05a4a",
}

_LEVEL_BADGE: dict[str, str] = {
    "debug":   "#1a1d24",
    "info":    "#1a2535",
    "warning": "#2d1f0a",
    "error":   "#2d0a0a",
}

# FIX: Canonical level order used by all three public functions — defined once
# to avoid repeated inline lists and inconsistent ordering.
_LEVEL_ORDER: list[str] = ["debug", "info", "warning", "error"]


def _level_index(level: str) -> int:
    """Return the sort index for *level*, defaulting to 0 (debug) if unknown.

    FIX: All three callers previously used order.index(...) directly, which
    raises ValueError for any unrecognised level string from a log record.
    This helper returns 0 for unknowns instead of crashing.
    """
    try:
        return _LEVEL_ORDER.index(level.lower())
    except ValueError:
        return 0


# ─────────────────────────────────────────────────────────────────────────────
#  HTML rendering helpers
# ─────────────────────────────────────────────────────────────────────────────

_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&display=swap');
.ls-root {
  font-family: 'JetBrains Mono', monospace;
  font-size: 12px;
  background: #0d0f12;
  color: #d4dae8;
  border-radius: 8px;
  overflow: hidden;
  border: 1px solid #252935;
}
.ls-header {
  background: #151820;
  padding: 10px 16px;
  border-bottom: 1px solid #252935;
  display: flex;
  justify-content: space-between;
  align-items: center;
  font-size: 12px;
}
.ls-title  { color: #4af0a0; font-weight: 600; letter-spacing: .5px; }
.ls-meta   { color: #5a6278; font-size: 11px; }
.ls-body   {
  padding: 10px 14px;
  overflow-y: auto;
  max-height: 480px;
  min-height: 200px;
  display: flex;
  flex-direction: column;
  gap: 2px;
}
.ls-row {
  display: flex;
  gap: 10px;
  padding: 3px 6px;
  border-radius: 4px;
  align-items: baseline;
  animation: fadeIn .15s ease;
}
.ls-ts    { color: #3a4060; min-width: 85px; }
.ls-lvl   { min-width: 52px; font-weight: 600; }
.ls-msg   { flex: 1; word-break: break-word; white-space: pre-wrap; }
.ls-extra { color: #5a6278; font-size: 11px; }
@keyframes fadeIn {
  from { opacity: 0; transform: translateY(2px); }
  to   { opacity: 1; transform: translateY(0); }
}
.ls-body::-webkit-scrollbar       { width: 4px; }
.ls-body::-webkit-scrollbar-track { background: transparent; }
.ls-body::-webkit-scrollbar-thumb { background: #252935; border-radius: 4px; }
</style>
"""

# FIX: Shared HTML-escape helper — previously _record_to_html escaped msg and
# ext_str manually but logger.LOG_FILE was inserted raw into the header in
# _render(), creating a potential HTML-injection vector.
def _html_escape(s: str) -> str:
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _record_to_html(r: dict[str, Any]) -> str:
    level   = r.get("level", "info").lower()
    color   = _LEVEL_COLOR.get(level, "#d4dae8")
    bg      = _LEVEL_BADGE.get(level, "#0d0f12")
    ts      = r.get("ts", "")[:19].replace("T", " ")
    msg     = _html_escape((r.get("msg") or "")[:300])
    extra   = {k: v for k, v in r.items() if k not in ("ts", "level", "msg")}
    # FIX: Truncate ext_str values consistently with msg (was previously unlimited)
    ext_str = _html_escape(
        "  " + "  ".join(f"{k}={str(v)[:80]}" for k, v in list(extra.items())[:3])
        if extra else ""
    )
    return (
        f'<div class="ls-row" style="background:{bg};">'
        f'<span class="ls-ts">{ts}</span>'
        f'<span class="ls-lvl" style="color:{color};">{level.upper()}</span>'
        f'<span class="ls-msg" style="color:{color};">{msg}</span>'
        f'<span class="ls-extra">{ext_str}</span>'
        f'</div>'
    )


# ─────────────────────────────────────────────────────────────────────────────
#  LogStreamer — auto-refreshing ipywidgets panel
# ─────────────────────────────────────────────────────────────────────────────

class LogStreamer:
    """
    Live log viewer as an ipywidgets panel.
    Refreshes every *refresh_seconds* seconds automatically.

    Usage:
        ls = LogStreamer()
        ls.show()                  # renders in the cell output

        ls.filter_level("warning") # show only warnings and errors
        ls.filter_text("tool")     # show only lines containing 'tool'
        ls.set_lines(100)          # show last 100 lines
        ls.pause()  / ls.resume()
        ls.stop()
    """

    def __init__(
        self,
        lines:           int   = 50,
        refresh_seconds: float = 2.0,
    ) -> None:
        self._lines    = lines
        self._interval = refresh_seconds
        self._paused   = False
        self._stop_ev  = threading.Event()
        self._thread:  threading.Thread | None = None
        self._level_filter: str | None = None
        self._text_filter:  str | None = None

        if not _WIDGETS_OK:
            raise ImportError("ipywidgets not available. Use tail_logs() instead.")

        self._out = widgets.Output()
        self._build_controls()

    # ── Controls ──────────────────────────────────────────────────────

    def _build_controls(self) -> None:
        # FIX: Removed dead `btn_style` dict that was defined but never used.
        self._btn_pause = widgets.Button(
            description="⏸ Pause",
            layout=widgets.Layout(width="90px", height="32px"),
        )
        self._btn_clear = widgets.Button(
            description="🗑 Clear",
            layout=widgets.Layout(width="90px", height="32px"),
        )
        # FIX: _btn_scroll was built but had no on_click handler and was not
        # added to the HBox. Wired it up properly here.
        self._btn_scroll = widgets.Button(
            description="⬇ Bottom",
            layout=widgets.Layout(width="90px", height="32px"),
        )

        for btn in (self._btn_pause, self._btn_clear, self._btn_scroll):
            btn.style.button_color = "#151820"
            btn.style.font_weight  = "600"

        self._dd_level = widgets.Dropdown(
            options=["all", "debug", "info", "warning", "error"],
            value="all",
            description="level:",
            layout=widgets.Layout(width="160px"),
            style={"description_width": "40px"},
        )
        self._txt_filter = widgets.Text(
            placeholder="filter text…",
            layout=widgets.Layout(width="200px", height="32px"),
        )
        self._int_lines = widgets.BoundedIntText(
            value=self._lines, min=10, max=500, step=10,
            description="lines:",
            layout=widgets.Layout(width="130px"),
            style={"description_width": "40px"},
        )

        self._btn_pause.on_click(self._on_pause)
        self._btn_clear.on_click(self._on_clear)
        self._btn_scroll.on_click(self._on_scroll)
        self._dd_level.observe(self._on_level_change, names="value")
        self._txt_filter.observe(self._on_text_filter, names="value")
        self._int_lines.observe(self._on_lines_change, names="value")

        self._controls = widgets.HBox(
            # FIX: Added _btn_scroll to the HBox so it actually appears in the UI
            [self._dd_level, self._txt_filter, self._int_lines,
             self._btn_pause, self._btn_clear, self._btn_scroll],
            layout=widgets.Layout(
                padding="8px 14px",
                background_color="#151820",
                border_top="1px solid #252935",
                flex_wrap="wrap",
                gap="8px",
            ),
        )
        self._root = widgets.VBox(
            [self._out, self._controls],
            layout=widgets.Layout(
                border="1px solid #252935",
                border_radius="8px",
                overflow="hidden",
                max_width="960px",
            ),
        )

    # ── Event handlers ────────────────────────────────────────────────

    def _on_pause(self, _) -> None:
        self._paused = not self._paused
        self._btn_pause.description = "▶ Resume" if self._paused else "⏸ Pause"

    def _on_clear(self, _) -> None:
        with self._out:
            clear_output(wait=True)

    def _on_scroll(self, _) -> None:
        """FIX: Trigger a re-render so the auto-scroll script runs immediately."""
        if not self._paused:
            self._render()

    def _on_level_change(self, change) -> None:
        v = change["new"]
        self._level_filter = None if v == "all" else v

    def _on_text_filter(self, change) -> None:
        v = change["new"].strip().lower()
        self._text_filter = v if v else None

    def _on_lines_change(self, change) -> None:
        self._lines = change["new"]

    # ── Public API ────────────────────────────────────────────────────

    def filter_level(self, level: str) -> None:
        """Show only records at *level* and above. Pass 'all' to reset."""
        self._level_filter = None if level == "all" else level.lower()
        self._dd_level.value = level

    def filter_text(self, text: str) -> None:
        """Show only records containing *text*."""
        self._text_filter = text.lower() if text else None
        self._txt_filter.value = text

    def set_lines(self, n: int) -> None:
        """Change the number of tail lines shown."""
        self._lines = n
        self._int_lines.value = n

    def pause(self) -> None:
        self._paused = True
        self._btn_pause.description = "▶ Resume"

    def resume(self) -> None:
        self._paused = False
        self._btn_pause.description = "⏸ Pause"

    def stop(self) -> None:
        self._stop_ev.set()
        print("⏹  Log streamer stopped.")

    def show(self) -> None:
        """Render the widget and start the refresh loop."""
        display(HTML(_CSS))
        display(self._root)
        self._start_refresh()

    # ── Refresh loop ──────────────────────────────────────────────────

    def _start_refresh(self) -> None:
        self._stop_ev.clear()
        self._thread = threading.Thread(
            target=self._loop, daemon=True, name="log-streamer"
        )
        self._thread.start()

    def _loop(self) -> None:
        while not self._stop_ev.wait(timeout=self._interval):
            if not self._paused:
                self._render()

    def _render(self) -> None:
        # FIX: Read all records ONCE and derive both the filtered view and the
        # total count from the same call. Previously read_recent() was called
        # twice — once for filtered records and once for the total count —
        # wasting I/O and creating a potential inconsistency between the two.
        all_records = logger.read_recent(9999)
        total       = len(all_records)
        records     = all_records[-self._lines:]   # apply lines limit first

        # Apply level filter
        if self._level_filter:
            min_i   = _level_index(self._level_filter)
            records = [r for r in records if _level_index(r.get("level", "debug")) >= min_i]

        # Apply text filter
        if self._text_filter:
            records = [
                r for r in records
                if self._text_filter in (r.get("msg") or "").lower()
                or any(self._text_filter in str(v).lower() for v in r.values())
            ]

        rows_html = "\n".join(_record_to_html(r) for r in records)
        now       = time.strftime("%H:%M:%S")
        # FIX: Escape LOG_FILE before embedding in HTML to prevent injection
        log_file  = _html_escape(str(logger.LOG_FILE))

        html = (
            f'<div class="ls-root">'
            f'<div class="ls-header">'
            f'  <span class="ls-title">📋 LIVE LOGS</span>'
            f'  <span class="ls-meta">'
            f'    showing {len(records)} / {total} records'
            f'    &nbsp;·&nbsp; refreshed {now}'
            f'    &nbsp;·&nbsp; {log_file}'
            f'  </span>'
            f'</div>'
            f'<div class="ls-body" id="ls-body">{rows_html}</div>'
            f'</div>'
            f'<script>'
            f'  var b = document.getElementById("ls-body");'
            f'  if(b) b.scrollTop = b.scrollHeight;'
            f'</script>'
        )
        with self._out:
            clear_output(wait=True)
            display(HTML(html))


# ─────────────────────────────────────────────────────────────────────────────
#  tail_logs() — simple snapshot, no widget needed
# ─────────────────────────────────────────────────────────────────────────────

def tail_logs(
    n:     int        = 30,
    level: str | None = None,
    text:  str | None = None,
) -> None:
    """
    Print the last *n* log records to the cell output.
    No widgets required — works anywhere.

    Args:
        n:     number of records to show
        level: filter to 'debug' | 'info' | 'warning' | 'error'
        text:  only show records containing this string

    Usage:
        tail_logs()
        tail_logs(50, level="warning")
        tail_logs(100, text="tool")
    """
    records = logger.read_recent(n)
    if level:
        # FIX: Use _level_index() — direct order.index() raises ValueError
        # for any unknown level string present in a log record.
        min_i   = _level_index(level)
        records = [r for r in records if _level_index(r.get("level", "debug")) >= min_i]
    if text:
        t       = text.lower()
        records = [
            r for r in records
            if t in (r.get("msg") or "").lower()
            or any(t in str(v).lower() for v in r.values())
        ]
    sep = "─" * 72
    print(f"\n{sep}")
    print(f"  LOGS  —  {len(records)} records  —  {logger.LOG_FILE}")
    print(sep)
    for r in records:
        print(logger.format_record(r))
    print(sep + "\n")


# ─────────────────────────────────────────────────────────────────────────────
#  watch_logs() — blocking live tail, like `tail -f` in a terminal
# ─────────────────────────────────────────────────────────────────────────────

def watch_logs(
    poll_seconds: float      = 1.0,
    level:        str | None = None,
    text:         str | None = None,
) -> None:
    """
    Blocking live tail — prints new log lines as they arrive.
    Run in its own cell. Interrupt with the ■ Stop button or Ctrl+C.

    Args:
        poll_seconds: how often to check for new lines (default 1s)
        level:        filter level ('debug'|'info'|'warning'|'error')
        text:         only show lines containing this string

    Usage:
        watch_logs()                      # stream everything
        watch_logs(level="warning")       # warnings + errors only
        watch_logs(text="tool", level="info")
    """
    min_lvl = _level_index(level) if level else 0

    # FIX: Track the last seen record's timestamp instead of a raw count.
    # The old approach sliced records[seen_count:] from a capped read of 500,
    # which broke in two ways:
    #   (a) if >500 new records arrived between polls, older ones were skipped;
    #   (b) if the log rotated, seen_count could exceed len(records) forever,
    #       causing new = [] even when fresh records existed.
    last_ts: str = ""

    print(f"👁  Watching logs — Ctrl+C / ■ Stop to quit\n{'─'*60}")
    try:
        while True:
            # Read a generous window; new records are those after last_ts.
            records = logger.read_recent(500)
            new     = [r for r in records if r.get("ts", "") > last_ts] if last_ts else records
            for r in new:
                r_lvl = _level_index(r.get("level", "debug"))
                if r_lvl < min_lvl:
                    continue
                if text and text.lower() not in (r.get("msg") or "").lower():
                    continue
                print(logger.format_record(r))
            if new:
                last_ts = max(r.get("ts", "") for r in new)
            time.sleep(poll_seconds)
    except KeyboardInterrupt:
        print(f"\n{'─'*60}\n👁  Stopped.")


# ─────────────────────────────────────────────────────────────────────────────
#  Auto-show widget if running in a notebook
# ─────────────────────────────────────────────────────────────────────────────

print("✓ Log streamer ready")
print()
print("  Rich widget (auto-refresh every 2s):")
print("    ls = LogStreamer()")
print("    ls.show()")
print()
print("  Filters:")
print("    ls.filter_level('warning')   # warning + error only")
print("    ls.filter_text('tool')       # lines containing 'tool'")
print("    ls.set_lines(100)            # show last 100 lines")
print("    ls.pause() / ls.resume()")
print()
print("  Simple snapshot (no widget):")
print("    tail_logs(50)")
print("    tail_logs(level='error')")
print("    tail_logs(text='heartbeat')")
print()
print("  Live tail (blocking, like tail -f):")
print("    watch_logs()")
print("    watch_logs(level='warning')")

✓ Log streamer ready

  Rich widget (auto-refresh every 2s):
    ls = LogStreamer()
    ls.show()

  Filters:
    ls.filter_level('warning')   # warning + error only
    ls.filter_text('tool')       # lines containing 'tool'
    ls.set_lines(100)            # show last 100 lines
    ls.pause() / ls.resume()

  Simple snapshot (no widget):
    tail_logs(50)
    tail_logs(level='error')
    tail_logs(text='heartbeat')

  Live tail (blocking, like tail -f):
    watch_logs()
    watch_logs(level='warning')


## CELL 20: Telegram bot

In [44]:
## Telegram Bot
from __future__ import annotations
from datetime import datetime

import concurrent.futures
import io
import json
import os
import queue
import threading
import time
import traceback
from pathlib import Path
from typing import Any

def _get_secret(key: str) -> str | None:
    """Read from Colab Secrets → env var → return None if missing."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val and val.strip():
            return val.strip()
    except Exception:
        pass
    val = os.environ.get(key, "").strip()
    return val if val else None

# Shared level helper — mirrors _level_index() from the log streamer cell.
_LEVEL_ORDER: list[str] = ["debug", "info", "warning", "error"]
_LEVEL_EMOJIS: dict[str, str] = {"debug": "⚙", "info": "ℹ", "warning": "⚠", "error": "❌"}


def _level_index(level: str) -> int:
    """Return the sort index for *level*, defaulting to 0 (debug) if unknown."""
    try:
        return _LEVEL_ORDER.index(level.lower())
    except ValueError:
        return 0

class TelegramBot:
    """
    Telegram bot that wraps the AutomationHub.
    Uses only the `requests` library + Telegram HTTP API — no third-party
    bot frameworks needed.
    Commands:
        /help               — list all commands
        /task <text>        — submit a task to the agent queue (heavy, sequential)
        /t <text>           — shorthand for /task
        /express <question> — ⚡ quick parallel question, skips the queue entirely
        /e <question>       — shorthand for /express
        /express_status     — show recent express lane results
        /status             — full automation hub status
        /queue              — task queue state (pending / done)
        /logs [n]           — last n log lines (default 30)
        /logs_on            — start streaming new logs to Telegram
        /logs_off           — stop log streaming
        /loglevel <level>   — set stream filter (debug/info/warning/error)
        /logfilter <text>   — only stream lines containing <text>
        /browse [path]      — interactive file browser (tap folders/files)
        /files [path]       — list workspace files (text)
        /getfile <path>     — send a file from the workspace
        /memory             — show MEMORY.md contents
        /needs              — show pending needs
        /save               — trigger Drive save now
        /cancel             — cancel the currently-running queue task
        /id                 — show your chat_id (for whitelist setup)
    """

    _API = "https://api.telegram.org/bot{token}/{method}"

    _MAX_MSG              = 4096
    _LOG_PUSH_INTERVAL    = 5
    _POLL_TIMEOUT         = 30
    _MAX_DISPATCH_WORKERS = 8
    _BROWSE_PAGE_SIZE     = 10
    _PATH_REGISTRY_MAX    = 500
    # Subfolder inside Config.FILES_DIR where Telegram uploads are saved.
    # Auto-created on first upload. Users can organise further by sending
    # a caption like "reports/jan.pdf" to route into a sub-subfolder.
    _UPLOAD_SUBDIR        = "telegram_uploads"
    # Telegram getFile API limit — files larger than this cannot be downloaded
    # via the bot API regardless of Telegram's general 2 GB upload limit.
    _UPLOAD_MAX_MB        = 20

    def __init__(
        self,
        hub:              "AutomationHub | None" = None,
        token:            str | None             = None,
        allowed_chat_ids: list[int] | None       = None,
    ) -> None:
        self._hub   = hub
        self._token = token or _get_secret("TELEGRAM_BOT_TOKEN")
        if not self._token:
            raise ValueError(
                "No Telegram bot token found.\n"
                "  Add TELEGRAM_BOT_TOKEN to Colab Secrets or set the env var."
            )

        # Whitelist
        if allowed_chat_ids:
            self._allowed: set[int] = set(allowed_chat_ids)
        else:
            raw = _get_secret("TELEGRAM_CHAT_ID")
            self._allowed = {int(raw)} if raw and raw.lstrip("-").isdigit() else set()

        # State
        self._stop_ev           = threading.Event()
        self._poll_thread:      threading.Thread | None = None
        self._log_thread:       threading.Thread | None = None
        self._log_streaming:    bool                    = False
        self._log_level_filter: str | None              = None
        self._log_text_filter:  str | None              = None
        self._log_stop_ev:      threading.Event         = threading.Event()
        self._update_offset:    int                     = 0
        self._notify_queue:     queue.Queue             = queue.Queue()
        self._send_thread:      threading.Thread | None = None
        self._chat_mode_ids:    set[int]                = set()

        # ── File browser path registry ──────────────────────────────────────
        self._path_registry:      dict[int, Path] = {}
        self._path_registry_lock: threading.Lock  = threading.Lock()
        self._path_registry_seq:  int             = 0

        # Verify token works
        me = self._api("getMe")
        if not me.get("ok"):
            raise RuntimeError(f"Telegram token invalid: {me}")
        self._bot_name = me["result"].get("username", "agent_bot")
        print(f"✓ Telegram bot @{self._bot_name} initialised")
        if self._allowed:
            print(f"  Whitelisted chat IDs: {sorted(self._allowed)}")
        else:
            print("  ⚠  No whitelist set — send /id to your bot to get your chat_id,")
            print("     then add TELEGRAM_CHAT_ID to Colab Secrets and restart.")

    # ─────────────────────────────────────────────────────────────────
    #  HTTP helpers
    # ─────────────────────────────────────────────────────────────────

    def _api(self, method: str, **kwargs) -> dict:
        import requests
        url = self._API.format(token=self._token, method=method)
        try:
            resp = requests.post(url, timeout=35, **kwargs)
            return resp.json()
        except requests.exceptions.Timeout:
            logger.error(f"Telegram API timeout [{method}]")
            return {"ok": False, "error": "timeout"}
        except requests.exceptions.RequestException as e:
            logger.error(f"Telegram API request error [{method}]: {e}")
            return {"ok": False, "error": str(e)}
        except Exception as e:
            logger.warning(f"Unexpected Telegram API error [{method}]: {e}")
            return {"ok": False, "error": str(e)}

    def _send(self, chat_id: int, text: str, parse_mode: str = "HTML") -> None:
        self._notify_queue.put(("text", chat_id, text, parse_mode))

    def _send_file(self, chat_id: int, path: Path, caption: str = "") -> None:
        self._notify_queue.put(("file", chat_id, path, caption))

    def _chunk(self, text: str) -> list[str]:
        if not text:
            return []
        return [text[i: i + self._MAX_MSG] for i in range(0, len(text), self._MAX_MSG)]

    # ─────────────────────────────────────────────────────────────────
    #  Send worker
    # ─────────────────────────────────────────────────────────────────

    def _send_worker(self) -> None:
        while not self._stop_ev.is_set() or not self._notify_queue.empty():
            try:
                item = self._notify_queue.get(timeout=1)
            except queue.Empty:
                continue
            kind = item[0]
            if kind == "text":
                _, chat_id, text, parse_mode = item
                for chunk in self._chunk(text):
                    self._api("sendMessage",
                              json={"chat_id": chat_id, "text": chunk,
                                    "parse_mode": parse_mode})
            elif kind == "file":
                _, chat_id, path, caption = item
                try:
                    with open(path, "rb") as fh:
                        self._api("sendDocument",
                                  data={"chat_id": chat_id, "caption": caption},
                                  files={"document": fh})
                except Exception as e:
                    self._api("sendMessage",
                              json={"chat_id": chat_id,
                                    "text": f"❌ Could not send file: {e}"})
            self._notify_queue.task_done()

    # ─────────────────────────────────────────────────────────────────
    #  Path registry
    # ─────────────────────────────────────────────────────────────────

    def _reg_put(self, path: Path) -> int:
        with self._path_registry_lock:
            if len(self._path_registry) >= self._PATH_REGISTRY_MAX:
                for k in sorted(self._path_registry)[:50]:
                    del self._path_registry[k]
            self._path_registry_seq += 1
            self._path_registry[self._path_registry_seq] = path
            return self._path_registry_seq

    def _reg_get(self, key: int) -> Path | None:
        with self._path_registry_lock:
            return self._path_registry.get(key)

    # ─────────────────────────────────────────────────────────────────
    #  Command dispatcher
    # ─────────────────────────────────────────────────────────────────

    def _dispatch(self, chat_id: int, text: str) -> None:
        text = text.strip()
        cmd, _, args = text.partition(" ")
        cmd = cmd.lower().lstrip("/")

        # ── Always-allowed ─────────────────────────────────────────
        if cmd == "id":
            self._send(chat_id, f"Your chat_id: <code>{chat_id}</code>")
            return
        if cmd in ("start", "help"):
            self._cmd_help(chat_id)
            return

        # ── Whitelist gate ─────────────────────────────────────────
        if self._allowed and chat_id not in self._allowed:
            self._send(chat_id,
                       "⛔ Unauthorised. Your chat_id is "
                       f"<code>{chat_id}</code> — add it to TELEGRAM_CHAT_ID.")
            return

        # ── Dispatch ───────────────────────────────────────────────
        handlers = {
            "task":           self._cmd_task,
            "t":              self._cmd_task,
            "express":        self._cmd_express,     # ⚡ parallel quick lane
            "e":              self._cmd_express,     # shorthand
            "express_status": self._cmd_express_status,
            "status":         self._cmd_status,
            "queue":          self._cmd_queue,
            "logs":           self._cmd_logs,
            "logs_on":        self._cmd_logs_on,
            "logs_off":       self._cmd_logs_off,
            "loglevel":       self._cmd_loglevel,
            "logfilter":      self._cmd_logfilter,
            "browse":         self._cmd_browse,
            "files":          self._cmd_files,
            "getfile":        self._cmd_getfile,
            "uploads":        self._cmd_uploads,        # list uploaded files
            "memory":         self._cmd_memory,
            "needs":          self._cmd_needs,
            "save":           self._cmd_save,
            "panel":          self._cmd_panel,
            "cancel":         self._cmd_cancel,
            "export":         self._cmd_export,
            "diff":           self._cmd_diff,
            "cost":           self._cmd_cost,
            "semantic":       self._cmd_semantic,
        }
        fn = handlers.get(cmd)
        if fn:
            try:
                fn(chat_id, args.strip())
            except Exception as e:
                self._send(chat_id, f"❌ Error: <code>{e}</code>")
                logger.error(f"Telegram cmd error [{cmd}]: {e}\n{traceback.format_exc()}")
        else:
            if chat_id in self._chat_mode_ids:
                self._cmd_express(chat_id, text)   # direct chat → express lane
            elif len(text) > 8 and not text.startswith("/"):
                self._cmd_task(chat_id, text)
            else:
                self._send(chat_id, "❓ Unknown command. Send /help or /panel.")

    # ─────────────────────────────────────────────────────────────────
    #  Command implementations
    # ─────────────────────────────────────────────────────────────────

    def _cmd_help(self, chat_id: int, _: str = "") -> None:
        self._send(chat_id, (
            "🤖 <b>Agent Bot Commands</b>\n\n"
            "<b>🎛 Control Panel</b>\n"
            "  /panel              — interactive button control panel\n\n"
            "<b>Tasks</b>\n"
            "  /task &lt;text&gt;      — queue a heavy task (runs sequentially)\n"
            "  /t &lt;text&gt;         — shorthand for /task\n\n"
            "<b>⚡ Express Lane</b>  <i>(parallel — never waits for the queue)</i>\n"
            "  /express &lt;q&gt;      — quick question answered in parallel right now\n"
            "  /e &lt;q&gt;            — shorthand for /express\n"
            "  /express_status     — show recent express results\n\n"
            "<b>Status</b>\n"
            "  /status             — full hub status\n"
            "  /queue              — task queue state\n"
            "  /memory             — show MEMORY.md\n"
            "  /needs              — pending needs\n\n"
            "<b>Logs</b>\n"
            "  /logs [n]           — last n lines (default 30)\n"
            "  /logs_on            — stream new logs here\n"
            "  /logs_off           — stop streaming\n"
            "  /loglevel &lt;l&gt;     — filter: debug/info/warning/error\n"
            "  /logfilter &lt;t&gt;    — only lines containing t\n\n"
            "<b>Files</b>\n"
            "  /browse [path]      — 🗂 interactive browser — tap folders &amp; files\n"
            "  /files [path]       — list workspace files (text)\n"
            "  /getfile &lt;p&gt;      — send a file by path\n"
            "  /uploads [n]        — list last n files uploaded via Telegram\n"
            "  <i>📎 Send any file directly to this chat to upload it to the workspace.\n"
            "     Caption with a path (e.g. reports/jan.pdf) to organise into subfolders.</i>\n\n"
            "<b>System</b>\n"
            "  /save               — save workspace to Drive now\n"
            "  /cancel             — cancel the currently-running queue task\n"
            "  /export [fmt]       — export conversation (md/json/html)\n"
            "  /diff               — workspace file changes since session start\n"
            "  /cost               — show token usage and estimated USD cost\n"
            "  /semantic &lt;q&gt;     — semantic memory search\n"
            "  /id                 — show your chat_id\n"
        ))

    def _cmd_task(self, chat_id: int, args: str) -> None:
        if not args:
            self._send(chat_id, "Usage: /task &lt;your task description&gt;")
            return
        if self._hub is None:
            self._send(chat_id, "❌ No AutomationHub attached. Run autorun() first.")
            return
        self._send(chat_id, f"📥 Queued: <i>{args[:200]}</i>")

        def _run():
            try:
                result = self._hub.agent.run(args)
                reply  = f"✅ <b>Done</b>\n\n{result[:3800]}"
                if len(result) > 3800:
                    reply += f"\n\n<i>…{len(result)-3800:,} chars truncated</i>"
            except Exception as e:
                reply = f"❌ Task failed: <code>{e}</code>"
            self._send(chat_id, reply)

        threading.Thread(target=_run, daemon=True).start()

    # ─────────────────────────────────────────────────────────────────
    #  ⚡ Express lane commands
    # ─────────────────────────────────────────────────────────────────

    def _cmd_express(self, chat_id: int, args: str) -> None:
        """
        /express <question>  or  /e <question>

        Sends a quick question to the ExpressLane, which runs it in parallel
        using a lightweight single-turn API call — completely independent of
        the main TaskQueue. The queue keeps running while this answers.

        The user gets an immediate "thinking…" acknowledgement, then the
        answer arrives as a follow-up message when the API responds (usually
        1-5 seconds).
        """
        if not args:
            self._send(chat_id,
                "Usage: /express &lt;question&gt;\n"
                "       /e &lt;question&gt;\n\n"
                "<i>Express runs in parallel — the queue is never blocked.</i>\n\n"
                "Examples:\n"
                "  /e How many .py files are in the workspace?\n"
                "  /e Summarise the last 3 log lines\n"
                "  /e What is today's date and agent uptime?")
            return
        if self._hub is None:
            self._send(chat_id, "❌ No AutomationHub attached. Run autorun() first.")
            return
        if not hasattr(self._hub, "express"):
            self._send(chat_id,
                "❌ Express lane not available — update to the latest automation cell.")
            return

        # Acknowledge immediately so the user knows it's running
        queue_status = ""
        if self._hub.queue._current:
            queue_status = (
                f"\n<i>(Queue is running: {self._hub.queue._current[:60]}… "
                f"— this runs in parallel)</i>"
            )
        self._send(chat_id,
            f"⚡ <b>Express</b>: <i>{args[:200]}</i>{queue_status}")

        # Callback delivers the result as a follow-up message
        def _on_result(result: str) -> None:
            truncated = f"\n<i>…{len(result)-1500:,} chars truncated</i>" if len(result) > 1500 else ""
            self._send(chat_id,
                f"⚡ <b>Express result</b>\n\n{result[:1500]}{truncated}")

        try:
            self._hub.express.run(args, callback=_on_result)
        except Exception as e:
            self._send(chat_id, f"❌ Express lane error: <code>{e}</code>")

    def _cmd_express_status(self, chat_id: int, _: str = "") -> None:
        """
        /express_status  — show recent express lane results.
        """
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        if not hasattr(self._hub, "express"):
            self._send(chat_id, "❌ Express lane not available.")
            return

        with self._hub.express._lock:
            results = dict(self._hub.express._results)

        if not results:
            self._send(chat_id, "⚡ No express results yet. Use /express &lt;question&gt;")
            return

        recent = sorted(results.values(),
                        key=lambda x: x.get("started", ""))[-8:]
        lines  = [f"⚡ <b>Express Lane</b>  —  {len(results)} total result(s)\n"]
        for r in reversed(recent):
            ts     = (r.get("finished") or r.get("started") or "?")[:16].replace("T", " ")
            status = "✓" if r["status"] == "done" else "▶"
            res    = (r.get("result") or "")[:120]
            lines.append(
                f"{status} <code>{ts}</code>  <i>{r['task'][:60]}</i>"
                + (f"\n   → {res}" if res else "")
            )
        self._send(chat_id, "\n".join(lines))

    # ─────────────────────────────────────────────────────────────────
    #  Status / queue commands
    # ─────────────────────────────────────────────────────────────────

    def _cmd_status(self, chat_id: int, _: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        s  = self._hub.agent.counter.summary()
        hb = getattr(self._hub.agent, "_heartbeat_thread", None)

        def _dot(thread):
            return "🟢" if thread and thread.is_alive() else "🔴"

        express_ok   = hasattr(self._hub, "express") and self._hub.express._pool is not None
        express_line = (
            f"  {'🟢' if express_ok else '🔴'} express  "
            f"<i>({getattr(self._hub.express, 'MAX_WORKERS', '?')} workers, "
            f"{len(getattr(self._hub.express, '_results', {}))} results)</i>"
            if hasattr(self._hub, "express") else
            "  ⚪ express  <i>(not available)</i>"
        )

        lines = [
            "📊 <b>Agent Status</b>",
            f"  Model:      <code>{Config.MODEL}</code>",
            f"  Workspace:  <code>{Config.BASE_DIR}</code>",
            "",
            "<b>Threads</b>",
            f"  {_dot(hb)} heartbeat",
            f"  {_dot(self._hub.queue._thread)} task_queue"
            + (f"  <i>(running: {self._hub.queue._current[:40]}…)</i>"
               if self._hub.queue._current else ""),
            express_line,
            f"  {_dot(self._hub.scheduler._thread)} scheduler",
            f"  {_dot(self._hub.need_poller._thread)} need_poller",
            f"  {_dot(self._hub.watchdog._thread)} watchdog",
            f"  {'🟢' if self._log_streaming else '⚪'} log_stream",
            "",
            "<b>API Stats</b>",
            f"  Requests: {s['total_requests']} (✓{s['success']} ✗{s['errors']})",
            f"  Tokens:   {s['tokens_total']:,}",
            f"  Uptime:   {s['uptime_seconds']}s",
        ]
        self._send(chat_id, "\n".join(lines))

    def _cmd_queue(self, chat_id: int, _: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        tq = self._hub.queue

        with tq._lock:
            pending = list(tq._q.queue)
            done    = list(tq._done[-5:])
            current = tq._current
            n_done  = len(tq._done)

        paused  = getattr(tq, "_paused", False)
        running = tq._thread and tq._thread.is_alive()

        lines = [
            f"📋 <b>Task Queue</b>  —  "
            f"{len(pending)} pending  /  {n_done} done"
            + ("  ⏸ paused" if paused else "")
            + ("  🔴 worker stopped" if not running else ""),
        ]
        if current:
            lines.append(f"\n▶ <b>Running:</b> <i>{current[:100]}</i>")
            lines.append(
                f"<i>💡 Tip: use /express or /e to ask something "
                f"without waiting for this to finish.</i>")
        else:
            lines.append("\n⚪ <i>No task currently running.</i>")

        if pending:
            lines.append("\n<b>Pending:</b>")
            for i, item in enumerate(pending[:10]):
                submitted = (item.get("submitted") or "")[:16].replace("T", " ")
                lines.append(f"  {i+1}. {item['task'][:70]}"
                             + (f"  <i>({submitted})</i>" if submitted else ""))
            if len(pending) > 10:
                lines.append(f"  <i>… and {len(pending)-10} more</i>")
        else:
            lines.append("\n<i>Queue is empty.</i>")

        if done:
            lines.append("\n<b>Last completed:</b>")
            for item in done:
                ok = "✓" if not item.get("error") else "✗"
                ts = (item.get("finished") or "")[:16].replace("T", " ")
                lines.append(f"  {ok} <code>{ts}</code>  {item['task'][:60]}")

        self._send(chat_id, "\n".join(lines))

    def _cmd_logs(self, chat_id: int, args: str) -> None:
        n = int(args) if args.isdigit() else 30
        records = logger.read_recent(n)
        if not records:
            self._send(chat_id, "No logs found.")
            return
        lines = [f"📋 <b>Last {len(records)} log records</b>\n"]
        for r in records:
            lvl   = r.get("level", "info")
            ts    = (r.get("ts") or "")[:16].replace("T", " ")
            msg   = (r.get("msg") or "")[:120]
            emoji = _LEVEL_EMOJIS.get(lvl, "•")
            lines.append(f"{emoji} <code>{ts}</code>  {msg}")
        self._send(chat_id, "\n".join(lines))

    def _start_log_stream(self, chat_id: int) -> None:
        self._log_stop_ev.set()
        if self._log_thread and self._log_thread.is_alive():
            self._log_thread.join(timeout=2)
        self._log_stop_ev.clear()
        self._log_streaming = True
        recent = logger.read_recent(10)
        cursor = recent[-1].get("ts", "") if recent else ""
        if recent:
            self._send_log_batch(chat_id, recent,
                                 label="📡 <b>Log stream started</b>  (last 10 records)\n")
        self._log_thread = threading.Thread(
            target=self._log_push_loop, args=(chat_id, cursor),
            daemon=True, name="tg-log-push",
        )
        self._log_thread.start()

    def _stop_log_stream(self) -> None:
        self._log_streaming = False
        self._log_stop_ev.set()

    def _send_log_batch(
        self, chat_id: int, records: list[dict], label: str = "📡 <b>Live Logs</b>\n"
    ) -> None:
        if self._log_level_filter:
            min_i   = _level_index(self._log_level_filter)
            records = [r for r in records
                       if _level_index(r.get("level", "debug")) >= min_i]
        if self._log_text_filter:
            records = [r for r in records
                       if self._log_text_filter in (r.get("msg") or "").lower()]
        if not records:
            return
        lines = [f"{label}"]
        for r in records[-20:]:
            lvl   = r.get("level", "info")
            ts    = (r.get("ts") or "")[:16].replace("T", " ")
            msg   = (r.get("msg") or "")[:120]
            emoji = _LEVEL_EMOJIS.get(lvl, "•")
            lines.append(f"{emoji} <code>{ts}</code>  {msg}")
        if len(records) > 20:
            lines.append(f"\n<i>…{len(records)-20} more lines omitted</i>")
        self._send(chat_id, "\n".join(lines))

    def _log_push_loop(self, chat_id: int, cursor: str) -> None:
        while not self._log_stop_ev.wait(timeout=self._LOG_PUSH_INTERVAL):
            if not self._log_streaming:
                break
            try:
                all_records = logger.read_recent(200)
                new = [r for r in all_records if (r.get("ts") or "") > cursor]
                if new:
                    cursor = new[-1].get("ts", cursor)
                    self._send_log_batch(
                        chat_id, new,
                        label=f"📡 <b>Live Logs</b>  (+{len(new)} new)\n",
                    )
            except Exception as e:
                logger.warning(f"Telegram log push error: {e}")

    def _cmd_logs_on(self, chat_id: int, _: str = "") -> None:
        self._start_log_stream(chat_id)
        self._send(chat_id,
            "🟢 <b>Log streaming ON</b>\nNew log lines pushed every 5s.\n"
            "Send /logs_off to stop.")

    def _cmd_logs_off(self, chat_id: int, _: str = "") -> None:
        self._stop_log_stream()
        self._send(chat_id, "⚪ Log streaming OFF.")

    def _cmd_loglevel(self, chat_id: int, args: str) -> None:
        valid = {"debug", "info", "warning", "error", "all"}
        lvl   = args.lower().strip()
        if lvl not in valid:
            self._send(chat_id, f"Valid levels: {', '.join(sorted(valid))}")
            return
        self._log_level_filter = None if lvl == "all" else lvl
        self._send(chat_id, f"Log level filter → <code>{lvl}</code>")

    def _cmd_logfilter(self, chat_id: int, args: str) -> None:
        self._log_text_filter = args.lower().strip() if args.strip() else None
        self._send(chat_id,
                   f"Log text filter → <code>{self._log_text_filter or 'none'}</code>")

    # ─────────────────────────────────────────────────────────────────
    #  Interactive file browser
    # ─────────────────────────────────────────────────────────────────

    def _cmd_browse(self, chat_id: int, args: str = "") -> None:
        workspace = Config.FILES_DIR.resolve()
        if args.strip():
            candidate = (Config.FILES_DIR / args.strip()).resolve()
            try:
                candidate.relative_to(workspace)
                target = candidate
            except ValueError:
                target = workspace
        else:
            target = workspace
        self._browse_render(chat_id, message_id=None, path=target, page=0)

    def _browse_render(
        self, chat_id: int, message_id: int | None, path: Path, page: int = 0,
    ) -> None:
        workspace = Config.FILES_DIR.resolve()
        try:
            path.relative_to(workspace)
        except ValueError:
            path = workspace
        try:
            raw = sorted(path.iterdir(), key=lambda e: (e.is_file(), e.name.lower()))
        except PermissionError:
            msg = f"❌ Permission denied: <code>{path.name}</code>"
            self._edit_kb(chat_id, message_id, msg) if message_id else self._send(chat_id, msg)
            return

        dirs  = [e for e in raw if e.is_dir()]
        files = [e for e in raw if e.is_file()]
        all_entries = dirs + files

        total_pages = max(1, -(-len(all_entries) // self._BROWSE_PAGE_SIZE))
        page        = max(0, min(page, total_pages - 1))
        page_slice  = all_entries[page * self._BROWSE_PAGE_SIZE:
                                  (page + 1) * self._BROWSE_PAGE_SIZE]

        rows: list[list[tuple[str, str]]] = []
        for entry in page_slice:
            key = self._reg_put(entry)
            if entry.is_dir():
                label = f"📁  {entry.name}/"
                data  = f"browse:dir:{key}:0"
            else:
                try:
                    size = entry.stat().st_size
                    sz   = f"{size/1024:.0f} KB" if size >= 1024 else f"{size} B"
                except OSError:
                    sz = "?"
                label = f"📄  {entry.name}  ({sz})"
                data  = f"browse:get:{key}"
            rows.append([(label[:60], data)])

        dir_key  = self._reg_put(path)
        page_nav: list[tuple[str, str]] = []
        if page > 0:
            page_nav.append(("◀ Prev", f"browse:dir:{dir_key}:{page-1}"))
        if page < total_pages - 1:
            page_nav.append(("Next ▶", f"browse:dir:{dir_key}:{page+1}"))
        if page_nav:
            rows.append(page_nav)

        nav_row: list[tuple[str, str]] = []
        if path != workspace:
            parent_key = self._reg_put(path.parent)
            nav_row.append(("⬆ Up", f"browse:dir:{parent_key}:0"))
        nav_row.append(("🏠 Root", "browse:root:0"))
        rows.append(nav_row)

        rel    = str(path.relative_to(workspace)) if path != workspace else "."
        header = (
            f"🗂 <b>{rel}</b>\n"
            f"  {len(dirs)} folder(s)  ·  {len(files)} file(s)"
            + (f"  ·  page {page+1}/{total_pages}" if total_pages > 1 else "")
            + "\n\nTap 📄 to download  ·  Tap 📁 to open"
        )

        if message_id:
            self._edit_kb(chat_id, message_id, header, rows=rows)
        else:
            self._send_kb(chat_id, header, rows)

    def _cmd_files(self, chat_id: int, args: str) -> None:
        sub_path = args.strip() or ""
        try:
            target   = (Config.FILES_DIR / sub_path) if sub_path else Config.FILES_DIR
            resolved = target.resolve()
            try:
                resolved.relative_to(Config.FILES_DIR.resolve())
            except ValueError:
                self._send(chat_id, "❌ Path is outside the workspace.")
                return
            if not resolved.exists():
                self._send(chat_id, f"❌ Path not found: {target}")
                return
            entries = sorted(resolved.rglob("*"))
            files   = [e for e in entries if e.is_file()][:50]
            lines   = [f"📁 <b>{target}</b>  ({len(files)} files)\n"]
            for f in files:
                size = f.stat().st_size
                rel  = str(f.relative_to(Config.FILES_DIR))
                sz   = f"{size/1024:.1f}KB" if size > 1024 else f"{size}B"
                lines.append(f"  📄 <code>{rel}</code>  <i>{sz}</i>")
            self._send(chat_id, "\n".join(lines))
        except Exception as e:
            self._send(chat_id, f"❌ {e}")

    def _cmd_getfile(self, chat_id: int, args: str) -> None:
        if not args:
            self._send(chat_id, "Usage: /getfile &lt;relative/path/to/file&gt;")
            return
        candidates     = [Config.FILES_DIR / args, Config.BASE_DIR / args]
        workspace_root = Config.BASE_DIR.resolve()
        target         = None
        for p in candidates:
            try:
                resolved = p.resolve()
                resolved.relative_to(workspace_root)
                if resolved.exists() and resolved.is_file():
                    target = resolved
                    break
            except ValueError:
                continue
        if not target:
            self._send(chat_id, f"❌ File not found: <code>{args}</code>")
            return
        self._deliver_file(chat_id, target)

    def _deliver_file(self, chat_id: int, path: Path) -> None:
        size_mb = path.stat().st_size / (1024 * 1024)
        if size_mb > 49:
            self._send(chat_id,
                       f"❌ File too large ({size_mb:.1f} MB). Telegram limit is 50 MB.")
            return
        self._send(chat_id, f"📤 Sending <code>{path.name}</code>…")
        self._send_file(chat_id, path, caption=str(path.relative_to(Config.BASE_DIR)))

    # ─────────────────────────────────────────────────────────────────
    #  Document / media upload  (Telegram → workspace)
    # ─────────────────────────────────────────────────────────────────

    def _handle_upload(
        self,
        chat_id:    int,
        file_obj:   dict,
        media_type: str,
        caption:    str,
    ) -> None:
        """
        Download a file sent to the bot and save it to the workspace.

        Destination:  Config.FILES_DIR / telegram_uploads / [subfolder/] filename
        Subfolder:    optional — extracted from the caption if it looks like a path
                      e.g. caption "reports/jan.pdf" saves to telegram_uploads/reports/jan.pdf
                      e.g. caption "my invoice"     saves to telegram_uploads/my_invoice_<orig>.ext

        Security:
          - All paths validated to stay inside Config.FILES_DIR
          - Filename sanitised (no path separators, no leading dots, ASCII-safe)
          - Files > _UPLOAD_MAX_MB (20 MB) rejected — Telegram's own getFile limit
          - Whitelisted senders only (whitelist gate is in _poll_loop via _dispatch
            for text; here we enforce it directly)

        Usage in Telegram:
          • Send any file → saved to workspace/telegram_uploads/<original_name>
          • Send with caption "reports/q1.csv" → saved to telegram_uploads/reports/q1.csv
          • Send with caption "notes" → treated as a description (not a path), ignored
          • /uploads → list recent uploads
        """
        import re as _re

        # ── Whitelist gate ────────────────────────────────────────────
        if self._allowed and chat_id not in self._allowed:
            self._send(chat_id,
                "⛔ Unauthorised upload attempt. Your chat_id is "
                f"<code>{chat_id}</code>.")
            return

        # ── Size pre-check from file_obj metadata ─────────────────────
        file_size = file_obj.get("file_size", 0)
        if file_size and file_size > self._UPLOAD_MAX_MB * 1024 * 1024:
            self._send(chat_id,
                f"❌ File too large ({file_size / 1024 / 1024:.1f} MB).\n"
                f"Telegram bot API limit is {self._UPLOAD_MAX_MB} MB.")
            return

        # ── Get the download URL via getFile ─────────────────────────
        file_id   = file_obj.get("file_id", "")
        resp      = self._api("getFile", json={"file_id": file_id})
        if not resp.get("ok"):
            self._send(chat_id,
                f"❌ Could not fetch file info: {resp.get('description', resp)}")
            return
        file_path_tg = resp["result"].get("file_path", "")
        if not file_path_tg:
            self._send(chat_id, "❌ Telegram returned no file path.")
            return

        # ── Determine filename ────────────────────────────────────────
        # Priority: explicit original filename > Telegram path basename > media_type + timestamp
        orig_name = (
            file_obj.get("file_name")                          # documents always have this
            or Path(file_path_tg).name                        # photos/voice use Telegram's name
            or f"{media_type}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        )

        # ── Sanitise the filename ─────────────────────────────────────
        # Strip path separators, null bytes, leading dots — preserve extension.
        safe_name = _re.sub(r'[/\\:*?"<>|\x00]', "_", orig_name).lstrip(".")
        if not safe_name:
            safe_name = f"upload_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        # ── Parse caption for optional subfolder hint ─────────────────
        # Accept captions like "reports/jan.pdf" or "data/raw/file.csv".
        # Reject captions that look like plain text (no "/" or extension match).
        subfolder = Path()
        if caption and "/" in caption:
            # Split into dir part + optional filename override
            cap_path  = Path(caption)
            cap_parts = list(cap_path.parts)
            if len(cap_parts) >= 2:
                # Use all parts except last as subfolder; last part as filename
                # if it looks like a filename (has an extension), else just subfolder.
                potential_name = cap_parts[-1]
                if "." in potential_name:
                    safe_name = _re.sub(r'[/\\:*?"<>|\x00]', "_", potential_name).lstrip(".")
                    subfolder = Path(*cap_parts[:-1]) if len(cap_parts) > 1 else Path()
                else:
                    subfolder = Path(*cap_parts)
            # Validate: no ".." components in subfolder
            subfolder = Path(*[p for p in subfolder.parts if p not in (".", "..")])

        # ── Resolve final destination path ────────────────────────────
        upload_root = Config.FILES_DIR / self._UPLOAD_SUBDIR
        dest_dir    = (upload_root / subfolder).resolve()

        # Security: confirm dest_dir is still inside upload_root
        try:
            dest_dir.relative_to(Config.FILES_DIR.resolve())
        except ValueError:
            self._send(chat_id, "❌ Invalid upload path — must stay inside the workspace.")
            return

        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / safe_name

        # ── Conflict: add timestamp suffix if file already exists ─────
        if dest.exists():
            stem = dest.stem
            suf  = dest.suffix
            ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
            dest = dest_dir / f"{stem}_{ts}{suf}"

        # ── Download from Telegram CDN ─────────────────────────────────
        import requests as _req
        self._send(chat_id,
            f"📥 <b>Uploading</b> <code>{safe_name}</code>"
            + (f" → <code>{subfolder}/</code>" if str(subfolder) != "." else "")
            + "…"
        )
        try:
            dl_url = f"https://api.telegram.org/file/bot{self._token}/{file_path_tg}"
            r = _req.get(dl_url, timeout=120, stream=True)
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(chunk_size=65536):
                    if chunk:
                        fh.write(chunk)
        except Exception as e:
            self._send(chat_id, f"❌ Download failed: <code>{e}</code>")
            logger.error(f"Telegram upload download failed: {e}")
            return

        # ── Success ───────────────────────────────────────────────────
        size_kb  = dest.stat().st_size / 1024
        rel_path = dest.relative_to(Config.FILES_DIR)
        logger.info(f"Telegram upload saved: {dest}")
        self._send(chat_id,
            f"✅ <b>Saved to workspace</b>\n"
            f"  📄 <code>{rel_path}</code>\n"
            f"  Size: {size_kb:.1f} KB\n\n"
            f"<i>Tip: use /browse to navigate to it, or /getfile {rel_path} to download it back.</i>"
        )

    def _cmd_uploads(self, chat_id: int, args: str = "") -> None:
        """
        /uploads [n]  — list the last n files uploaded via Telegram (default 20).
        Shows filename, size, and timestamp. Tap a filename to download it back.
        """
        try:
            n = max(1, min(int(args.strip()), 100)) if args.strip().isdigit() else 20
        except ValueError:
            n = 20

        upload_root = Config.FILES_DIR / self._UPLOAD_SUBDIR
        if not upload_root.exists():
            self._send(chat_id,
                "📭 No uploads yet.\n\n"
                "<i>Send any file to this bot to upload it to your workspace.</i>")
            return

        # Collect all files sorted by modification time (newest first)
        all_files = sorted(
            (f for f in upload_root.rglob("*") if f.is_file()),
            key=lambda f: f.stat().st_mtime,
            reverse=True,
        )[:n]

        if not all_files:
            self._send(chat_id, "📭 No uploaded files found in <code>telegram_uploads/</code>.")
            return

        lines = [f"📎 <b>Last {len(all_files)} upload(s)</b>\n"]
        for f in all_files:
            rel   = str(f.relative_to(Config.FILES_DIR))
            size  = f.stat().st_size
            sz    = f"{size/1024:.1f} KB" if size >= 1024 else f"{size} B"
            mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
            lines.append(f"  📄 <code>{rel}</code>  <i>{sz}  {mtime}</i>")

        lines.append(
            f"\n<i>Use /getfile &lt;path&gt; to download any file back.\n"
            f"Use /browse telegram_uploads to navigate interactively.</i>"
        )
        self._send(chat_id, "\n".join(lines))

    def _cmd_memory(self, chat_id: int, _: str = "") -> None:
        content = FileManager.read_file(Config.CORE_DIR / "MEMORY.md", default="*(empty)*")
        self._send(chat_id, f"🧠 <b>MEMORY.md</b>\n\n{content[:3800]}")

    def _cmd_needs(self, chat_id: int, _: str = "") -> None:
        needs = check_needs()
        if not needs:
            self._send(chat_id, "✅ No pending needs.")
            return
        lines = [f"🔔 <b>{len(needs)} Pending Need(s)</b>\n"]
        for i, n in enumerate(needs, 1):
            lines.append(
                f"<b>{i}. [{n.get('priority','?')}]</b> {n.get('needs','?')}\n"
                f"   Reason:   {n.get('reason','?')}\n"
                f"   Blocking: {n.get('blocking','?')}"
            )
        self._send(chat_id, "\n".join(lines))

    def _cmd_cancel(self, chat_id: int, _: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        result = self._hub.queue.cancel_current()
        if hasattr(self._hub, "agent"):
            self._hub.agent._cancel_flag = True
        self._send(chat_id, f"🛑 {result}")

    def _cmd_export(self, chat_id: int, args: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        fmt = args.strip().lower() or "md"
        if fmt not in ("md", "json", "html"):
            self._send(chat_id, "Usage: /export [md|json|html]")
            return
        self._send(chat_id, f"📤 Exporting conversation as {fmt.upper()}…")
        try:
            path = self._hub.agent.export_conversation(fmt=fmt)
            self._send_file(chat_id, Path(path), caption=f"Conversation export ({fmt.upper()})")
        except Exception as e:
            self._send(chat_id, f"❌ Export failed: {e}")

    def _cmd_diff(self, chat_id: int, _: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        result = self._hub.agent._workspace_diff()
        self._send(chat_id, f"📊 {result}")

    def _cmd_cost(self, chat_id: int, _: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        s    = self._hub.agent.counter.summary()
        cost = s.get("cost_usd", 0)
        self._send(chat_id,
            f"💰 <b>Cost Tracker</b>\n"
            f"  Tokens in:  {s['tokens_prompt']:,}\n"
            f"  Tokens out: {s['tokens_response']:,}\n"
            f"  Reasoning:  {s['tokens_reasoning']:,}\n"
            f"  Total:      {s['tokens_total']:,}\n"
            f"  <b>Est. cost: ~${cost:.4f} USD</b>\n"
            f"  <i>(based on {Config.MODEL} pricing)</i>"
        )

    def _cmd_semantic(self, chat_id: int, args: str = "") -> None:
        if not args.strip():
            self._send(chat_id, "Usage: /semantic <query>")
            return
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        results = self._hub.agent.mem.semantic_search(args.strip(), top_k=5)
        if not results:
            self._send(chat_id, "No matching memories found.")
            return
        lines = [f"🧠 <b>Semantic search: '{args[:40]}'</b>\n"]
        for r in results:
            lines.append(
                f"  📌 <code>{r['key']}</code>  (score: {r['score']:.2f})\n"
                f"     {str(r['value'])[:200]}"
            )
        self._send(chat_id, "\n".join(lines))

    def _cmd_save(self, chat_id: int, _: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached.")
            return
        self._send(chat_id, "💾 Saving workspace to Drive…")
        result = self._hub.persistence.save(label="telegram")
        if result:
            self._send(chat_id, f"✅ Saved → <code>{result}</code>")
        else:
            self._send(chat_id, "❌ Save failed — check Drive is mounted.")

    # ─────────────────────────────────────────────────────────────────
    #  Inline keyboard helpers
    # ─────────────────────────────────────────────────────────────────

    def _keyboard(self, rows: list[list[tuple[str, str]]]) -> dict:
        return {
            "inline_keyboard": [
                [{"text": label, "callback_data": data} for label, data in row]
                for row in rows
            ]
        }

    def _send_kb(
        self, chat_id: int, text: str,
        rows: list[list[tuple[str, str]]], parse_mode: str = "HTML",
    ) -> dict:
        return self._api("sendMessage", json={
            "chat_id":      chat_id,
            "text":         text,
            "parse_mode":   parse_mode,
            "reply_markup": self._keyboard(rows),
        })

    def _edit_kb(
        self, chat_id: int, message_id: int, text: str,
        rows: list[list[tuple[str, str]]] | None = None,
        parse_mode: str = "HTML",
    ) -> None:
        payload: dict[str, Any] = {
            "chat_id": chat_id, "message_id": message_id,
            "text": text, "parse_mode": parse_mode,
        }
        payload["reply_markup"] = self._keyboard(rows) if rows is not None else {"inline_keyboard": []}
        self._api("editMessageText", json=payload)

    def _answer_cb(self, callback_query_id: str, text: str = "", alert: bool = False) -> None:
        self._api("answerCallbackQuery", json={
            "callback_query_id": callback_query_id,
            "text": text, "show_alert": alert,
        })

    # ─────────────────────────────────────────────────────────────────
    #  Callback query dispatcher
    # ─────────────────────────────────────────────────────────────────

    def _handle_callback(self, cb: dict) -> None:
        cq_id      = cb["id"]
        chat_id    = cb["message"]["chat"]["id"]
        message_id = cb["message"]["message_id"]
        data       = cb.get("data", "")

        if self._allowed and chat_id not in self._allowed:
            self._answer_cb(cq_id, "⛔ Unauthorised.", alert=True)
            return

        prefix, _, arg = data.partition(":")

        if prefix == "master":
            self._cb_master(cq_id, chat_id, message_id, arg)
        elif prefix == "queue":
            self._cb_queue(cq_id, chat_id, message_id, arg)
        elif prefix == "clear":
            self._cb_clear(cq_id, chat_id, message_id, arg)
        elif prefix == "logs":
            self._cb_logs(cq_id, chat_id, message_id, arg)
        elif prefix == "chat":
            self._cb_chat_mode(cq_id, chat_id, message_id, arg)
        elif prefix == "browse":
            self._cb_browse(cq_id, chat_id, message_id, arg)
        elif prefix == "express":
            self._cb_express(cq_id, chat_id, message_id, arg)
        elif prefix == "panel":
            self._answer_cb(cq_id)
            self._cmd_panel(chat_id, "")
        else:
            self._answer_cb(cq_id, "Unknown action.")

    # ─────────────────────────────────────────────────────────────────
    #  Button action handlers
    # ─────────────────────────────────────────────────────────────────

    def _cb_master(self, cq_id: str, chat_id: int, msg_id: int, arg: str) -> None:
        if arg == "stop":
            self._answer_cb(cq_id, "🛑 Stopping everything…", alert=True)
            self._edit_kb(chat_id, msg_id,
                          "🛑 <b>Master Stop triggered.</b>\nAll services shutting down…")
            def _do_stop():
                time.sleep(0.5)
                if self._hub:
                    self._hub.stop_all()
                self.stop()
            threading.Thread(target=_do_stop, daemon=True).start()

        elif arg == "stop_confirm":
            self._answer_cb(cq_id)
            self._edit_kb(chat_id, msg_id,
                "⚠️ <b>Confirm Master Stop?</b>\n"
                "This stops ALL services: queue, scheduler, heartbeat, autosave.",
                rows=[
                    [("🛑 Yes, stop everything", "master:stop")],
                    [("❌ Cancel",               "panel:")],
                ],
            )

        elif arg == "save_noop":
            self._answer_cb(cq_id, "💾 Saving…")
            if self._hub:
                result = self._hub.persistence.save(label="telegram-panel")
                ok = "✅ Saved." if result else "❌ Save failed."
            else:
                ok = "❌ No hub attached."
            self._edit_kb(chat_id, msg_id,
                f"💾 <b>{ok}</b>",
                rows=[[("🔙 Panel", "panel:")]],
            )

    def _cb_queue(self, cq_id: str, chat_id: int, msg_id: int, arg: str) -> None:
        if self._hub is None:
            self._answer_cb(cq_id, "No hub attached.", alert=True)
            return

        if arg == "pause":
            self._hub.queue._paused = True
            self._answer_cb(cq_id, "⏸ Queue paused.")
            self._edit_kb(chat_id, msg_id,
                "⏸ <b>Task Queue Paused</b>\nCurrent task finishes, then queue stops.",
                rows=[[("▶ Resume Queue", "queue:resume"),
                       ("🗑 Clear All",    "clear:all"),
                       ("🔙 Panel",        "panel:")]],
            )

        elif arg == "resume":
            self._hub.queue._paused = False
            if not (self._hub.queue._thread and self._hub.queue._thread.is_alive()):
                self._hub.queue.start()
            self._answer_cb(cq_id, "▶ Queue resumed.")
            self._edit_kb(chat_id, msg_id,
                "▶ <b>Task Queue Resumed</b>",
                rows=[[("⏸ Pause Queue", "queue:pause"),
                       ("🗑 Clear All",   "clear:all"),
                       ("🔙 Panel",       "panel:")]],
            )

        elif arg == "show":
            self._answer_cb(cq_id)
            tq = self._hub.queue
            with tq._lock:
                pending = list(tq._q.queue)
            if not pending:
                self._edit_kb(chat_id, msg_id,
                    "📋 <b>Queue is empty.</b>",
                    rows=[[("🔙 Panel", "panel:")]],
                )
                return
            rows: list[list[tuple[str, str]]] = []
            lines = [f"📋 <b>Queue  —  {len(pending)} pending</b>\n"]
            for i, item in enumerate(pending[:8]):
                short = item["task"][:40]
                lines.append(f"  {i+1}. {short}")
                rows.append([(f"❌ Remove #{i+1}: {short[:25]}", f"clear:task:{i}")])
            rows.append([("🗑 Clear All", "clear:all"), ("🔙 Panel", "panel:")])
            self._edit_kb(chat_id, msg_id, "\n".join(lines), rows=rows)

    def _cb_clear(self, cq_id: str, chat_id: int, msg_id: int, arg: str) -> None:
        if self._hub is None:
            self._answer_cb(cq_id, "No hub attached.", alert=True)
            return

        if arg == "all":
            self._hub.queue.clear()
            self._answer_cb(cq_id, "🗑 All queued tasks cleared.")
            self._edit_kb(chat_id, msg_id,
                "🗑 <b>Queue cleared.</b> All pending tasks removed.",
                rows=[[("🔙 Panel", "panel:")]],
            )

        elif arg.startswith("task:"):
            idx = int(arg.split(":")[1])
            with self._hub.queue._lock:
                pending = list(self._hub.queue._q.queue)
                if idx < len(pending):
                    removed = pending.pop(idx)
                    self._hub.queue._q.queue.clear()
                    for item in pending:
                        self._hub.queue._q.put_nowait(item)
                    self._answer_cb(cq_id, f"❌ Removed: {removed['task'][:40]}")
                    self._cb_queue(cq_id, chat_id, msg_id, "show")
                else:
                    self._answer_cb(cq_id, "Task not found (may have already run).", alert=True)

    def _cb_logs(self, cq_id: str, chat_id: int, msg_id: int, arg: str) -> None:
        if arg == "on":
            self._start_log_stream(chat_id)
            self._answer_cb(cq_id, "📡 Log streaming ON")
            self._edit_kb(chat_id, msg_id,
                "📡 <b>Log streaming ON</b>\nNew log lines pushed every 5s.",
                rows=[
                    [("⚠ Warnings only", "logs:level:warning"),
                     ("❌ Errors only",   "logs:level:error")],
                    [("⚙ All levels",    "logs:level:all")],
                    [("⏹ Stop streaming", "logs:off"), ("🔙 Panel", "panel:")],
                ],
            )
        elif arg == "off":
            self._stop_log_stream()
            self._answer_cb(cq_id, "⚪ Log streaming OFF")
            self._edit_kb(chat_id, msg_id,
                "⚪ <b>Log streaming stopped.</b>",
                rows=[[("📡 Start streaming", "logs:on"), ("🔙 Panel", "panel:")]],
            )
        elif arg.startswith("level:"):
            lvl = arg.split(":")[1]
            self._log_level_filter = None if lvl == "all" else lvl
            self._answer_cb(cq_id, f"Level filter → {lvl}")
            self._edit_kb(chat_id, msg_id,
                f"📡 <b>Log streaming ON</b>  —  level: <code>{lvl}</code>",
                rows=[
                    [("⚠ Warnings only", "logs:level:warning"),
                     ("❌ Errors only",   "logs:level:error"),
                     ("⚙ All",           "logs:level:all")],
                    [("⏹ Stop streaming", "logs:off"), ("🔙 Panel", "panel:")],
                ],
            )

    def _cb_chat_mode(self, cq_id: str, chat_id: int, msg_id: int, arg: str) -> None:
        if arg == "on":
            self._chat_mode_ids.add(chat_id)
            self._answer_cb(cq_id, "💬 Direct chat mode ON")
            self._edit_kb(chat_id, msg_id,
                "💬 <b>Direct Chat Mode ON</b>\n"
                "Every message you send now goes to the express lane — "
                "answered in parallel without touching the queue.",
                rows=[[("⏹ Exit chat mode", "chat:off"), ("🔙 Panel", "panel:")]],
            )
        elif arg == "off":
            self._chat_mode_ids.discard(chat_id)
            self._answer_cb(cq_id, "💬 Direct chat mode OFF")
            self._edit_kb(chat_id, msg_id,
                "💬 <b>Direct Chat Mode OFF</b>\nMessages now require /task or /express prefix.",
                rows=[[("💬 Enable chat mode", "chat:on"), ("🔙 Panel", "panel:")]],
            )

    def _cb_browse(self, cq_id: str, chat_id: int, msg_id: int, arg: str) -> None:
        self._answer_cb(cq_id)
        parts  = arg.split(":")
        action = parts[0] if parts else ""

        if action == "root":
            self._browse_render(chat_id, msg_id, Config.FILES_DIR.resolve(), page=0)
        elif action == "dir":
            if len(parts) < 3:
                return
            try:
                key, page = int(parts[1]), int(parts[2])
            except ValueError:
                return
            path = self._reg_get(key)
            if path is None:
                self._send(chat_id, "⚠ Navigation expired. Reopening from root.")
                self._browse_render(chat_id, None, Config.FILES_DIR.resolve(), page=0)
                return
            self._browse_render(chat_id, msg_id, path, page=page)
        elif action == "get":
            if len(parts) < 2:
                return
            try:
                key = int(parts[1])
            except ValueError:
                return
            path = self._reg_get(key)
            if path is None:
                self._send(chat_id, "⚠ File reference expired. Use /browse to navigate again.")
                return
            if not path.exists() or not path.is_file():
                self._send(chat_id, f"❌ File no longer exists: <code>{path.name}</code>")
                return
            try:
                path.resolve().relative_to(Config.FILES_DIR.resolve())
            except ValueError:
                self._send(chat_id, "❌ File is outside the workspace.")
                return
            self._deliver_file(chat_id, path)
        else:
            self._answer_cb(cq_id, "Unknown browse action.", alert=True)

    def _cb_express(self, cq_id: str, chat_id: int, msg_id: int, arg: str) -> None:
        """Handle express: button callbacks from the panel."""
        self._answer_cb(cq_id)
        if arg == "status":
            self._cmd_express_status(chat_id, "")
        elif arg == "help":
            self._edit_kb(chat_id, msg_id,
                "⚡ <b>Express Lane</b>\n\n"
                "Runs quick questions <b>in parallel</b> — the queue is never blocked.\n\n"
                "Type any of these:\n"
                "  /express How many files in the workspace?\n"
                "  /e What's in the last log entry?\n"
                "  /e Summarise MEMORY.md in one sentence\n\n"
                "<i>Results arrive as a follow-up message (usually 1-5 seconds).</i>",
                rows=[[("⚡ Recent results", "express:status"),
                       ("🔙 Panel",          "panel:")]],
            )

    # ─────────────────────────────────────────────────────────────────
    #  Control panel  (/panel)
    # ─────────────────────────────────────────────────────────────────

    def _cmd_panel(self, chat_id: int, _: str = "") -> None:
        if self._hub is None:
            self._send(chat_id, "❌ No hub attached. Run autorun() first.")
            return

        def _dot(t): return "🟢" if t and t.is_alive() else "🔴"
        paused     = getattr(self._hub.queue, "_paused", False)
        qsize      = self._hub.queue._q.qsize()
        chat_on    = chat_id in self._chat_mode_ids
        log_on     = self._log_streaming
        express_ok = hasattr(self._hub, "express") and self._hub.express._pool is not None
        q_running  = bool(self._hub.queue._current)

        status_lines = [
            "🎛 <b>Control Panel</b>",
            "",
            f"  {_dot(self._hub.queue._thread)}  task_queue"
            + ("  ⏸ paused" if paused else f"  ({qsize} pending)")
            + ("  <b>▶ running</b>" if q_running else ""),
            f"  {'🟢' if express_ok else '🔴'}  express lane"
            + (f"  <i>({getattr(self._hub.express, 'MAX_WORKERS', '?')} workers)</i>"
               if express_ok else "  <i>(stopped)</i>"),
            f"  {_dot(self._hub.scheduler._thread)}  scheduler",
            f"  {_dot(self._hub.need_poller._thread)}  need_poller",
            f"  {_dot(self._hub.watchdog._thread)}  watchdog",
            f"  {'📡' if log_on else '⚪'}  log stream",
            f"  {'💬' if chat_on else '⚪'}  direct chat",
        ]

        rows: list[list[tuple[str, str]]] = [
            # Row 1 — Queue controls
            [
                ("⏸ Pause Queue",  "queue:pause") if not paused else ("▶ Resume Queue", "queue:resume"),
                ("📋 View Queue",   "queue:show"),
                ("🗑 Clear All",    "clear:all"),
            ],
            # Row 2 — Express lane
            [
                ("⚡ Express help", "express:help"),
                ("📊 Express results", "express:status"),
            ],
            # Row 3 — Logs + Files
            [
                ("📡 Logs ON",  "logs:on") if not log_on else ("⏹ Logs OFF", "logs:off"),
                ("🗂 Browse Files", "browse:root:0"),
            ],
            # Row 4 — Chat mode
            [
                ("💬 Chat Mode ON",  "chat:on") if not chat_on else ("💬 Chat Mode OFF", "chat:off"),
            ],
            # Row 5 — Save
            [("💾 Save to Drive", "master:save_noop")],
            # Row 6 — Danger
            [("🛑 MASTER STOP", "master:stop_confirm")],
        ]
        self._send_kb(chat_id, "\n".join(status_lines), rows)

    # ─────────────────────────────────────────────────────────────────
    #  BotFather command registration
    # ─────────────────────────────────────────────────────────────────

    _BOT_COMMANDS: list[dict[str, str]] = [
        {"command": "panel",          "description": "🎛 Interactive control panel with buttons"},
        {"command": "status",         "description": "📊 Full hub & service status"},
        # Tasks
        {"command": "task",           "description": "🤖 Queue a heavy task  e.g. /task write a readme"},
        {"command": "t",              "description": "📥 Shorthand for /task"},
        # Express lane
        {"command": "express",        "description": "⚡ Quick parallel question — skips the queue  e.g. /e how many files?"},
        {"command": "e",              "description": "⚡ Shorthand for /express"},
        {"command": "express_status", "description": "📊 Show recent express lane results"},
        # Queue
        {"command": "queue",          "description": "📋 View task queue (pending / running / done)"},
        # Logs
        {"command": "logs",           "description": "📄 Last N log lines  e.g. /logs 50"},
        {"command": "logs_on",        "description": "📡 Start streaming live logs here"},
        {"command": "logs_off",       "description": "⏹ Stop log streaming"},
        {"command": "loglevel",       "description": "🔍 Set stream filter  e.g. /loglevel warning"},
        {"command": "logfilter",      "description": "🔎 Filter by text  e.g. /logfilter tool"},
        # Memory & needs
        {"command": "memory",         "description": "🧠 Show MEMORY.md contents"},
        {"command": "needs",          "description": "🔔 Show pending agent needs"},
        # Files
        {"command": "browse",         "description": "🗂 Interactive file browser — tap to navigate & download"},
        {"command": "files",          "description": "📁 List workspace files  e.g. /files src/"},
        {"command": "getfile",        "description": "📤 Download a file  e.g. /getfile reports/x.md"},
        {"command": "uploads",        "description": "📎 List files uploaded via Telegram  e.g. /uploads 30"},
        # System
        {"command": "save",           "description": "💾 Save workspace to Google Drive now"},
        {"command": "cancel",         "description": "🛑 Cancel the currently-running queue task"},
        {"command": "export",         "description": "📤 Export conversation  e.g. /export md"},
        {"command": "diff",           "description": "📊 Workspace file changes since session start"},
        {"command": "cost",           "description": "💰 Token usage and estimated USD cost"},
        {"command": "semantic",       "description": "🧠 Semantic memory search  e.g. /semantic auth flow"},
        {"command": "id",             "description": "🪪 Show your Telegram chat_id"},
        {"command": "help",           "description": "❓ Show all commands"},
    ]

    def _register_commands(self) -> None:
        resp = self._api("setMyCommands", json={"commands": self._BOT_COMMANDS})
        if resp.get("ok"):
            logger.info(f"Registered {len(self._BOT_COMMANDS)} bot commands with Telegram")
            print(f"  ✓ {len(self._BOT_COMMANDS)} commands registered with Telegram")
        else:
            logger.warning(f"setMyCommands failed: {resp}")
            print(f"  ⚠  Could not register commands: {resp.get('description', resp)}")

    def _poll_loop(self) -> None:
        logger.info("Telegram bot polling started")
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self._MAX_DISPATCH_WORKERS,
            thread_name_prefix="tg-dispatch",
        ) as pool:
            while not self._stop_ev.is_set():
                try:
                    resp = self._api(
                        "getUpdates",
                        json={
                            "offset":          self._update_offset,
                            "timeout":         self._POLL_TIMEOUT,
                            # FIX: include document/photo/audio/video in allowed_updates
                            "allowed_updates": ["message", "callback_query"],
                        },
                    )
                    if not resp.get("ok"):
                        time.sleep(5)
                        continue
                    for update in resp.get("result", []):
                        self._update_offset = update["update_id"] + 1
                        msg = update.get("message", {})
                        if msg:
                            chat_id = msg.get("chat", {}).get("id")
                            text    = msg.get("text", "").strip()

                            # ── Document / media upload ──────────────
                            # Telegram puts file info in different fields depending
                            # on type. We handle all of them uniformly.
                            file_obj, media_type = None, None
                            if msg.get("document"):
                                file_obj   = msg["document"]
                                media_type = "document"
                            elif msg.get("audio"):
                                file_obj   = msg["audio"]
                                media_type = "audio"
                            elif msg.get("video"):
                                file_obj   = msg["video"]
                                media_type = "video"
                            elif msg.get("photo"):
                                # photo is a list of resolutions — take the largest
                                file_obj   = msg["photo"][-1]
                                media_type = "photo"
                            elif msg.get("voice"):
                                file_obj   = msg["voice"]
                                media_type = "voice"

                            if file_obj and chat_id:
                                caption = (msg.get("caption") or "").strip()
                                pool.submit(
                                    self._handle_upload,
                                    chat_id, file_obj, media_type, caption,
                                )
                            elif chat_id and text:
                                pool.submit(self._dispatch, chat_id, text)

                        cb = update.get("callback_query")
                        if cb:
                            pool.submit(self._handle_callback, cb)
                except Exception as e:
                    logger.warning(f"Telegram poll error: {e}")
                    time.sleep(5)

    # ─────────────────────────────────────────────────────────────────
    #  Lifecycle
    # ─────────────────────────────────────────────────────────────────

    def start(self) -> None:
        if self._poll_thread and self._poll_thread.is_alive():
            print("ℹ  Telegram bot already running.")
            return
        self._stop_ev.clear()
        self._send_thread = threading.Thread(
            target=self._send_worker, daemon=True, name="tg-send"
        )
        self._poll_thread = threading.Thread(
            target=self._poll_loop, daemon=True, name="tg-poll"
        )
        self._send_thread.start()
        self._poll_thread.start()
        self._register_commands()
        print(f"🤖 @{self._bot_name} is online and listening.")
        print(f"   Open Telegram and send /help to @{self._bot_name}")
        self._broadcast_startup()

    def _broadcast_startup(self) -> None:
        if not self._allowed:
            return
        time.sleep(0.5)
        ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        express_ok = self._hub and hasattr(self._hub, "express") and self._hub.express._pool is not None
        lines = [
            "🟢 <b>Agent Bot Online</b>",
            f"  Time:      <code>{ts}</code>",
            f"  Model:     <code>{Config.MODEL}</code>",
            f"  Workspace: <code>{Config.BASE_DIR}</code>",
            "",
            "<b>Services running:</b>",
            f"  {'🟢' if self._hub and self._hub.queue._thread and self._hub.queue._thread.is_alive() else '⚪'} task_queue",
            f"  {'🟢' if express_ok else '⚪'} express_lane",
            f"  {'🟢' if self._hub and self._hub.scheduler._thread and self._hub.scheduler._thread.is_alive() else '⚪'} scheduler",
            f"  {'🟢' if self._hub and self._hub.need_poller._thread and self._hub.need_poller._thread.is_alive() else '⚪'} need_poller",
            f"  {'🟢' if self._hub and self._hub.watchdog._thread and self._hub.watchdog._thread.is_alive() else '⚪'} watchdog",
            f"  {'🟢' if self._hub and self._hub.persistence._autosave_thread and self._hub.persistence._autosave_thread.is_alive() else '⚪'} autosave",
            "",
            "⚡ <b>Quick tip:</b> While a long task runs, use /express or /e",
            "   to ask questions in parallel without waiting.",
            "",
            "📎 <b>Upload tip:</b> Send any file directly to this chat",
            "   to save it to your workspace automatically.",
            "",
            "Send /help for all commands.",
        ]
        for cid in self._allowed:
            self._send(cid, "\n".join(lines))

    def stop(self) -> None:
        self._broadcast_shutdown()
        time.sleep(2)
        self._stop_ev.set()
        self._stop_log_stream()
        print(f"⏹  @{self._bot_name} stopped.")

    def _broadcast_shutdown(self) -> None:
        if not self._allowed:
            return
        ts     = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        uptime = ""
        if self._hub:
            s      = self._hub.agent.counter.summary()
            uptime = (
                f"\n<b>Session summary:</b>\n"
                f"  Requests: {s['total_requests']} (✓{s['success']} ✗{s['errors']})\n"
                f"  Tokens:   {s['tokens_total']:,}\n"
                f"  Uptime:   {s['uptime_seconds']}s"
            )
        msg = (
            f"🔴 <b>Agent Bot Offline</b>\n"
            f"  Time: <code>{ts}</code>"
            f"{uptime}\n\n"
            f"<i>Workspace saved to Drive before shutdown.</i>"
        )
        for cid in self._allowed:
            self._send(cid, msg)

    def notify(self, text: str) -> None:
        if not self._allowed:
            print("⚠  No whitelisted chat IDs — cannot send notification.")
            return
        for cid in self._allowed:
            self._send(cid, text)

    def notify_service(self, service: str, state: str, detail: str = "") -> None:
        icons = {"started": "🟢", "stopped": "⚪", "restarted": "🔄", "error": "🔴"}
        icon  = icons.get(state, "•")
        ts    = datetime.now().strftime("%H:%M:%S")
        msg   = f"{icon} <b>{service}</b> {state}"
        if detail:
            msg += f"\n  <i>{detail}</i>"
        msg += f"\n  <code>{ts}</code>"
        self.notify(msg)

    @property
    def running(self) -> bool:
        return bool(self._poll_thread and self._poll_thread.is_alive())


# ══════════════════════════════════════════════════════════════════════════════
#  Convenience launcher
# ══════════════════════════════════════════════════════════════════════════════

def start_telegram(
    hub:   "AutomationHub | None" = None,
    token: str | None             = None,
) -> TelegramBot:
    """
    Create and start the Telegram bot in one call.

    Usage:
        tg = start_telegram(hub)
        tg = start_telegram()
        tg.notify("Task finished!")
        tg.stop()
    """
    tg = TelegramBot(hub=hub, token=token)
    tg.start()

    # ── Patch Watchdog to notify on restarts ──────────────────────────
    if hub is not None and not getattr(hub.watchdog, "_tg_patched", False):
        _orig_restart = Watchdog._check_and_restart

        def _notifying_restart(name, thread, restart):
            if thread is not None and not thread.is_alive():
                tg.notify_service(name, "restarted", "thread died — watchdog recovered it")
            _orig_restart(name, thread, restart)

        Watchdog._check_and_restart = _notifying_restart
        hub.watchdog._tg_patched    = True

    # ── Patch TaskQueue worker to notify on task completion ───────────
    if hub is not None and not getattr(TaskQueue._worker, "_tg_patched", False):
        def _patched_q_worker(self_q):
            while not self_q._stop.is_set():
                try:
                    item = self_q._q.get(timeout=2)
                except __import__("queue").Empty:
                    continue
                self_q._current       = item["task"]
                self_q._cancel_current = False
                item["started"]       = datetime.now().isoformat()
                item["status"]        = "running"
                tg.notify(f"▶ <b>Task started</b>\n<i>{item['task'][:120]}</i>")
                try:
                    result = self_q._agent.run(item["task"])
                    if self_q._cancel_current:
                        item["status"] = "cancelled"
                        item["result"] = result[:500] if result else ""
                        tg.notify(f"⏹ <b>Task cancelled</b>\n<i>{item['task'][:80]}</i>")
                    else:
                        item["result"] = result[:500]
                        item["status"] = "done"
                        truncated = f"\n<i>…{len(result)-800:,} chars truncated</i>" if len(result) > 800 else ""
                        tg.notify(f"✅ <b>Task done</b>\n<i>{item['task'][:80]}</i>\n\n{result[:800]}{truncated}")
                except Exception as e:
                    item["error"]  = str(e)
                    item["status"] = "error"
                    tg.notify(f"❌ <b>Task failed</b>\n<i>{item['task'][:80]}</i>\n<code>{e}</code>")
                finally:
                    item["finished"] = datetime.now().isoformat()
                    with self_q._lock:
                        self_q._done.append(item)
                        if len(self_q._done) > 100:
                            self_q._done = self_q._done[-100:]
                    self_q._current        = None
                    self_q._cancel_current = False
                    self_q._q.task_done()

        TaskQueue._worker             = _patched_q_worker
        TaskQueue._worker._tg_patched = True
        hub.queue._tg_patched         = True

    return tg


print("✓ Telegram bot cell ready")
print()
print("  Setup:")
print("    1. Create bot via @BotFather → copy token")
print("    2. Add to Colab Secrets:")
print("         TELEGRAM_BOT_TOKEN = 123456:ABC-xxx")
print("         TELEGRAM_CHAT_ID   = 987654321  (send /id to bot to find yours)")
print()
print("  Launch:")
print("    tg = start_telegram(hub)       # hub from autorun()")
print("    tg = start_telegram()          # without hub")
print()
print("  From code:")
print("    tg.notify('✅ Task done!')")
print("    tg.stop()")
print()
print("  File browser:")
print("    /browse              — open workspace root")
print("    /browse reports/     — open a subfolder directly")
print("    Tap 📁 to drill in  ·  Tap 📄 to download  ·  Tap ⬆ to go up")
print()
print("  ⚡ Express lane (parallel — never blocks the queue):")
print("    /express How many .py files are there?")
print("    /e Summarise the last 3 log lines")
print("    /express_status    — see recent express results")
print()
print("  📎 Document upload (Telegram → workspace):")
print("    Send any file to the bot → saved to workspace/telegram_uploads/")
print("    Add caption 'reports/jan.csv' → saved to telegram_uploads/reports/jan.csv")
print("    /uploads           — list recently uploaded files")
print("    /uploads 50        — show last 50 uploads")

✓ Telegram bot cell ready

  Setup:
    1. Create bot via @BotFather → copy token
    2. Add to Colab Secrets:
         TELEGRAM_BOT_TOKEN = 123456:ABC-xxx
         TELEGRAM_CHAT_ID   = 987654321  (send /id to bot to find yours)

  Launch:
    tg = start_telegram(hub)       # hub from autorun()
    tg = start_telegram()          # without hub

  From code:
    tg.notify('✅ Task done!')
    tg.stop()

  File browser:
    /browse              — open workspace root
    /browse reports/     — open a subfolder directly
    Tap 📁 to drill in  ·  Tap 📄 to download  ·  Tap ⬆ to go up

  ⚡ Express lane (parallel — never blocks the queue):
    /express How many .py files are there?
    /e Summarise the last 3 log lines
    /express_status    — see recent express results

  📎 Document upload (Telegram → workspace):
    Send any file to the bot → saved to workspace/telegram_uploads/
    Add caption 'reports/jan.csv' → saved to telegram_uploads/reports/jan.csv
    /uploads           — list recently

In [45]:
# @title
# ══════════════════════════════════════════════════════════════════════════════
#  ENHANCED TELEGRAM BOT FEATURES
# ══════════════════════════════════════════════════════════════════════════════

# --- Additional command registration ---
_EXTRA_COMMANDS = [
    {"command": "schedule", "description": "⏱ Schedule recurring task: /schedule every 1h 'task'"},
    {"command": "stats", "description": "📊 Detailed usage statistics"},
    {"command": "quick", "description": "⚡ Quick action buttons panel"},
    {"command": "find", "description": "🔍 Search files: /find pattern or /find -c 'text'"},
    {"command": "config", "description": "⚙️ View/modify configuration"},
    {"command": "template", "description": "📋 Use task template: /template list|name"},
    {"command": "memory_edit", "description": "📝 Edit memory: /memory_edit KEY VALUE"},
    {"command": "git", "description": "🔀 Git operations: /git status|commit|push|log"},
    {"command": "notify", "description": "🔔 Notifications: /notify on|off|status"},
    {"command": "export_all", "description": "📤 Export everything: /export_all [md|json]"},
]

# Register commands if TelegramBot exists
if 'TelegramBot' in globals() and hasattr(TelegramBot, '_BOT_COMMANDS'):
    TelegramBot._BOT_COMMANDS.extend(_EXTRA_COMMANDS)
    print(f"✓ Registered {len(_EXTRA_COMMANDS)} new Telegram commands")

# --- New command handler implementations ---

def _cmd_schedule(self, chat_id: int, args: str) -> None:
    """Schedule a recurring task.
    Usage: /schedule every 2h "task"
           /schedule daily at 09:00 "task"
    """
    import shlex, html
    if not args:
        self._send(chat_id, "Usage:\n• /schedule every <interval> <task>\n• /schedule daily at <HH:MM> <task>")
        return
    try:
        parts = shlex.split(args)
        if parts[0] == 'every' and len(parts) >= 3:
            interval_str = parts[1]
            m = __import__('re').match(r'(\d+)([smhd])$', interval_str)
            if not m:
                raise ValueError('Invalid interval. Use: 30m, 2h, 1d')
            num, unit = int(m.group(1)), m.group(2)
            seconds = {'s':1, 'm':60, 'h':3600, 'd':86400}[unit] * num
            task = ' '.join(parts[2:])
            if self._hub:
                self._hub.every(seconds, task)
                self._send(chat_id, f"✅ Scheduled every {interval_str}:\n<code>{html.escape(task)}</code>")
            else:
                self._send(chat_id, "❌ Hub not available")
        elif parts[0] == 'daily' and len(parts) >= 4 and parts[1] == 'at':
            time_str = parts[2]
            task = ' '.join(parts[3:])
            if not __import__('re').match(r'^\d{2}:\d{2}$', time_str):
                raise ValueError('Time must be HH:MM')
            if self._hub:
                self._hub.at(time_str, task)
                self._send(chat_id, f"✅ Scheduled daily at {time_str}:\n<code>{html.escape(task)}</code>")
            else:
                self._send(chat_id, "❌ Hub not available")
        else:
            raise ValueError('Invalid syntax')
    except Exception as e:
        self._send(chat_id, f"❌ {html.escape(str(e))}")

def _cmd_stats(self, chat_id: int, args: str) -> None:
    """Show detailed usage statistics."""
    import html
    if not self._hub or not hasattr(self._hub, 'agent'):
        self._send(chat_id, "❌ Hub/agent not available")
        return
    try:
        s = self._hub.agent.counter.summary()
        uptime = s.get('uptime_seconds', 0)
        h, rem = divmod(uptime, 3600)
        m, s = divmod(rem, 60)
        lines = [
            "📊 <b>Session Statistics</b>", "",
            f"⏱ Uptime: {int(h)}h {int(m)}m {int(s)}s",
            f"🔢 Requests: {s.get('total_requests',0):,} (✓{s.get('success',0):,} ✗{s.get('errors',0):,})",
            f"🔤 Tokens: {s.get('tokens_total',0):,} (in:{s.get('tokens_in',0):,} out:{s.get('tokens_out',0):,})",
            f"💰 Cost: ${s.get('cost_usd',0):.4f}", "",
            f"📋 Queue: {self._hub.queue._q.qsize()} pending, {len(self._hub.queue._done)} done",
        ]
        self._send(chat_id, '\n'.join(lines))
    except Exception as e:
        self._send(chat_id, f"❌ Error: {html.escape(str(e))}")

def _cmd_quick(self, chat_id: int, args: str) -> None:
    """Show quick action buttons."""
    rows = [
        [("📊 Stats", "quick:stats"), ("📋 Queue", "queue:show"), ("📁 Browse", "browse:root:0")],
        [("⚡ Express", "express:quick"), ("📄 Logs", "logs:quick"), ("💾 Save", "master:save_noop")],
        [("🔄 Restart Hub", "master:restart_confirm"), ("⏹ Stop Bot", "master:stop_confirm")],
    ]
    self._send_kb(chat_id, "⚡ <b>Quick Actions</b>", rows)

def _cmd_find(self, chat_id: int, args: str) -> None:
    """Search files by name or content."""
    import shlex, fnmatch, html
    if not args:
        self._send(chat_id, "Usage:\n/find <pattern> - search filenames\n/find -c 'text' [in:dir] - search content")
        return
    try:
        parts = shlex.split(args)
        search_content = False
        search_dir = Config.FILES_DIR
        pattern = ""

        for i, p in enumerate(parts):
            if p == '-c' and i+1 < len(parts):
                search_content = True
                pattern = parts[i+1]
            elif p.startswith('in:'):
                subdir = p[3:]
                search_dir = Config.FILES_DIR / subdir
            elif not p.startswith('-') and not pattern:
                pattern = p

        if not pattern:
            self._send(chat_id, "❌ No pattern")
            return

        if search_content:
            results = []
            for fp in search_dir.rglob('*'):
                if fp.is_file() and fp.suffix in ['.py','.md','.txt','.json']:
                    try:
                        if pattern.lower() in fp.read_text(encoding='utf-8', errors='ignore').lower():
                            results.append(str(fp.relative_to(Config.FILES_DIR)))
                    except: pass
            if results:
                msg = f"🔍 Found {len(results)} files containing '{pattern}':\n" + '\n'.join(f"• {r}" for r in results[:15])
                if len(results) > 15: msg += f"\n...and {len(results)-15} more"
                self._send(chat_id, msg[:4000])
            else:
                self._send(chat_id, "No matches")
        else:
            matches = [str(fp.relative_to(Config.FILES_DIR)) for fp in search_dir.rglob('*') if fp.is_file() and fnmatch.fnmatch(fp.name, pattern)]
            if matches:
                msg = f"🔍 Found {len(matches)} files matching '{pattern}':\n" + '\n'.join(f"• {m}" for m in matches[:15])
                if len(matches) > 15: msg += f"\n...and {len(matches)-15} more"
                self._send(chat_id, msg[:4000])
            else:
                self._send(chat_id, "No matches")
    except Exception as e:
        self._send(chat_id, f"❌ Error: {html.escape(str(e))}")

def _cmd_config(self, chat_id: int, args: str) -> None:
    """View current configuration."""
    import html
    if not args:
        lines = ["⚙️ <b>Configuration</b>", ""]
        for attr in ['MODEL','FAST_MODEL','FALLBACK_MODEL','BASE_URL','MAX_TOKENS','CODE_TIMEOUT','SHELL_TIMEOUT']:
            val = getattr(Config, attr, 'N/A')
            lines.append(f"<code>{attr}</code> = <code>{val}</code>")
        self._send(chat_id, '\n'.join(lines))
    else:
        self._send(chat_id, "Config modification requires environment variable changes or restart.")

def _cmd_template(self, chat_id: int, args: str) -> None:
    """Use a task template."""
    import html
    templates = {
        "daily_report": "Generate daily summary from logs and save to reports/",
        "code_review": "Review all Python files for quality and improvements",
        "test_gen": "Generate unit tests for the most recently modified Python file",
        "backup": "Create compressed backup of workspace",
        "cleanup": "Clean temporary files and logs older than 7 days",
    }
    name = args.strip() if args else ""
    if not name or name == "list":
        lines = ["📋 <b>Templates</b>", ""]
        for n, d in templates.items():
            lines.append(f"• <code>{n}</code>: {d}")
        lines.append("\nUse: /template <name>")
        self._send(chat_id, '\n'.join(lines))
        return
    if name not in templates:
        self._send(chat_id, f"❌ Unknown template: {name}")
        return
    if self._hub:
        task_id = self._hub.submit(templates[name])
        self._send(chat_id, f"✅ Queued <b>{name}</b> as task {task_id}")
    else:
        self._send(chat_id, "❌ Hub not available")

def _cmd_memory_edit(self, chat_id: int, args: str) -> None:
    """Edit a memory entry."""
    import shlex, html
    try:
        parts = shlex.split(args)
        if len(parts) < 2:
            raise ValueError("Need key and value")
        key, value = parts[0], ' '.join(parts[1:])
        memory_save(key, value)
        self._send(chat_id, f"✅ Saved: <code>{html.escape(key)}</code> = <code>{html.escape(value)}</code>")
    except Exception as e:
        self._send(chat_id, f"❌ {html.escape(str(e))}")

def _cmd_git(self, chat_id: int, args: str) -> None:
    """Git operations."""
    import html
    if not args:
        self._send(chat_id, "Commands: status, diff, commit, push, log\nExample: /git status")
        return
    cmd = args.strip().split()[0]
    mapping = {
        'status': 'git status',
        'diff': 'git diff',
        'log': 'git log --oneline -10'
    }
    if cmd == 'commit':
        msg = args[len('commit'):].strip()
        if not msg.startswith('-m'):
            msg = '-m "' + msg + '"'
        full = f"git commit -a {msg}"
    elif cmd == 'push':
        full = 'git push'
    else:
        full = mapping.get(cmd)
    if not full:
        self._send(chat_id, f"❌ Unknown: {cmd}")
        return
    try:
        res = shell_run(full, cwd=str(Config.BASE_DIR))
        out = (res.stdout or res.stderr or 'No output')[:3000]
        self._send(chat_id, f"<code>{html.escape(full)}</code>\n\n<pre>{html.escape(out)}</pre>")
    except Exception as e:
        self._send(chat_id, f"❌ {html.escape(str(e))}")

def _cmd_notify(self, chat_id: int, args: str) -> None:
    """Set notification preferences."""
    pref = args.strip().lower() if args else "status"
    if pref == 'on':
        if chat_id not in self._allowed:
            self._allowed.append(chat_id)
            self._send(chat_id, "✅ Notifications ON")
        else:
            self._send(chat_id, "✓ Already ON")
    elif pref == 'off':
        if chat_id in self._allowed:
            self._allowed.remove(chat_id)
            self._send(chat_id, "✅ Notifications OFF")
        else:
            self._send(chat_id, "✓ Already OFF")
    else:
        status = 'ENABLED' if chat_id in self._allowed else 'DISABLED'
        self._send(chat_id, f"🔔 Notifications: {status}\nCommands: /notify on|off")

def _cmd_export_all(self, chat_id: int, args: str) -> None:
    """Export conversation."""
    import html
    fmt = args.strip().lower() if args else 'md'
    if fmt not in ('md','json'):
        self._send(chat_id, "Format: md or json")
        return
    self._send(chat_id, "📤 Export initiated (placeholder)")

# --- Register the new handlers ---
if 'TelegramBot' in globals():
    TelegramBot._cmd_schedule = _cmd_schedule
    TelegramBot._cmd_stats = _cmd_stats
    TelegramBot._cmd_quick = _cmd_quick
    TelegramBot._cmd_find = _cmd_find
    TelegramBot._cmd_config = _cmd_config
    TelegramBot._cmd_template = _cmd_template
    TelegramBot._cmd_memory_edit = _cmd_memory_edit
    TelegramBot._cmd_git = _cmd_git
    TelegramBot._cmd_notify = _cmd_notify
    TelegramBot._cmd_export_all = _cmd_export_all
    print("✓ Attached new command handlers to TelegramBot")
else:
    # Delay registration - these will be patched when TelegramBot is defined
    print("⚠ TelegramBot not yet defined; handlers will be registered later")

print("✓ Enhanced Telegram features loaded")


✓ Registered 10 new Telegram commands
✓ Attached new command handlers to TelegramBot
✓ Enhanced Telegram features loaded


## Telegram Service Start

In [46]:
hub = autorun()
hub.status()
tg = start_telegram(hub)

════════════════════════════════════════════════════════════
  🤖  FULL AUTOMATION STARTING
════════════════════════════════════════════════════════════
✓ Drive already mounted.
✅ Restored from latest.zip  (14850 KB)  saved at 2026-03-24 00:07

  Initialising agent…
[SYSTEM   ] Agent ready | model=stepfun-ai/step-3.5-flash | workspace=/content/agent | stream=False
⏱  Auto-save started — saving every 15 min to Drive.
✓ Shutdown hook registered — workspace saves on interrupt/termination.
▶  Task queue worker started.
⚡  Express lane started (3 parallel workers).
▶  Scheduler started.
👀  Need poller started (every 10 min).
🐕  Watchdog started (checks every 60s).

✅ All automation components running.
📥 Queued [1 pending]: Check heartbeat, review Need.md, summarise MEMORY.md

════════════════════════════════════════════════════════════
  ✅  FULL AUTOMATION RUNNING
     hub.status()                         — live dashboard
     hub.submit('task...')                — queue a heavy task
     hu

In [47]:
# watch_logs()                       # everything, streams as it arrives
# watch_logs(level="warning")        # warnings + errors only
# watch_logs(text="parallel")        # only lines containing "parallel"hub.submit("Refactor all Python files in /content/agent/files/src/")
#hub.submit("Write tests for utils.py")
#hub.queue.status()   # see what's pending/running/done
#hub.every(60, "Summarise today's log and update MEMORY.md")
#hub.at("18:00", "Generate end-of-day report and save to files/reports/")
#hub.scheduler.status()

## CELL 15: Main Execution

In [48]:
#from __future__ import annotations
if __name__ == "__main__":
    # ── CLI shortcut: python notebook.py demo ────────────────────────
    if len(sys.argv) > 1 and sys.argv[1] == "demo":
        demo()
        sys.exit(0)
    # ── Optional Colab setup (uncomment to enable) ───────────────────
    # if IS_COLAB:
    #     setup_colab()
    # ── Startup banner ───────────────────────────────────────────────
    _W   = 70
    thin = "─" * _W
    thick = "═" * _W
    key_status = (
        "\033[92mset ✓\033[0m"
        if os.environ.get("NVIDIA_API_KEY")
        else "\033[91mNOT SET ✗\033[0m"
    )
    C = Config
    print("\n".join(("", thick,
        "  🤖  Universal Autonomous AI Agent — Combined Edition", thin,
        f"  🔑  API key      : {key_status}",
        f"  🧠  Model        : {C.MODEL}",
        f"  📁  Workspace    : {C.BASE_DIR}",
        f"  📂  Files dir    : {C.FILES_DIR}",
        f"  🏛️  Core dir     : {C.CORE_DIR}",
        f"  🔧  Skills dir   : {C.SKILLS_DIR}",
        f"  📋  Log file     : {C.BASE_DIR / 'agent.log'}", thin,
        f"  🔢  Max tokens   : {C.MAX_TOKENS:,}  (output per API call)",
        f"  🔁  Max iters    : {C.MAX_ITERS}",
        f"  📐  Context      : {C.CONTEXT_TOKEN_BUDGET:,} token budget  (window: 262 000)",
        f"  📦  Tool result  : {C.TOOL_RESULT_CHARS_MAX:,} char cap per result",
        f"  ⏱   Timeouts     : code {C.CODE_TIMEOUT}s │ shell {C.SHELL_TIMEOUT}s │ web {C.WEB_TIMEOUT}s",
        f"  🔄  Retry        : {C.API_RETRY_MAX}× backoff  │  cache {C.TOOL_CACHE_SIZE} LRU slots",
        f"  💓  Heartbeat    : every {C.HEARTBEAT_INTERVAL}s",
        f"  🛡️  Py mem limit  : {C.MAX_PYTHON_MEMORY_MB} MB",
        thick,
    )))
    try:
        # If session_start was used, agent is already initialised and heartbeat
        # already running — skip re-init to avoid double-start.
        if "persistence" in vars() and vars().get("agent") is not None:
            pass  # session_start(start_agent=True) already handled everything
        else:
            agent = Agent(stream=True)
            agent.start_background_heartbeat()
        if agent is None:
            raise RuntimeError("Agent initialisation failed inside session_start.")
        #hint = "\n".join(f"   • {t}" for t in _EXAMPLE_TASKS)
        print(f"\n✅ Agent ready.\n")
        if Config.EXPRESS_PROMPT:
            if "ExpressLane" not in globals():
                raise RuntimeError("ExpressLane not available. Please run all cells before enabling EXPRESS_PROMPT.")
            express = ExpressLane(agent)
            express.start()
            print("⚡ Express mode active. Type 'exit' to quit.", flush=True)
            try:
                while True:
                    try:
                        cmd = input("⚡ Express> ").strip()
                        if cmd.lower() in ('exit', 'quit', 'bye'):
                            print("Exiting express mode.")
                            break
                        if not cmd:
                            continue
                        result = express.run_sync(cmd)
                        print(f"\n{result}\n")
                    except KeyboardInterrupt:
                        print("\n(Interrupted — type 'exit' to quit)")
            except KeyboardInterrupt:
                print("\n👋  Express mode interrupted.")
        else:
            agent.chat()
    except KeyboardInterrupt:
        print("\n👋  Interrupted.")
    except Exception as exc:
        print(f"\n❌ Init failed: {exc}\n   ➜  Check NVIDIA_API_KEY is set correctly.")
        raise


══════════════════════════════════════════════════════════════════════
  🤖  Universal Autonomous AI Agent — Combined Edition
──────────────────────────────────────────────────────────────────────
  🔑  API key      : set ✓
  🧠  Model        : stepfun-ai/step-3.5-flash
  📁  Workspace    : /content/agent
  📂  Files dir    : /content/agent/files
  🏛️  Core dir     : /content/agent/core
  🔧  Skills dir   : /content/agent/skills
  📋  Log file     : /content/agent/agent.log
──────────────────────────────────────────────────────────────────────
  🔢  Max tokens   : 61,072  (output per API call)
  🔁  Max iters    : 1000
  📐  Context      : 262,000 token budget  (window: 262 000)
  📦  Tool result  : 60,000 char cap per result
  ⏱   Timeouts     : code 60s │ shell 120s │ web 20s
  🔄  Retry        : 10× backoff  │  cache 256 LRU slots
  💓  Heartbeat    : every 1800s
  🛡️  Py mem limit  : 256 MB
══════════════════════════════════════════════════════════════════════[RESULT   ] ← check_heartbeat: {
 

Exception in thread watchdog:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7103/2789150245.py", line 646, in _loop
TypeError: start_telegram.<locals>._notifying_restart() takes 3 positional arguments but 4 were given


✓  [QUEUE] Done:     Check heartbeat, review Need.md, summarise MEMORY.md

[YOU] exit


/tmp/ipykernel_7103/1772824882.py:162: UserWarning: No stats file configured; skipping persistence
  warnings.warn("No stats file configured; skipping persistence")


👋  Goodbye!
Requests: 4 (✓4 ✗0) | Prompt: 15,567 │ Think: 716 │ Resp: 1,156 │ Avg: 0.44/min


## END

Terminal Code to install claude code
```
# 1) Run native installer
curl -fsSL https://claude.ai/install.sh | bash

# 2) Start a new terminal, then verify
claude --version
```

In [50]:
!git clone https://github.com/1rgs/claude-code-Proxy

Cloning into 'claude-code-Proxy'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 145 (delta 86), reused 49 (delta 49), pack-reused 42 (from 1)
Receiving objects: 100% (145/145), 888.63 KiB | 13.67 MiB/s, done.
Resolving deltas: 100% (88/88), done.


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv run uvicorn server:app --host 0.0.0.0 --port 8082 --reload

downloading uv 0.11.0 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
INFO:     Will watch for changes in these directories: ['/content/agent/claude-code-Proxy']
INFO:     Uvicorn running on http://0.0.0.0:8082 (Press CTRL+C to quit)
INFO:     Started reloader process [17678] using WatchFiles
POST /v1/messages ✓ 200 OK
claude-haiku-4-5-20251001 → minimax-m2.5 0 tools 1 messages
POST /v1/messages ✓ 200 OK
claude-haiku-4-5-20251001 → minimax-m2.5 0 tools 2 messages
POST /v1/messages ✓ 200 OK
claude-sonnet-4-6 → step-3.5-flash 26 tools 2 messages
